In [ ]:
import pandas as pd
import gdown

# =============================================================================
# Step 1: Read CSV from Google Drive
# =============================================================================

# Your Google Drive link
drive_link = "https://drive.google.com/file/d/1SgmH-S5d3-s3SpOzC3-LzBUWX27Ydc6P/view?usp=drive_link" #(282k rows)
#drive_link="https://drive.google.com/file/d/14_BPfVvpas2Vu2W5rE-yql6-65KK3yI4/view?usp=drive_link" (130K rows)
# Extract file ID
file_id = drive_link.split("/d/")[1].split("/")[0]

In [ ]:
# Download
print("Downloading...")
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, "temp.csv", quiet=False)

# Read CSV
print("\nReading CSV...")
df = pd.read_csv("temp.csv", encoding='utf-8', low_memory=False)
print(f"✅ Loaded {len(df):,} rows")

In [ ]:
import re

# Generic wildlife keywords (broader coverage)
keywords = re.compile(
    r"(wildlife|wild|animal|nature|forest|jungle|ocean|sea|marine|bird|mammal|reptile|fish|"
    r"elephant|lion|tiger|bear|whale|dolphin|turtle|shark|penguin|polar bear|panda|rhino|gorilla|"
    r"endangered|extinction|species|habitat|ecosystem|biodiversity|conservation|protect|save|"
    r"climate|environment|pollution|deforestation|poaching|rescue|rehabilitation|"

    # Arabic - Generic wildlife terms
    r"حيوان|حيوانات|حياة برية|طبيعة|غابة|محيط|بحر|بحرية|طير|طيور|"
    r"فيل|أسد|نمر|دب|حوت|دلفين|سلحفاة|قرش|بطريق|دب قطبي|باندا|وحيد القرن|غوريلا|"
    r"نادر|انقراض|مهدد|موطن|نظام بيئي|تنوع|حماية|حفاظ|إنقاذ|"
    r"مناخ|بيئة|تلوث|غابات|صيد|إعادة تأهيل|"

    # Dialectal Arabic
    r"حيوانات البرية|الطبيعة|بيئة|حماية الحيوانات|حيوان نادر|"
    r"بيموت|بيختفي|بينقرض|إنقذوه|ساعدوه|نحميه)",

    flags=re.IGNORECASE
)

# Apply filter
df_filtered = df[df["Message"].str.contains(keywords, na=False)]

print(f"✓ Original: {len(df):,} posts")
print(f"✓ Filtered: {len(df_filtered):,} posts")
print(f"✓ Kept: {(len(df_filtered)/len(df)*100):.2f}%")

In [ ]:
!pip install langdetect

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0  # Ensures consistent results

def safe_detect(text):
    try:
        if isinstance(text, str) and text.strip():
            return detect(text)
        return None
    except Exception:
        return None

# Create a new language column
df['lang'] = df['Message'].apply(safe_detect)
# Count occurrences of each language
print(df['lang'].value_counts())

In [ ]:
print(df['lang'].value_counts())

In [ ]:
import pandas as pd
import numpy as np

# Dates
df2['Post Created Date'] = pd.to_datetime(
    df2['Post Created Date'],
    errors='coerce'
)

summary = {
    'Total posts': len(df2),
    'Unique pages': df2['Page Name'].nunique(),
    'Start date': df2['Post Created Date'].min(),
    'End date': df2['Post Created Date'].max(),
    'Collection days': (
        df2['Post Created Date'].max()
        - df2['Post Created Date'].min()
    ).days + 1,

    'Mean interactions': df2['Total Interactions'].mean(),
    'Median interactions': df2['Total Interactions'].median(),

    'Mean comments': df2['Comments'].mean(),
    'Median comments': df2['Comments'].median(),

    'Mean shares': df2['Shares'].mean(),
    'Median shares': df2['Shares'].median(),

    'Posts with reactions': (
        df2[['Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care']]
        .fillna(0)
        .sum(axis=1)
        > 0
    ).sum()
}

pd.Series(summary)

In [ ]:
len(df2)

In [ ]:
df.columns.tolist()

In [ ]:
"""
Multilingual embedding-based topic assignment for wildlife posts.

Each post is embedded with a Sentence Transformers model (default: BAAI/bge-m3)
and assigned to the most similar of 12 standardized bilingual (English/Arabic)
topic descriptions, provided the cosine similarity clears a threshold.
"""

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION  (edit these)
# ============================================================

INPUT_FILE = "wildlife_posts.csv"            # <-- path to your input data
OUTPUT_FILE = "Wildlife_12_topics_e5_large22.csv"
MESSAGE_COLUMN = "Message"
#MODEL_NAME = "BAAI/bge-m3"
MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
SIMILARITY_THRESHOLD = 0.3                   # must be calibrated on human labels
BATCH_SIZE = 32
EXPECTED_NUM_TOPICS = 12
UNASSIGNED_LABEL = "UNASSIGNED"


# ============================================================
# 12 STANDARDIZED WILDLIFE TOPICS (English + Arabic)
# ============================================================

TOPIC_DEFINITIONS = {
    "Captivity, Zoos & Ethical Wildlife Management": """
Wild animals kept in zoos, aquariums, breeding facilities,
or captivity, including animal welfare, living conditions,
captive breeding, management, and ethical treatment.

الحيوانات البرية الموجودة في حدائق الحيوان أو الأحواض المائية
أو مراكز التكاثر أو الأسر، بما في ذلك رفاهية الحيوانات
وظروف الاحتجاز والتكاثر في الأسر والإدارة والمعاملة الأخلاقية.
""",

    "Climate Change Impact on Wildlife": """
Effects of climate change, global warming, extreme heat,
climate-related drought, changing weather patterns, and
other climatic changes on wild animals, their habitats,
migration, reproduction, and survival.

تأثير تغير المناخ والاحتباس الحراري وارتفاع درجات الحرارة
والجفاف المرتبط بالمناخ وتغير أنماط الطقس على الحيوانات
البرية وموائلها وهجرتها وتكاثرها وبقائها.
""",

    "Habitat Loss, Deforestation & Ecosystem Degradation": """
Destruction, fragmentation, or degradation of natural
habitats and ecosystems affecting wildlife, including
deforestation, land conversion, urban expansion,
terrestrial pollution, and loss of biodiversity.

تدمير الموائل الطبيعية وتجزئتها وتدهور النظم البيئية
التي تعتمد عليها الحياة البرية، بما في ذلك إزالة الغابات
وتغيير استخدام الأراضي والتوسع العمراني والتلوث البري
وفقدان التنوع البيولوجي.
""",

    "Wildlife Rehabilitation & Rescue": """
Rescue, veterinary treatment, rehabilitation, recovery,
and release of individual wild animals that are injured,
stranded, orphaned, trapped, or in distress.

إنقاذ الحيوانات البرية المصابة أو العالقة أو اليتيمة
أو المعرضة للخطر، وعلاجها بيطرياً وإعادة تأهيلها
وتعافيها وإطلاقها مجدداً في البرية.
""",

    "Wildlife Awareness, Media & Activism": """
Public education, awareness campaigns, environmental
advocacy, wildlife documentaries, photography, observation,
species information, animal behavior, and general
appreciation or promotion of wildlife and nature.

التوعية العامة والتثقيف البيئي وحملات حماية الحياة البرية
والنشاط البيئي والأفلام الوثائقية وتصوير الحيوانات
ومشاهدتها والمعلومات عن الأنواع وسلوك الحيوانات
والتقدير العام للحياة البرية والطبيعة.
""",

    "Human-Wildlife Conflict": """
Harmful or disputed interactions between wild animals
and humans, including attacks on people, crop damage,
livestock predation, retaliatory killing, property
damage, and efforts to improve human-wildlife coexistence.

الصراعات والتفاعلات الضارة بين الإنسان والحيوانات البرية،
بما في ذلك الهجمات على البشر وإتلاف المحاصيل وافتراس
الماشية والقتل الانتقامي والأضرار بالممتلكات والجهود
المبذولة لتحسين التعايش بين الإنسان والحياة البرية.
""",

    "Marine & Ocean Conservation": """
Protection and management of marine wildlife and aquatic
ecosystems, including whales, dolphins, sea turtles,
coral reefs, marine pollution, plastic waste, overfishing,
bycatch, and threats to marine biodiversity.

حماية وإدارة الحياة البرية البحرية والنظم البيئية المائية،
بما في ذلك الحيتان والدلافين والسلاحف البحرية والشعاب
المرجانية والتلوث البحري والنفايات البلاستيكية والصيد
الجائر والصيد العرضي والتهديدات للتنوع البيولوجي البحري.
""",

    "Endangered Species & Extinction Risk": """
Conservation status and survival of wildlife species,
including endangered and threatened species, declining
populations, extinction risk, species-level recovery,
and protection or reintroduction of threatened species.

حالة حفظ الأنواع البرية وبقائها، بما في ذلك الأنواع
المهددة بالانقراض وتناقص أعدادها وخطر انقراضها
وبرامج استعادة الأنواع وحمايتها وإعادة توطينها.
""",

    "Wildlife Crime, Poaching & Enforcement": """
Illegal hunting, killing, capturing, trading, or
trafficking of wild animals and wildlife products,
including poaching, illegal wildlife trade, investigations,
seizures, arrests, and enforcement against wildlife crime.

الصيد غير القانوني وقتل الحيوانات البرية وأسرها
والاتجار غير المشروع بها أو بمنتجاتها، بما في ذلك
الصيد الجائر وتهريب الحيوانات والتحقيقات والمصادرات
والاعتقالات وإنفاذ قوانين مكافحة جرائم الحياة البرية.
""",

    "Conservation Policy, Law & Funding": """
Government policies, legislation, regulations, funding,
donations, institutional programs, and activities by
conservation organizations intended to protect wildlife,
natural resources, and biodiversity.

السياسات الحكومية والتشريعات واللوائح والتمويل
والتبرعات والبرامج المؤسسية وأنشطة منظمات الحفاظ
على البيئة الهادفة إلى حماية الحياة البرية والموارد
الطبيعية والتنوع البيولوجي.
""",

    "Indigenous & Community-Based Conservation": """
Wildlife conservation involving indigenous peoples
or local communities, including traditional ecological
knowledge, community-led protection, local natural
resource management, and land rights connected to
biodiversity conservation.

حماية الحياة البرية بمشاركة السكان الأصليين أو المجتمعات
المحلية، بما في ذلك المعرفة البيئية التقليدية ومبادرات
الحماية المجتمعية وإدارة الموارد الطبيعية محلياً
وحقوق الأراضي المرتبطة بالحفاظ على التنوع البيولوجي.
""",

    "National Parks & Protected Areas": """
Establishment, management, conservation, and visitation
of national parks, nature reserves, wildlife sanctuaries,
and designated protected areas, including their ecological
importance and role in protecting wildlife.

إنشاء وإدارة وحماية وزيارة الحدائق الوطنية والمحميات
الطبيعية ومحميات الحياة البرية والمناطق المحمية،
بما في ذلك أهميتها البيئية ودورها في حماية الحيوانات البرية.
""",
}


# ============================================================
# TOPIC ASSIGNMENT
# ============================================================

def apply_sbert_topic_modeling(
    df,
    topic_definitions,
    message_column="Message",
    similarity_threshold=0.3,
    model_name="intfloat/multilingual-e5-large-instruct",
    #model_name="BAAI/bge-m3",
    batch_size=32,
    expected_num_topics=12,
    unassigned_label="UNASSIGNED",
):
    """
    Assign wildlife topics using multilingual embeddings.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset containing social media messages.

    topic_definitions : dict
        Mapping of topic name -> bilingual topic description.

    message_column : str
        Column containing message text.

    similarity_threshold : float
        Minimum cosine similarity required for assignment.
        This threshold must be calibrated using human labels.

    model_name : str
        Sentence Transformers compatible embedding model.

    batch_size : int
        Number of posts encoded and classified per batch.

    expected_num_topics : int or None
        If set, raise an error unless exactly this many topics
        are supplied. Use None to skip the check.

    unassigned_label : str
        Label given to posts below the threshold or with no text.

    Returns
    -------
    pandas.DataFrame
        Copy of the input dataframe with additional topic columns:
        Assigned_Topic, SBERT_Best_Topic, Topic_Similarity,
        Second_Topic_Similarity, Topic_Similarity_Margin,
        Topic_Confidence.
    """

    print("=" * 90)
    print("MULTILINGUAL WILDLIFE TOPIC ASSIGNMENT".center(90))
    print("=" * 90)

    # --------------------------------------------------------
    # Step 1: Validate inputs
    # --------------------------------------------------------

    if message_column not in df.columns:
        raise ValueError(
            f"Column '{message_column}' not found. "
            f"Available columns: {list(df.columns)}"
        )

    if len(topic_definitions) < 2:
        raise ValueError(
            "At least 2 topic definitions are required."
        )

    if (
        expected_num_topics is not None
        and len(topic_definitions) != expected_num_topics
    ):
        raise ValueError(
            f"Expected exactly {expected_num_topics} topic "
            f"definitions, got {len(topic_definitions)}."
        )

    if not -1.0 <= similarity_threshold <= 1.0:
        raise ValueError(
            "Cosine similarity threshold must be "
            "between -1 and 1."
        )

    if batch_size < 1:
        raise ValueError("batch_size must be >= 1.")

    df = df.copy()

    topic_names = list(topic_definitions.keys())

    # Collapse line breaks / extra whitespace so the layout of
    # the definitions in the source file does not affect embeddings.
    topic_descriptions = [
        " ".join(str(description).split())
        for description in topic_definitions.values()
    ]

    empty_topics = [
        name
        for name, description in zip(topic_names, topic_descriptions)
        if not description
    ]
    if empty_topics:
        raise ValueError(
            f"Empty topic description(s): {empty_topics}"
        )

    print(f"\nTotal posts: {len(df):,}")
    print(f"Number of topics: {len(topic_names)}")
    print(f"Model: {model_name}")
    print(f"Similarity threshold: {similarity_threshold}")

    # --------------------------------------------------------
    # Step 2: Prepare messages
    # --------------------------------------------------------

    messages = (
        df[message_column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    valid_positions = np.flatnonzero(
        messages.ne("").to_numpy()
    )

    print(f"Valid messages: {len(valid_positions):,}")
    print(
        f"Empty messages: "
        f"{len(df) - len(valid_positions):,}"
    )

    # --------------------------------------------------------
    # Step 3: Initialize output arrays
    # --------------------------------------------------------

    n = len(df)

    assigned_topics = np.full(n, unassigned_label, dtype=object)
    best_topics = np.full(n, None, dtype=object)
    topic_similarities = np.full(n, np.nan, dtype=np.float32)
    second_similarities = np.full(n, np.nan, dtype=np.float32)
    similarity_margins = np.full(n, np.nan, dtype=np.float32)

    # --------------------------------------------------------
    # Step 4: Load embedding model
    # --------------------------------------------------------

    print("\nLoading embedding model...")
    model = SentenceTransformer(model_name)
    print("Model loaded successfully.")

    # --------------------------------------------------------
    # Step 5: Encode standardized topic descriptions
    # --------------------------------------------------------

    print("\nEncoding topic descriptions...")

    topic_embeddings = model.encode(
        topic_descriptions,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype(np.float32)

    topic_names_array = np.array(topic_names, dtype=object)

    print("Topic embeddings generated.")

    # --------------------------------------------------------
    # Step 6: Encode and classify posts in batches
    # --------------------------------------------------------

    print("\nEncoding and classifying messages...")

    batch_starts = range(0, len(valid_positions), batch_size)

    for start in tqdm(batch_starts, desc="Batches", unit="batch"):

        positions = valid_positions[start:start + batch_size]
        batch_messages = messages.iloc[positions].tolist()

        post_embeddings = model.encode(
            batch_messages,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        ).astype(np.float32)

        # Cosine similarity: both sides are L2-normalized,
        # so the dot product is the cosine similarity.
        similarities = post_embeddings @ topic_embeddings.T

        # Highest and second-highest topic similarity per post
        sorted_indices = np.argsort(similarities, axis=1)
        best_indices = sorted_indices[:, -1]
        second_indices = sorted_indices[:, -2]

        rows = np.arange(len(positions))
        best_scores = similarities[rows, best_indices]
        second_scores = similarities[rows, second_indices]

        batch_topics = topic_names_array[best_indices]

        # Threshold-based classification
        is_assigned = best_scores >= similarity_threshold

        assigned_topics[positions] = np.where(
            is_assigned, batch_topics, unassigned_label
        )
        best_topics[positions] = batch_topics
        topic_similarities[positions] = best_scores
        second_similarities[positions] = second_scores
        similarity_margins[positions] = best_scores - second_scores

    # --------------------------------------------------------
    # Step 7: Store results
    # --------------------------------------------------------

    # Final label (UNASSIGNED if below threshold or empty text)
    df["Assigned_Topic"] = assigned_topics

    # Best semantic match, even if below threshold
    df["SBERT_Best_Topic"] = best_topics

    # Highest and second-highest cosine similarity
    df["Topic_Similarity"] = topic_similarities
    df["Second_Topic_Similarity"] = second_similarities

    # Difference between highest and second-highest scores
    df["Topic_Similarity_Margin"] = similarity_margins

    # Retained for compatibility with the original column name.
    # This is a similarity-margin heuristic,
    # NOT a calibrated classification probability.
    df["Topic_Confidence"] = similarity_margins

    # --------------------------------------------------------
    # Step 8: Print classification summary
    # --------------------------------------------------------

    print("\n" + "=" * 90)
    print("TOPIC ASSIGNMENT SUMMARY".center(90))
    print("=" * 90)

    topic_counts = df["Assigned_Topic"].value_counts()

    print(f"\n{'Topic':<55} {'Count':>10} {'%':>8}")
    print("-" * 75)

    for topic, count in topic_counts.items():
        pct = 100 * count / n if n else 0
        print(f"{topic:<55} {count:>10,} {pct:>7.2f}%")

    assigned_count = int(
        (df["Assigned_Topic"] != unassigned_label).sum()
    )
    coverage = 100 * assigned_count / n if n else 0

    print(
        f"\nAssignment coverage: {coverage:.2f}% "
        f"({assigned_count:,} of {n:,} posts)"
    )

    valid_scores = topic_similarities[valid_positions]
    if len(valid_scores):
        print(
            "Best-topic similarity (non-empty posts): "
            f"min={valid_scores.min():.3f}, "
            f"median={np.median(valid_scores):.3f}, "
            f"max={valid_scores.max():.3f}"
        )

    print("=" * 90)

    return df


# ============================================================
# RUN BGE-M3 TOPIC ASSIGNMENT
# ============================================================

if __name__ == "__main__":

    #df = pd.read_csv(INPUT_FILE)

    df_with_topics = apply_sbert_topic_modeling(
        df=df,
        topic_definitions=TOPIC_DEFINITIONS,
        message_column=MESSAGE_COLUMN,
        similarity_threshold=SIMILARITY_THRESHOLD,
        model_name=MODEL_NAME,
        batch_size=BATCH_SIZE,
        expected_num_topics=EXPECTED_NUM_TOPICS,
        unassigned_label=UNASSIGNED_LABEL,
    )

    df_with_topics.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"Saved {len(df_with_topics):,} posts to {OUTPUT_FILE}"
    )

In [ ]:

import numpy as np
import pandas as pd

thresholds = np.arange(0.30, 0.61, 0.025)

results = []

for threshold in thresholds:

    assigned = (
        df_with_topics["Topic_Similarity"] >= threshold
    ).sum()

    unassigned = len(df_with_topics) - assigned

    results.append({
        "Threshold": round(float(threshold), 3),
        "Assigned": assigned,
        "Unassigned": unassigned,
        "Assignment_Rate": round(
            assigned / len(df_with_topics) * 100, 2
        ),
        "Unassigned_Rate": round(
            unassigned / len(df_with_topics) * 100, 2
        )
    })

threshold_results = pd.DataFrame(results)

print(threshold_results.to_string(index=False))


In [ ]:

import numpy as np
import pandas as pd

thresholds = np.arange(0.76, 0.891, 0.01)

results = []

for threshold in thresholds:
    assigned = (
        df_with_topics["Topic_Similarity"] >= threshold
    ).sum()

    total = len(df_with_topics)

    results.append({
        "Threshold": round(float(threshold), 3),
        "Assigned": assigned,
        "Unassigned": total - assigned,
        "Assigned_%": round(100 * assigned / total, 2),
        "Unassigned_%": round(
            100 * (total - assigned) / total, 2
        )
    })

e5_threshold_results = pd.DataFrame(results)

print(e5_threshold_results.to_string(index=False))


In [ ]:

from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

# human_labels:
# Human-verified topic labels, including UNASSIGNED

y_true = (
    df_e5["Human_Topic"] != "UNASSIGNED"
).astype(int)

models = {
    "E5-large": df_e5["Topic_Similarity"],
    "BGE-M3": df_bge["Topic_Similarity"]
}

results = []

for name, scores in models.items():

    valid = scores.notna() & y_true.notna()

    if y_true[valid].nunique() < 2:
        raise ValueError(
            "Both relevant and UNASSIGNED human labels "
            "are required to calculate AUROC."
        )

    results.append({
        "Model": name,
        "AUROC": roc_auc_score(
            y_true[valid],
            scores[valid]
        ),
        "Average_Precision": average_precision_score(
            y_true[valid],
            scores[valid]
        )
    })

print(pd.DataFrame(results))




In [ ]:
df2 = df2.merge(
    df[['URL', 'language']],
    on='URL',
    how='left'
)

In [ ]:
language_stats = (
    df['language']
    .value_counts(dropna=False)
    .to_frame('Count')
)

language_stats['Percent'] = (
    language_stats['Count'] / len(df) * 100
).round(2)

language_stats

In [ ]:
language_stats = (
    df2['language']
    .value_counts(dropna=False)
    .to_frame('Count')
)

language_stats['Percent'] = (
    language_stats['Count'] / len(df2) * 100
).round(2)

language_stats

In [ ]:
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- setup
REACTIONS = ['Likes', 'Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care']

num = lambda s: pd.to_numeric(s.astype(str).str.replace(',', '', regex=False),
                              errors='coerce')

d = df.copy()
d['TI'] = num(d['Total Interactions'])
for c in REACTIONS:
    d[c] = num(d[c]).fillna(0)
d['followers'] = num(d['Followers at Posting'])

msg = d['Message'].fillna('').astype(str)
d['n_chars']    = msg.str.len()
d['n_hashtags'] = msg.str.count(r'#[\w\u0600-\u06FF_]+')
d['has_url']    = msg.str.contains(r'https?://|www\.', regex=True)
d['empty_msg']  = msg.str.strip().eq('')

ctry = d['Page Admin Top Country'].astype(str).str.strip()
d['country'] = ctry.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

ARAB = {'DZ','BH','KM','DJ','EG','IQ','JO','KW','LB','LY','MR','MA',
        'OM','PS','QA','SA','SO','SD','SY','TN','AE','YE'}

N = len(d)
out = []
add = lambda k, v: out.append((k, v))

# ---------------------------------------------------------------- corpus
add('Posts', f"{N:,}")
add('Collection window',
    f"{pd.to_datetime(d['Post Created Date']).min():%d %b %Y} – "
    f"{pd.to_datetime(d['Post Created Date']).max():%d %b %Y}")
add('Unique pages (Facebook ID)', f"{d['Facebook Id'].nunique():,}")
add('Page categories', f"{d['Page Category'].nunique():,}")

ppp = d.groupby('Facebook Id').size()
top1p = ppp.nlargest(max(1, int(np.ceil(.01 * len(ppp))))).sum()
add('Posts per page, median (IQR)',
    f"{ppp.median():.0f} ({ppp.quantile(.25):.0f}–{ppp.quantile(.75):.0f})")
add('Share of posts from top 1\\% of pages', f"{100*top1p/N:.1f}\\%")
add('Page followers at posting, median',
    f"{d['followers'].median():,.0f}")

# ---------------------------------------------------------------- content
for lab, n in d['Type'].value_counts().head(3).items():
    add(f'Content type: {lab}', f"{n:,} ({100*n/N:.1f}\\%)")

langcol = 'language'
for lab, n in d[langcol].value_counts().items():
    add(f'Language: {lab}', f"{n:,} ({100*n/N:.1f}\\%)")

add('Message length (chars), median (IQR)',
    f"{d['n_chars'].median():,.0f} "
    f"({d['n_chars'].quantile(.25):,.0f}–{d['n_chars'].quantile(.75):,.0f})")
add('Posts with no message text', f"{100*d['empty_msg'].mean():.1f}\\%")
add('Hashtags per post, mean', f"{d['n_hashtags'].mean():.2f}")
add('Posts with $\\geq$1 hashtag', f"{100*(d['n_hashtags']>0).mean():.1f}\\%")
add('Posts containing a URL', f"{100*d['has_url'].mean():.1f}\\%")

# ---------------------------------------------------------------- engagement
add('Total interactions', f"{d['TI'].sum():,.0f}")
add('Interactions per post, median (IQR)',
    f"{d['TI'].median():,.0f} "
    f"({d['TI'].quantile(.25):,.0f}–{d['TI'].quantile(.75):,.0f})")
add('Interactions per post, mean', f"{d['TI'].mean():,.1f}")
add('Share of interactions from top 1\\% of posts',
    f"{100*d['TI'].nlargest(int(np.ceil(.01*N))).sum()/d['TI'].sum():.1f}\\%")

# ---------------------------------------------------------------- reactions
rtot = d[REACTIONS].sum()
like_share_corpus = 100 * rtot['Likes'] / rtot.sum()
add('\\emph{Like} share of all reactions', f"{like_share_corpus:.1f}\\%")

by_c = d[d['country'].isin(ARAB)].groupby('country')[REACTIONS].sum()
by_c = by_c[by_c.sum(axis=1) >= 10_000]
ls = 100 * by_c['Likes'] / by_c.sum(axis=1)
add('\\emph{Like} share, range across countries',
    f"{ls.min():.1f}–{ls.max():.1f}\\%")
add('Posts with $\\geq$5 reactions',
    f"{100*(d[REACTIONS].sum(axis=1) >= 5).mean():.1f}\\%")

# ---------------------------------------------------------------- geography
has_c = d['country'].notna()
geo = d[has_c].groupby('country').size().sort_values(ascending=False)
add('Posts with page-admin country', f"{100*has_c.mean():.1f}\\%")
add('Distinct admin countries', f"{d['country'].nunique()}")
add('Posts from Arab League states',
    f"{100*d['country'].isin(ARAB).sum()/has_c.sum():.1f}\\% of located posts")
add('Share from top 10 countries', f"{100*geo.head(10).sum()/has_c.sum():.1f}\\%")
add('Largest single country',
    f"{geo.index[0]} ({100*geo.iloc[0]/has_c.sum():.1f}\\%)")

# ---------------------------------------------------------------- topics
if 'Assigned_Topic' in d.columns:
    t = d['Assigned_Topic'].astype(str)
    unass = t.str.upper().str.contains('UNASSIGNED|NONE')
    add('Topics assigned', f"{t[~unass].nunique()}")
    add('Posts unassigned', f"{100*unass.mean():.1f}\\%")

# ---------------------------------------------------------------- output
stats = pd.DataFrame(out, columns=['Statistic', 'Value'])
print(stats.to_string(index=False))

print('\n' + '='*70 + '\nLaTeX\n' + '='*70)
print(r"""\begin{table}[t]
\centering
\caption{Corpus descriptive statistics.}
\label{tab:descriptives}
\small
\begin{tabular}{@{}lr@{}}
\toprule
\textbf{Statistic} & \textbf{Value} \\
\midrule""")
for k, v in out:
    print(f"{k} & {v} \\\\")
print(r"""\bottomrule
\end{tabular}
\end{table}""")

# ---------------------------------------------------------------- appendix
print('\n--- for appendix ---')
print('\nContent type:\n',
      (d['Type'].value_counts(normalize=True)*100).round(2).to_string())
print('\nTop 15 countries:\n',
      (geo.head(15)/has_c.sum()*100).round(2).to_string())
print('\nLike share by country:\n', ls.sort_values().round(1).to_string())

In [ ]:
import numpy as np
import pandas as pd

REACTIONS = ['Likes', 'Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care']

num = lambda s: pd.to_numeric(s.astype(str).str.replace(',', '', regex=False),
                              errors='coerce')

reac = df[REACTIONS].apply(num).fillna(0)
df['n_reactions'] = reac.sum(axis=1)

pct5 = 100 * (df['n_reactions'] >= 5).mean()
print(f"Posts with >=5 reactions: {(df['n_reactions'] >= 5).sum():,} ({pct5:.1f}%)")

# threshold sweep — needed for the cohesion sensitivity analysis
print("\nThreshold   Posts      % of corpus")
for k in [1, 2, 5, 10, 25, 50]:
    n = (df['n_reactions'] >= k).sum()
    print(f"  >={k:<3}    {n:>8,}    {100*n/len(df):>5.1f}%")

print(f"\nMedian reactions per post: {df['n_reactions'].median():.0f}")
print(f"Posts with 0 reactions:    {100*(df['n_reactions'] == 0).mean():.1f}%")

In [ ]:
import re
from collections import Counter

def extract_hashtags(text):
    if pd.isna(text):
        return []

    return re.findall(
        r'#[\w\u0600-\u06FF]+',
        str(text),
        flags=re.UNICODE
    )

df['hashtags'] = df['Message'].apply(extract_hashtags)
df['hashtag_count'] = df['hashtags'].apply(len)

print(
    'Posts with hashtags:',
    (df['hashtag_count'] > 0).sum()
)

print(
    'Percent with hashtags:',
    round((df['hashtag_count'] > 0).mean() * 100, 2)
)

print(
    'Unique hashtags:',
    len(set(
        tag
        for tags in df['hashtags']
        for tag in tags
    ))
)

hashtags = Counter(
    tag.lower()
    for tags in df['hashtags']
    for tag in tags
)

top_hashtags = pd.DataFrame(
    hashtags.most_common(20),
    columns=['Hashtag', 'Count']
)

top_hashtags

In [ ]:
!pip install wordcloud arabic-reshaper python-bidi plotly tashaphyne


In [ ]:
import pandas as pd

df1['Total Interactions'] = pd.to_numeric(
    df1['Total Interactions'].astype(str).str.replace(',', '', regex=False).str.strip(),
    errors='coerce'          # anything unparseable -> NaN
).fillna(0).astype(int)

print(df1['Total Interactions'].dtype)   # should print int64

In [ ]:
# -----------------------------------------------------------------------------
# Global visual style
# -----------------------------------------------------------------------------

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'  # fallback for Matplotlib
PREF_PLOTLY_FONT = dict(family="Noto Naskh Arabic, Amiri, Cairo, DejaVu Sans, Arial", size=16)
ARABIC_REGEX = r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]+'

# -----------------------------------------------------------------------------
# Ultra-light Arabic stemming (strip only the article "ال")
# -----------------------------------------------------------------------------
try:
    from tashaphyne.stemming import ArabicLightStemmer
    _TL_STEMMER = ArabicLightStemmer()
    _TL_STEMMER.set_prefix_list(["ال"])   # only strip the article
    _TL_STEMMER.set_suffix_list([""])     # keep suffixes intact

    def ultralight_stem(w: str) -> str:
        if not isinstance(w, str) or not w:
            return w
        _TL_STEMMER.light_stem(w)
        return _TL_STEMMER.get_stem() or w

    from functools import lru_cache
    @lru_cache(maxsize=100_000)
    def cached_safe_stem_token(tok: str) -> str:
        return safe_stem_token(tok)

    def safe_stem_token(w: str) -> str:
      SPECIAL_TOKENS_NO_STEM = {'الله'}  # you can add more later if needed
      """
        Wrapper around ultralight_stem that:
          - keeps الله (and any other special tokens) as-is
          - otherwise applies ultralight_stem
        """
      if not isinstance(w, str) or not w:
            return w
      if w in SPECIAL_TOKENS_NO_STEM:
            return w
      return ultralight_stem(w)

except Exception:
    _TL_STEMMER = None
    def ultralight_stem(w: str) -> str:
        return w  # fallback if Tashaphyne isn't available

    def safe_stem_token(w: str) -> str:
        """Fallback: identity function when tashaphyne is unavailable."""
        if not isinstance(w, str) or not w:
            return w
        return w

    from functools import lru_cache
    @lru_cache(maxsize=100_000)
    def cached_safe_stem_token(tok: str) -> str:
        return safe_stem_token(tok)

In [ ]:
from functools import lru_cache

# Make sure you have a safe_stem_token(word: str) defined somewhere
@lru_cache(maxsize=100_000)
def cached_safe_stem_token(tok: str) -> str:
    return safe_stem_token(tok)


# =============================================================================
# Arabic Keyword Extractor & Visualizer
# =============================================================================
class ArabicKeywordExtractor:
    """Extracts and analyzes keywords from post messages"""

    def __init__(self, df, message_column='Message'):
        import re
        import pandas as pd

        self.df = df
        self.message_column = message_column

        # 1️⃣ Detect if any English letters appear in the corpus (before cleaning)
        try:
            self.contains_english = (
                self.df[self.message_column]
                .astype(str)
                .str.contains(r'[A-Za-z]')
                .any()
            )
        except Exception:
            self.contains_english = False

        # 2️⃣ Arabic stop words: MSA list + stopword file + dialect/religious additions
        #    Already normalized (no diacritics/tatweel, أ/إ/آ -> ا).
        #    Tokens must be normalized the SAME way before this check.

        self.stop_words_ar = {
                # ------------------------------------------------------------------
                # Punctuation / isolated letters
                # ------------------------------------------------------------------
                '،', 'ء', 'ؤ', 'ئ',
                'ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز',
                'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك',
                'ل', 'م', 'ن', 'ه', 'و', 'ي', 'ى', 'ة',

                'باء', 'تاء', 'ثاء', 'جيم', 'حاء', 'خاء', 'دال', 'ذال',
                'راء', 'زاي', 'سين', 'شين', 'صاد', 'ضاد', 'طاء', 'ظاء',
                'عين', 'غين', 'فاء', 'قاف', 'كاف', 'لام', 'ميم', 'نون',
                'هاء', 'واو', 'ياء', 'همزة',

                # ------------------------------------------------------------------
                # Basic function words / particles
                # ------------------------------------------------------------------
                'ال', 'الا', 'إلا', 'الى', 'إلى', 'الي',
                'او', 'أو', 'ام', 'أما', 'اما',
                'ان', 'أن', 'إن', 'انه', 'أنها', 'انها',
                'انى', 'أني', 'اني', 'انك',
                'اذ', 'إذ', 'اذا', 'إذا', 'اذن', 'إذن',
                'اذما', 'انما', 'إنما',
                'بل', 'بلى', 'ثم',
                'قد', 'فقد',
                'لا', 'لم', 'لن', 'لما',
                'لو', 'لولا', 'لوما',
                'لكن', 'لكنه', 'ولكن',
                'كذلك', 'لذلك',
                'حتى', 'حتي',
                'سوف',
                'فقط',
                'أيضا', 'ايضا',
                'غير',
                'سوى',
                'دون',
                'ضمن',
                'ضد',
                'مثل',
                'نحو',
                'حسب',
                'مقابل',

                # ------------------------------------------------------------------
                # Prepositions
                # ------------------------------------------------------------------
                'في', 'فى',
                'من',
                'عن',
                'على', 'علي',
                'مع',
                'بين',
                'خلال',
                'قبل',
                'بعد',
                'منذ',
                'مذ',
                'عند',
                'عندما',
                'حول',
                'تحت',
                'فوق',
                'خلف',
                'خارج',
                'تجاه',
                'ازاء',
                'أمام', 'امام',
                'وراء',

                # ------------------------------------------------------------------
                # Pronouns
                # ------------------------------------------------------------------
                'انا', 'أنا',
                'انت', 'أنت',
                'انتي', 'أنتي',
                'انتم', 'أنتم',
                'انتما', 'أنتما',
                'انتن', 'أنتن',
                'انتو',

                'هو',
                'هي', 'هى',
                'هما',
                'هم',
                'هن',

                'نحن',
                'نحنا',
                'احنا',
                'واحنا',

                'اياي', 'اياك', 'اياكم', 'اياكما', 'اياكن',
                'اياه', 'اياها', 'اياهم', 'اياهما', 'اياهن',
                'ايانا',

                'نفسي',
                'نفسك',
                'نفسه',
                'نفسها',
                'نفسنا',
                'نفسهم',
                'نفسكم',
                'انفسنا',
                'انفسكم',
                'انفسهم',

                # ------------------------------------------------------------------
                # Attached-pronoun forms / frequent combinations
                # ------------------------------------------------------------------
                'لي',
                'لك', 'لكم',
                'له', 'لها', 'لهم', 'لهما',
                'لنا',

                'بي',
                'بك', 'بكم', 'بكما',
                'به', 'بها', 'بهم', 'بهما',

                'فيه', 'فيها',

                'عليه', 'عليها', 'عليك', 'عليا',
                'عنه', 'عنها',

                'معه', 'معها',

                'منه', 'منها', 'منكم',

                'اليك', 'اليكم', 'اليكما', 'اليكن',
                'اليه', 'اليها',

                # ------------------------------------------------------------------
                # Demonstratives
                # ------------------------------------------------------------------
                'هذا', 'هذه', 'هذي', 'هاذا',
                'هذان', 'هذين',
                'هؤلاء',
                'هذولا',

                'ذلك', 'تلك',
                'ذاك',
                'ذان', 'ذانك',
                'ذين', 'ذينك',

                'هاد', 'هادا', 'هادي',
                'هاته',
                'هاتان', 'هاتين',
                'هاي',
                'هيدا',
                'هدول',

                'اولئك', 'أولئك',
                'اولاء',

                'بهذا',
                'لهذا',
                'وهذا',
                'وذلك',

                # ------------------------------------------------------------------
                # Relative pronouns
                # ------------------------------------------------------------------
                'الذي', 'الذى',
                'التي', 'التى',
                'الذين',
                'اللذين',
                'اللذان',
                'اللتان',
                'اللتين',
                'اللاتي',
                'اللواتي',
                'والذي',
                'والتي',

                # ------------------------------------------------------------------
                # Interrogatives
                # ------------------------------------------------------------------
                'ما',
                'ماذا',
                'متى',
                'اين', 'أين',
                'كيف',
                'كيفما',
                'لماذا',
                'هل',
                'كم',
                'اي', 'أي',
                'ايش',
                'ليش',
                'ليه',
                'فين',
                'وين',
                'منين',
                'شنو',
                'شو',
                'وش',

                # ------------------------------------------------------------------
                # Quantifiers / generic modifiers
                # ------------------------------------------------------------------
                'كل',
                'كلا',
                'كلتا',
                'كلما',
                'كلنا',
                'كلكم',
                'كلهم',

                'بعض',
                'بضع',
                'جميع',
                'اجمع',
                'معظم',

                'اكثر', 'أكثر',
                'اقل', 'أقل',
                'كثير',
                'كثيرا',
                'كتير',
                'قليل',

                'عدة',
                'اضافي', 'إضافي',
                'خاصة',
                'عامة',
                'غالبا',
                'جدا',
                'جداً',

                'جديد',
                'جديدة',
                'اخر', 'آخر',
                'اخري', 'اخرى', 'أخرى',
                'افضل', 'أفضل',
                'كبير',

                # ------------------------------------------------------------------
                # Time-related low-information terms
                # ------------------------------------------------------------------
                'الان', 'الآن',
                'اليوم',
                'يوم',
                'ايام', 'أيام',

                'امس', 'أمس',
                'غدا', 'غداً',
                'صباح',
                'صباحا',
                'مساء',

                'الوقت',
                'حين',
                'حاليا',
                'الحالي',
                'السابق',
                'الماضي',
                'المقبل',
                'الاخيرة',

                'عام',
                'عاما',
                'سنة',
                'سنوات',
                'فترة',
                'مدة',
                'ساعة',
                'نهاية',

                'تارة',
                'بغتة',
                'انفا',
                'غداة',
                'ضحوة',

                # ------------------------------------------------------------------
                # Months
                # ------------------------------------------------------------------
                'يناير',
                'فبراير',
                'مارس',
                'ابريل', 'أبريل',
                'مايو',
                'يونيو',
                'يوليو',
                'اغسطس', 'أغسطس',
                'سبتمبر',
                'اكتوبر', 'أكتوبر',
                'نوفمبر',
                'ديسمبر',

                'كانون',
                'شباط',
                'اذار', 'آذار',
                'نيسان',
                'ايار', 'أيار',
                'حزيران',
                'تموز',
                'ايلول', 'أيلول',
                'تشرين',

                'جانفي',
                'فيفري',
                'افريل',
                'جوان',
                'جويلية',
                'اوت',

                # ------------------------------------------------------------------
                # Days
                # ------------------------------------------------------------------
                'سبت',
                'احد', 'أحد',
                'اثنين',
                'ثلاثاء',
                'اربعاء', 'أربعاء',
                'خميس',
                'جمعة',

                # ------------------------------------------------------------------
                # Numbers / ordinals
                # ------------------------------------------------------------------
                'واحد',
                'اثنان',
                'اثنا',
                'اثني',
                'اثنين',
                'ثلاث',
                'ثلاثة',
                'اربع', 'أربع',
                'اربعة', 'أربعة',
                'خمس',
                'خمسة',
                'ست',
                'ستة',
                'سبع',
                'سبعة',
                'ثمان',
                'ثماني',
                'ثمانية',
                'تسع',
                'تسعة',
                'عشر',
                'عشرة',

                'اول', 'أول',
                'الاول', 'الأول',
                'الاولى', 'الأولى',
                'ثان',
                'ثاني',
                'الثاني',
                'الثانية',
                'ثالث',
                'رابع',
                'خامس',
                'سادس',
                'سابع',
                'ثامن',
                'تاسع',
                'عاشر',
                'حادي',

                'عشرون',
                'عشرين',
                'ثلاثون',
                'ثلاثين',
                'اربعون',
                'اربعين',
                'خمسون',
                'خمسين',
                'ستون',
                'ستين',
                'سبعون',
                'سبعين',
                'ثمانون',
                'ثمانين',
                'تسعون',
                'تسعين',

                'مئة',
                'مائة',
                'مئتان',
                'ثلاثمئة',
                'ثلاثمائة',
                'اربعمئة',
                'اربعمائة',
                'خمسمئة',
                'خمسمائة',
                'ستمئة',
                'ستمائة',
                'سبعمئة',
                'سبعمائة',
                'ثمانمئة',
                'ثمنمئة',
                'تسعمئة',
                'تسعمائة',

                'الف', 'ألف',
                'مليون',
                'مليار',

                # ------------------------------------------------------------------
                # Currency terms
                # ------------------------------------------------------------------
                'جنيه',
                'دولار',
                'دينار',
                'درهم',
                'ريال',
                'ليرة',
                'يورو',
                'يوان',
                'فلس',
                'قرش',
                'مليم',
                'سنتيم',
                'شيكل',
                'هللة',

                # ------------------------------------------------------------------
                # Common verbs / auxiliaries with little standalone thematic value
                # ------------------------------------------------------------------
                'كان',
                'كانت',
                'كانوا',
                'كانو',
                'وكان',
                'وكانت',

                'يكون',
                'تكون',
                'ستكون',

                'ليس',
                'وليس',

                'صار',
                'اصبح', 'أصبح',
                'اضحى', 'أضحى',
                'امسى', 'أمسى',
                'ظل',
                'بات',

                'زال',
                'مازال',
                'مايزال',
                'لازال',
                'لايزال',
                'ولايزال',

                'فعل',
                'يفعلون',
                'يفعلان',
                'تفعلون',
                'تفعلان',
                'تفعلين',

                'جعل',
                'اخذ', 'أخذ',
                'اتخذ',
                'تخذ',
                'قام',
                'رجع',
                'عاد',
                'راح',
                'رح',
                'ترك',

                'قال',
                'وقالت',
                'وقال',

                'اكد', 'أكد',
                'واكد', 'وأكد',
                'اعلنت', 'أعلنت',
                'واضاف', 'وأضاف',
                'واضافت', 'وأضافت',
                'واوضح', 'وأوضح',

                'وجد',
                'يجري',
                'يمكن',
                'ينبغي',
                'اعلم', 'أعلم',
                'تعلم',

                # ------------------------------------------------------------------
                # Dialectal / conversational fillers
                # ------------------------------------------------------------------
                'بس',
                'برضه',
                'برضو',
                'كمان',
                'شوي',
                'شوية',
                'شي',
                'شيء',

                'يعني',
                'طبعا',
                'صراحة',
                'اصلا', 'أصلا',
                'اكيد', 'أكيد',
                'حقا',
                'فعلا',

                'طيب',
                'خلاص',
                'اه',
                'اها',
                'أها',
                'اوه',
                'أوه',

                'ايه',
                'إيه',
                'هيك',
                'هيج',

                'دحين',
                'دلوقتي',
                'هسه',
                'الحين',

                'لسا',
                'لسه',

                'مو',
                'موش',
                'مش',
                'مفيش',
                'مافي',
                'مافيش',

                'عايز',
                'عاوز',
                'ابغى',
                'تبغى',
                'ابي',
                'بدي',
                'بدك',
                'حاب',

                'عشان',
                'علشان',
                'مشان',
                'علاش',

                'زي',
                'كدا',
                'كده',
                'كذا',

                'دا',
                'ده',
                'دي',
                'ديه',

                'يلي',
                'اللي',

                'وايد',

                # ------------------------------------------------------------------
                # Formulaic expressions / discourse fillers
                # ------------------------------------------------------------------
                'الله',
                'اللهم',
                'والله',
                'يارب',
                'ربي',
                'امين',
                'آمين',

                'الحمد',
                'سبحان',
                'تعالى',
                'صلى',
                'وسلم',

                'انشاءالله',
                'انشاء',
                'ماشاءالله',

                'اهلا',
                'أهلا',
                'لبيك',

                'حمدا',
                'شكرا',

                # ------------------------------------------------------------------
                # Generic discourse words
                # ------------------------------------------------------------------
                'امر', 'أمر',
                'حاجة',
                'حاجه',

                'مرة',

                'بسبب',
                'بشكل',

                'اطار', 'إطار',

                'اعادة', 'إعادة',

                'بداية',
                'ابتدا',

                'اصل', 'أصل',

                'نفس',

                'شخصا',

                # ------------------------------------------------------------------
                # Titles / generic names that often add little to a word cloud
                # ------------------------------------------------------------------
                'عبد',
                'ابن',
                'بنت',
                'سيد',
                'دكتور',

                # ------------------------------------------------------------------
                # Miscellaneous traditional Arabic stopword-list items
                # ------------------------------------------------------------------
                'االا',
                'االتى',
                'اب',
                'ابدا',
                'ابو',
                'ابين',

                'اجل',
                'احدى',

                'اخ',
                'اخبر',
                'اخو',

                'اخلولق',

                'اوشك',
                'اولاء',

                'اى',
                'ايا',
                'ايان',

                'بؤسا',
                'بئس',
                'باسم',
                'باش',
                'بان',
                'بخ',
                'بد',
                'بدلا',
                'بعيدا',
                'بله',
                'بمابكم',
                'بن',
                'بيد',

                'تان',
                'تانك',
                'تبدل',
                'تحول',
                'تعسا',
                'تلقاء',
                'ته',
                'تي',
                'تين',
                'تينك',

                'ثمة',

                'جلل',
                'جير',

                'حا',
                'حاشا',
                'حاي',
                'حبذا',
                'حبيب',
                'حجا',
                'حذار',
                'حرى',
                'حم',
                'حمو',
                'حنا',
                'حوالى',

                'خال',
                'خالص',
                'خلا',
                'خلافا',

                'دواليك',
                'دول',
                'دونك',

                'ذا',
                'ذات',
                'ذه',
                'ذهب',
                'ذو',
                'ذي',
                'ذيت',

                'رب',
                'رويدك',
                'ريث',

                'زعم',
                'زود',

                'ساء',
                'سحقا',
                'سرا',
                'سرعان',
                'سقى',
                'سمعا',

                'شاء',
                'شبه',
                'شتان',

                'صبر',
                'صبرا',
                'صدقا',
                'صفر',
                'صه',

                'طرا',
                'طفق',
                'طق',

                'ظن',

                'عد',
                'عدا',
                'عدس',
                'عدم',
                'عسى',
                'عل',
                'علق',
                'عوض',
                'عيانا',

                'غادر',
                'غين',

                'فان',
                'فانه',
                'فرادى',
                'فضلا',
                'فكان',
                'فلان',
                'فما',
                'فهو',
                'فهى',
                'فهي',
                'فو',

                'قاطبة',
                'قط',
                'قلما',

                'كاد',
                'كاي',
                'كاين',
                'كخ',
                'كرب',
                'كسا',
                'كلم',
                'كن',
                'كنت',
                'كى',
                'كيت',

                'لات',
                'لاسيما',
                'لان',
                'لدن',
                'لدى',
                'لدي',
                'لديه',
                'لعل',
                'لعمر',
                'لقاء',
                'للامم',

                'ماانفك',
                'مابرح',
                'مادام',
                'مافتئ',
                'ماهو',

                'مب',
                'معاذ',

                'مكانك',
                'مكانكم',
                'مكانكما',
                'مكانكن',

                'مما',

                'مه',
                'مهما',

                'نا',
                'ناس',
                'الناس',
                'نبا',
                'نخ',
                'نعم',
                'نيف',

                'ها',
                'هاك',
                'هب',
                'هج',
                'هكذا',
                'هلا',
                'هلم',
                'همه',
                'هنا',
                'هناك',
                'هنالك',
                'هيا',
                'هيهات',

                'وا',
                'وابو',
                'وان',
                'وانا',
                'وانت',
                'واها',
                'وبين',
                'وثي',
                'ورد',
                'وعلى',
                'وفي',
                'وقد',
                'وقف',
                'وكل',
                'ولا',
                'ولم',
                'ولن',
                'وله',
                'وما',
                'ومع',
                'ومن',
                'وهب',
                'وهم',
                'وهو',
                'وهى',
                'وهي',
                'وي',

                'يا',
                'ياخي',
                'يمين',
                'ين'
            }
        # 3️⃣ English stop words
        self.stop_words_en = {
            # articles / conjunctions / prepositions
            'a','an','the','and','or','but','if','then','than','so','as','nor','yet','because','while','until','unless','although','though',
            'by','for','of','to','in','on','at','from','into','onto','about','over','under','after','before','between','through','during',
            'with','without','within','out','up','down','off','above','below','against','among','around','across','along','behind','per','via',
            # be / do / have / modals
            'is','am','are','was','were','be','been','being','do','does','did','doing','done',
            'has','have','had','having','will','would','shall','should','can','could','may','might','must','ought',
            # pronouns / determiners
            'i','you','he','she','we','they','me','him','her','us','them','it','its','itself',
            'my','your','his','our','their','mine','yours','ours','theirs','hers',
            'myself','yourself','himself','herself','ourselves','themselves','yourselves',
            'this','that','these','those','there','here','where','when','why','how','which','who','whom','whose','what',
            'all','any','both','each','few','more','most','other','some','such','only','own','same','every','either','neither',
            'much','many','several','another','none','nothing','something','anything','everything','someone','anyone','everyone',
            # negation / adverbs / fillers
            'not','no','very','too','also','just','even','still','already','again','ever','never','always','often','now','then','once',
            'really','quite','almost','well','yes','ok','okay','oh','lol','omg','pls','plz','thanks','thank',
            # common light verbs
            'get','got','go','going','goes','went','make','made','know','see','say','said','says','let','like','want','need','one',
            # contractions (with and without apostrophe, since tokenizers differ)
            "it's","i'm","don't","can't","won't","isn't","aren't","wasn't","doesn't","didn't","that's","you're","we're","they're","i've","there's",
            'its','im','dont','cant','wont','isnt','arent','wasnt','doesnt','didnt','thats','youre','were','theyre','ive','theres','u','ur',
            # social-media / web noise
            'rt','amp','http','https','www','com','org','net','html','bit','ly',
            'facebook','video','photo','post','posted','shared','page','live','watch','link','click',
        }

        # 4️⃣ Union stop words
        self.stop_words = set(self.stop_words_ar) | set(self.stop_words_en)

        # 5️⃣ Precompiled regex helpers
        self._re_urls    = re.compile(r'http\S+|www\S+|https\S+')
        self._re_email   = re.compile(r'\S+@\S+')
        self._re_mention = re.compile(r'@\w+')
        self._re_hashtag = re.compile(r'#(\w+)')
        self._re_diac    = re.compile(r'[ًٌٍَُِّْـ]')
        self._re_alef    = re.compile('[إأآٱ]')

        # 6️⃣ Pre-clean messages once
        self.df['_clean_msg'] = (
            self.df[self.message_column]
            .fillna('')
            .astype(str)
            .apply(self.clean_text)
        )

        # 7️⃣ Keyword index (built once, reused)
        self._kw_base_df = None
        self._n_posts = len(self.df)
        self._index_min_length = None
        self._index_remove_numbers = None
        # default index (can be rebuilt with other settings if needed)
        self._build_keyword_index(min_length=2, remove_numbers=True)

    # -------- cleaning ----------
    def clean_text(self, text: str) -> str:
        """
        Normalizes text and keeps:
          - Arabic only, if the dataset has no English
          - Arabic + English, if English is detected in the dataset
        English tokens are lowercased for stopword matching.
        """
        import re
        import pandas as pd
        if pd.isna(text):
            return ""
        t = str(text)

        # Remove URLs, emails, mentions; un-hashtag
        t = self._re_urls.sub('', t)
        t = self._re_email.sub('', t)
        t = self._re_mention.sub('', t)
        t = self._re_hashtag.sub(r'\1', t)

        # Arabic normalization
        t = self._re_alef.sub('ا', t)   # normalize alef variants
        t = t.replace('ى', 'ي')
        t = self._re_diac.sub('', t)

        # Case-normalize English
        t = t.lower()

        # Keep allowed scripts
        if self.contains_english:
            # keep Arabic letters, English letters, digits and underscore/space
            t = re.sub(r'[^\w\s\u0600-\u06FFA-Za-z]', ' ', t)
        else:
            # keep Arabic letters only (plus digits/underscore/space)
            t = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', t)

        # Collapse spaces
        t = re.sub(r'\s+', ' ', t).strip()
        return t

    # -------- core helpers ----------
    @staticmethod
    def _normalize_token(tok: str) -> str:
        # English already lowercased in clean_text; Arabic unaffected by lower()
        return tok.strip()

    def _is_stopword(self, tok: str) -> bool:
        return tok in self.stop_words

    # -------- keyword index (one-time heavy work) ----------
    def _build_keyword_index(self, min_length: int = 2, remove_numbers: bool = True):
        """
        Build a base keyword table ONCE:
          - Keyword (stem)
          - Frequency (token count)
          - Post_Freq (in how many posts it appears)
          - Percentage (of all token occurrences)
          - Post_Percentage (of posts)
        If called again with the same min_length/remove_numbers, it does nothing.
        """
        import pandas as pd
        from collections import Counter

        # If already built with the same parameters, skip
        if (
            self._kw_base_df is not None and
            self._index_min_length == min_length and
            self._index_remove_numbers == remove_numbers
        ):
            return

        self._index_min_length = min_length
        self._index_remove_numbers = remove_numbers

        n_posts = len(self.df)
        self._n_posts = n_posts

        if n_posts == 0:
            self._kw_base_df = pd.DataFrame(
                columns=['Keyword', 'Frequency', 'Percentage',
                         'Post_Freq', 'Post_Percentage']
            )
            return

        print("⚙️ Building global keyword index (one-time, with stemming)...")

        global_counts: Counter = Counter()
        doc_freq: Counter = Counter()

        for txt in self.df['_clean_msg'].fillna('').astype(str):
            words = txt.split()
            tokens_in_post = set()

            for w in words:
                w = self._normalize_token(w)

                if len(w) < min_length:
                    continue
                if remove_numbers and w.isdigit():
                    continue
                if self._is_stopword(w):
                    continue

                stem = cached_safe_stem_token(w)
                if not stem:
                    continue
                if self._is_stopword(stem):
                    continue

                global_counts[stem] += 1
                tokens_in_post.add(stem)

            for tok in tokens_in_post:
                doc_freq[tok] += 1

        if not global_counts:
            print("⚠️ No keywords found when building the index.")
            self._kw_base_df = pd.DataFrame(
                columns=['Keyword', 'Frequency', 'Percentage',
                         'Post_Freq', 'Post_Percentage']
            )
            return

        df_kw = pd.DataFrame(
            [{'Keyword': k, 'Frequency': v} for k, v in global_counts.items()]
        )
        total_freq = df_kw['Frequency'].sum()
        df_kw['Percentage'] = df_kw['Frequency'] / total_freq * 100
        df_kw['Post_Freq'] = df_kw['Keyword'].map(lambda k: doc_freq.get(k, 0)).astype(int)
        df_kw['Post_Percentage'] = df_kw['Post_Freq'] / n_posts * 100

        df_kw = df_kw.sort_values('Frequency', ascending=False)

        self._kw_base_df = df_kw

        print(f"✅ Keyword index built with {len(df_kw)} unique stems from {n_posts} posts.")

    # -------- extractors ----------
    def extract_keywords_quick(
        self,
        min_length: int = 2,
        remove_numbers: bool = True,
        min_frequency: int | None = None,
        top_n: int | None = None,
        max_keywords: int | None = None,
    ):
        """
        Non-interactive version for dashboard options (Top keywords, etc.).
        Uses the precomputed keyword index (fast).
        """
        return self._extract_keywords_core(
            min_length=min_length,
            remove_numbers=remove_numbers,
            min_frequency=min_frequency,
            top_n=top_n,
            min_percentage=None,
            min_post_percentage=None,
            max_keywords=max_keywords,
        )

    def _extract_keywords_core(
        self,
        min_length: int = 2,
        remove_numbers: bool = True,
        min_frequency: int | None = None,
        top_n: int | None = None,
        min_percentage: float | None = None,
        min_post_percentage: float | None = None,
        max_keywords: int | None = None,
    ):
        import pandas as pd

        if self.message_column not in self.df.columns:
            print(f"❌ Column '{self.message_column}' not found")
            return None

        if '_clean_msg' not in self.df.columns:
            # safety: rebuild if someone dropped it
            self.df['_clean_msg'] = (
                self.df[self.message_column]
                .fillna('')
                .astype(str)
                .apply(self.clean_text)
            )

        # Ensure keyword index exists and matches min_length/remove_numbers
        self._build_keyword_index(min_length=min_length, remove_numbers=remove_numbers)

        if self._kw_base_df is None or self._kw_base_df.empty:
            print("⚠️ Keyword index is empty.")
            return pd.DataFrame(columns=['Keyword', 'Frequency', 'Percentage',
                                         'Post_Freq', 'Post_Percentage'])

        df_kw = self._kw_base_df.copy()
        n_posts = self._n_posts if self._n_posts else len(self.df)

        # ---------- FILTERS ----------
        if min_frequency is not None:
            df_kw = df_kw[df_kw['Frequency'] >= min_frequency]
        if min_percentage is not None:
            df_kw = df_kw[df_kw['Percentage'] >= min_percentage]
        if min_post_percentage is not None:
            df_kw = df_kw[df_kw['Post_Percentage'] >= min_post_percentage]

        if df_kw.empty:
            print("⚠️ All keywords were filtered out with the chosen thresholds.")
            return pd.DataFrame(columns=['Keyword', 'Frequency', 'Percentage',
                                         'Post_Freq', 'Post_Percentage'])

        # ---------- SORT & LIMIT ----------
        df_kw = df_kw.sort_values('Frequency', ascending=False)

        if max_keywords is not None:
            df_kw = df_kw.head(max_keywords)
        if top_n is not None:
            df_kw = df_kw.head(top_n)

        print(
            f"✅ Selected {len(df_kw)} keywords from {n_posts} posts (using cached index)"
        )
        return df_kw

    def extract_keywords(
        self,
        min_length: int = 2,
        remove_numbers: bool = True,
        user_interactive: bool = False,
        min_frequency: int | None = None,
        top_n: int | None = None,
        min_percentage: float | None = None,
        min_post_percentage: float | None = None,
        max_keywords: int | None = None
    ):
        """
        Extract keywords from the dataset with flexible filtering.

        Parameters
        ----------
        min_length : int
            Minimum token length to be considered a keyword.
        remove_numbers : bool
            If True, numeric tokens are ignored.
        user_interactive : bool
            If True, ask the user (via input) to specify:
              - minimum frequency
              - minimum percentage (of all keyword occurrences)
              - minimum post percentage (of posts that contain the word)
              - top N
            Values passed as arguments are used as defaults
            if the user leaves the input blank.
        """

        # -------------------------------------------------------
        # 1) Optional interactive configuration
        # -------------------------------------------------------
        if user_interactive:
            print("\n================ Keyword Extraction Options ================\n")
            print("Customize how keywords are selected from your dataset.")
            print("You may fill one, several, or all fields. Leave any blank to skip.\n")
            print("• Minimum Frequency      → keep only keywords that appear at least X times")
            print("• Minimum Percentage     → keep only keywords that represent at least X% of all keyword occurrences")
            print("• Minimum Post %         → keep only keywords that appear in at least X% of posts")
            print("• Top N Keywords         → return only the top N most frequent keywords\n")

            # Minimum frequency
            freq_input = input(
                f"Minimum frequency (current: {min_frequency if min_frequency is not None else 'none'}), e.g., 20: "
            ).strip()
            if freq_input:
                try:
                    min_frequency = int(freq_input)
                except ValueError:
                    print("⚠️ Invalid frequency value, ignoring user input.")

            # Minimum percentage (of all keyword occurrences)
            pct_input = input(
                f"Minimum percentage of all keyword occurrences "
                f"(current: {min_percentage if min_percentage is not None else 'none'}), "
                f"e.g., 0.1 for 0.1%: "
            ).strip()
            if pct_input:
                try:
                    min_percentage = float(pct_input)
                except ValueError:
                    print("⚠️ Invalid percentage value, ignoring user input.")

            # Minimum percentage of posts
            post_pct_input = input(
                f"Minimum percentage of posts containing the keyword "
                f"(current: {min_post_percentage if min_post_percentage is not None else 'none'}), "
                f"e.g., 20 for 20% of posts: "
            ).strip()
            if post_pct_input:
                try:
                    min_post_percentage = float(post_pct_input)
                except ValueError:
                    print("⚠️ Invalid post-percentage value, ignoring user input.")

            # Top N
            top_n_input = input(
                f"Top N keywords (current: {top_n if top_n is not None else 'none'}), e.g., 300: "
            ).strip()
            if top_n_input:
                try:
                    top_n = int(top_n_input)
                except ValueError:
                    print("⚠️ Invalid Top N value, ignoring user input.")

            print("\n=============================================================\n")

        # ✅ Now pass ALL filters (including percentages) to core
        return self._extract_keywords_core(
            min_length=min_length,
            remove_numbers=remove_numbers,
            min_frequency=min_frequency,
            top_n=top_n,
            min_percentage=min_percentage,
            min_post_percentage=min_post_percentage,
            max_keywords=max_keywords,
        )

    def extract_keywords_by_category(self, category_column, min_length=2, top_n=20):
        import pandas as pd
        from collections import Counter

        if category_column not in self.df.columns or self.message_column not in self.df.columns:
            print("❌ Required columns not found")
            return None

        print(f"🔍 Extracting keywords by {category_column}...")

        # Ensure _clean_msg exists
        if '_clean_msg' not in self.df.columns:
            self.df['_clean_msg'] = (
                self.df[self.message_column]
                .fillna('')
                .astype(str)
                .apply(self.clean_text)
            )

        out = {}
        for cat in self.df[category_column].dropna().unique():
            subset = self.df[self.df[category_column] == cat]

            # ✅ Use pre-cleaned text to avoid re-normalizing
            text = ' '.join(subset['_clean_msg'].fillna('').astype(str))
            words = text.split()

            words = [
                self._normalize_token(w)
                for w in words
                if len(w) >= min_length and not w.isdigit() and not self._is_stopword(w)
            ]
            stems = [cached_safe_stem_token(w) for w in words if w]
            stems = [s for s in stems if s and not self._is_stopword(s)]

            cnt = Counter(stems).most_common(top_n)
            out[str(cat)] = pd.DataFrame(cnt, columns=['Keyword', 'Frequency'])

        print(f"✅ Extracted keywords for {len(out)} categories")
        return out

    def extract_keywords_with_engagement(self, min_length=2, top_n=50):
        import pandas as pd

        if self.message_column not in self.df.columns or 'Total Interactions' not in self.df.columns:
            print("❌ Required columns not found")
            return None

        print("🔍 Extracting keywords with engagement metrics...")

        # Ensure _clean_msg exists
        if '_clean_msg' not in self.df.columns:
            self.df['_clean_msg'] = (
                self.df[self.message_column]
                .fillna('')
                .astype(str)
                .apply(self.clean_text)
            )

        agg = {}

        for _, row in self.df[['_clean_msg', 'Total Interactions']].dropna().iterrows():
            words = str(row['_clean_msg']).split()
            toks = []
            for w in words:
                w = self._normalize_token(w)
                if len(w) < min_length:
                    continue
                if w.isdigit():
                    continue
                if self._is_stopword(w):
                    continue
                stem = cached_safe_stem_token(w)
                if not stem or self._is_stopword(stem):
                    continue
                toks.append(stem)

            stems = {w for w in toks if w}  # unique per post
            for w in stems:
                d = agg.setdefault(w, {'count': 0, 'total': 0})
                d['count'] += 1
                d['total'] += row['Total Interactions']

        if not agg:
            print("⚠️ No keywords found.")
            return None

        rows = [
            {
                'Keyword': k,
                'Frequency': v['count'],
                'Avg_Engagement': v['total'] / v['count'],
                'Total_Engagement': v['total']
            }
            for k, v in agg.items()
        ]
        dfk = pd.DataFrame(rows).sort_values('Frequency', ascending=False).head(top_n)
        print(f"✅ Extracted {len(dfk)} keywords with engagement metrics")
        return dfk

    def extract_ngrams(self, n=2, min_freq=5, top_n=50):
        import pandas as pd
        from collections import Counter

        if self.message_column not in self.df.columns:
            print(f"❌ Column '{self.message_column}' not found")
            return None
        print(f"🔍 Extracting {n}-grams...")

        # Ensure _clean_msg exists
        if '_clean_msg' not in self.df.columns:
            self.df['_clean_msg'] = (
                self.df[self.message_column]
                .fillna('')
                .astype(str)
                .apply(self.clean_text)
            )

        all_text = ' '.join(self.df['_clean_msg'].tolist())
        words = all_text.split()

        words = [
            cached_safe_stem_token(self._normalize_token(w))
            for w in words
            if len(w) >= 2 and not w.isdigit() and not self._is_stopword(w)
        ]
        words = [w for w in words if w]

        grams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
        cnt = Counter(grams)
        items = [(g, c) for g, c in cnt.items() if c >= min_freq]
        items.sort(key=lambda x: x[1], reverse=True)
        df = pd.DataFrame(items[:top_n], columns=['Phrase', 'Frequency'])
        print(f"✅ Extracted {len(df)} {n}-grams")
        return df

    def add_custom_stop_words(self, words):
        if isinstance(words, str):
            words = [words]
        normalized = [str(w).lower() for w in words]
        self.stop_words.update(set(normalized))
        print(f"✅ Added {len(words)} custom stop words")

    def compute_tfidf_keywords(self, top_n=20, min_df=5):
        """
        Real TF-IDF over the corpus (one document = one post), using the same
        clean_text() preprocessing already applied for frequency-based extraction.
        Reports both TF-IDF ranking and document frequency, so Tables 3/4 can
        show what TF-IDF actually contributes vs. raw frequency.
        """
        from sklearn.feature_extraction.text import TfidfVectorizer

        texts = self.df[self.message_column].fillna('').astype(str).apply(self.clean_text).tolist()
        vectorizer = TfidfVectorizer(min_df=min_df, token_pattern=r'(?u)\b\w+\b')
        tfidf_matrix = vectorizer.fit_transform(texts)
        vocab = np.array(vectorizer.get_feature_names_out())

        mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
        top_idx = mean_tfidf.argsort()[::-1][:top_n]

        return pd.DataFrame({
            'keyword': vocab[top_idx],
            'mean_tfidf': mean_tfidf[top_idx],
            'doc_frequency': np.asarray((tfidf_matrix[:, top_idx] > 0).sum(axis=0)).ravel(),
        })


In [ ]:
# =============================================================================
# ArabicKeywordVisualizer
# =============================================================================
class ArabicKeywordVisualizer:
    FONT_AR_PATH = "/content/NotoNaskhArabic-Regular.ttf"   # Arabic (Noto Naskh)
    FONT_LATIN_PATH = "/content/DejaVuSans.ttf"             # Latin (DejaVu Sans)
    ARABIC_REGEX = r'[\u0600-\u06FF]+'                      # Built-in Arabic range
    FONT_AR_URL = ("https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/"
                   "NotoNaskhArabic/NotoNaskhArabic-Regular.ttf")

    # ------------------------------ Helpers ------------------------------
    @staticmethod
    def _reshape_arabic(text: str) -> str:
        """Join Arabic letters + RTL order for Pillow. Skipped if Pillow has libraqm
        (it shapes Arabic itself, and shaping twice breaks the text)."""
        import re
        from PIL import features
        text = str(text)
        if not re.search(r'[\u0600-\u06FF]', text) or features.check('raqm'):
            return text
        try:
            import arabic_reshaper
            from bidi.algorithm import get_display
        except ImportError:
            raise ImportError("Arabic shaping needs: pip install arabic-reshaper python-bidi")
        return get_display(arabic_reshaper.reshape(text))

    @staticmethod
    def _normalize_arabic(w: str) -> str:
        """Light normalization so spelling variants merge before counting."""
        import re
        w = re.sub(r'[\u064B-\u0652\u0670\u0640]', '', w)   # tashkeel + tatweel
        w = re.sub('[إأآ]', 'ا', w)
        # Optional, more aggressive (decide per project):
        # w = w.replace('ى', 'ي').replace('ة', 'ه')
        return w

    @staticmethod
    def _script_of(w: str):
        """Assign a keyword to ONE script (dominant letters) -> no double counting."""
        import re
        n_ar = len(re.findall(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]', w))
        n_en = len(re.findall(r'[A-Za-z]', w))
        if n_ar == 0 and n_en == 0:
            return None
        return 'ar' if n_ar >= n_en else 'en'

    @staticmethod
    def _ensure_font(path: str, url: str):
        """Download font if missing (silent if exists)."""
        import os, urllib.request
        if not os.path.exists(path):
            print(f"⏬ Downloading font: {os.path.basename(path)} ...")
            urllib.request.urlretrieve(url, path)
        return path

    # ------------------------------ BAR: Top Keywords ------------------------------
    @staticmethod
    def plot_top_keywords(keywords_df, top_n=20, title="Top Keywords",
                          colorscale="Teal", log_color=True):
        import numpy as np
        import plotly.graph_objects as go

        if keywords_df is None or len(keywords_df) == 0:
            print("❌ No keywords to visualize"); return

        topk = keywords_df.head(top_n).copy()
        freq = topk['Frequency'].astype(float)

        # Color value: log scale spreads the gradient across all bars
        c = np.log10(freq) if log_color else freq
        lo, hi = c.min(), c.max()
        cmin = lo - 0.35 * (hi - lo)   # keeps the lightest bar from turning white

        fig = go.Figure(go.Bar(
            y=topk['Keyword'],
            x=freq,
            orientation='h',
            text=freq,
            texttemplate='%{x:,.0f}',
            textposition='outside',
            cliponaxis=False,
            marker=dict(color=c, colorscale=colorscale,
                        cmin=cmin, cmax=hi, showscale=False)
        ))
        fig.update_layout(
            template='plotly_white',
            title={'text': f'{title} (Top {top_n})', 'x': 0.5, 'xanchor': 'center'},
            xaxis_title='Total mentions',
            yaxis_title='Keyword',
            height=max(600, top_n * 30),
            yaxis={'categoryorder': 'total ascending'},
            margin=dict(l=120, r=80),
            font=dict(family="Noto Naskh Arabic, DejaVu Sans, Arial", size=16)
        )
        fig.update_xaxes(range=[0, freq.max() * 1.12], gridcolor='#E5E5E5')
        fig.show()

    # ------------------------------ WORD CLOUD: Language-Aware ------------------------------
    @staticmethod
    def plot_wordcloud(
        keywords_df,
        title="Keywords Word Cloud",
        width=1600,              # render resolution (sharp for export)
        height=900,
        display_width=700,       # on-screen size per panel
        colormap='viridis',
        max_words=150,
        normalize_arabic=True,
        random_state=42,         # same layout every run
        save_prefix=None,        # e.g. "figures/wc" -> wc_arabic.png, wc_english.png
    ):
        """
        Language-aware word clouds: Arabic-only, English-only, or both side by side.
        Arabic is normalized, shaped and kept horizontal, rendered with Noto Naskh Arabic.
        """
        import pandas as pd
        import plotly.graph_objects as go
        from pathlib import Path
        from plotly.subplots import make_subplots
        from wordcloud import WordCloud

        V = ArabicKeywordVisualizer

        if keywords_df is None or len(keywords_df) == 0:
            print("❌ No keywords to visualize"); return
        if not {'Keyword', 'Frequency'}.issubset(keywords_df.columns):
            print("❌ DataFrame must have columns: 'Keyword' and 'Frequency'"); return

        # ---- Clean + aggregate (duplicates summed, not overwritten) ----
        df = keywords_df[['Keyword', 'Frequency']].copy()
        df['Frequency'] = pd.to_numeric(df['Frequency'], errors='coerce')
        df = df[df['Frequency'] > 0].dropna()
        df['Keyword'] = df['Keyword'].astype(str).str.strip()
        df = df[df['Keyword'].str.fullmatch(r'[\u0621-\u064A]{2,}|[A-Za-z]{2,}')]
        if normalize_arabic:
            df['Keyword'] = df['Keyword'].map(V._normalize_arabic)
        df['script'] = df['Keyword'].map(V._script_of)
        df = df.dropna(subset=['script'])

        buckets = {}
        for script, g in df.groupby('script'):
            freq = g.groupby('Keyword')['Frequency'].sum().sort_values(ascending=False)
            if max_words:
                freq = freq.head(max_words)
            buckets[script] = freq.to_dict()

        if not buckets:
            print("⚠️ No Arabic or English tokens detected."); return

        # ---- One cloud per script ----
        panels = []
        for script, label in (('ar', 'Arabic'), ('en', 'English')):
            if script not in buckets:
                continue
            freq = buckets[script]
            is_ar = script == 'ar'
            if is_ar:
                V._ensure_font(V.FONT_AR_PATH, V.FONT_AR_URL)
                freq = {V._reshape_arabic(w): f for w, f in freq.items()}

            wc = WordCloud(
                font_path=V.FONT_AR_PATH if is_ar else None,
                width=width, height=height,
                background_color='white', mode='RGB',
                max_words=len(freq), colormap=colormap,
                relative_scaling=0.5, min_font_size=10,
                prefer_horizontal=1.0 if is_ar else 0.9,  # rotated Arabic is hard to read
                collocations=False, margin=4,
                random_state=random_state,
            ).generate_from_frequencies(freq)
            panels.append((label, wc))

            if save_prefix:
                out = Path(f"{save_prefix}_{label.lower()}.png")
                out.parent.mkdir(parents=True, exist_ok=True)
                wc.to_file(str(out))
                print(f"💾 Saved {out}")

        # ---- Display ----
        n = len(panels)
        fig = make_subplots(
            rows=1, cols=n, horizontal_spacing=0.02,
            subplot_titles=[label for label, _ in panels] if n > 1 else None)
        for i, (_, wc) in enumerate(panels, start=1):
            fig.add_trace(go.Image(z=wc.to_array(), hoverinfo='skip'), row=1, col=i)

        fig.update_xaxes(visible=False)
        fig.update_yaxes(visible=False)
        fig.update_layout(
            title={'text': title, 'x': 0.5, 'xanchor': 'center'},
            width=display_width * n,
            height=int(display_width * height / width) + 90,
            margin=dict(l=10, r=10, t=70, b=10),
        )
        fig.show()

    # ------------------------------ Scatter: Keywords vs Engagement ------------------------------
    @staticmethod
    def plot_keywords_with_engagement(keywords_df, top_n=20, colorscale="Teal",
                                      log_x=True):
        import numpy as np
        import plotly.graph_objects as go

        if keywords_df is None or len(keywords_df) == 0:
            print("❌ No keywords to visualize"); return
        if 'Avg_Engagement' not in keywords_df.columns:
            print("❌ No engagement data available"); return

        # Works with either column name (post count in this DataFrame)
        x_col = 'Post_Freq' if 'Post_Freq' in keywords_df.columns else 'Frequency'
        topk = keywords_df.sort_values(x_col, ascending=False).head(top_n).copy()

        # Bubble size: sqrt scale with a floor, so small keywords stay visible
        share = topk['Total_Engagement'] / topk['Total_Engagement'].max()
        sizes = 10 + 40 * np.sqrt(share)

        # Color: single hue; cmin keeps the lowest point from turning near-white
        c = topk['Avg_Engagement']
        cmin = c.min() - 0.35 * (c.max() - c.min())

        # Alternate label position to reduce overlap in the crowded cluster
        positions = ['top center' if i % 2 == 0 else 'bottom center'
                     for i in range(len(topk))]

        fig = go.Figure(go.Scatter(
            x=topk[x_col], y=topk['Avg_Engagement'],
            mode='markers+text', text=topk['Keyword'],
            textposition=positions,
            textfont=dict(family="Noto Naskh Arabic, DejaVu Sans, Arial", size=14),
            customdata=np.stack([topk['Total_Engagement']], axis=-1),
            hovertemplate=("<b>%{text}</b><br>Posts: %{x:,.0f}"
                           "<br>Avg interactions: %{y:,.1f}"
                           "<br>Total interactions: %{customdata[0]:,.0f}<extra></extra>"),
            marker=dict(
                size=sizes, sizemode='diameter',
                color=c, colorscale=colorscale, cmin=cmin, cmax=c.max(),
                opacity=0.85, line=dict(width=1, color='white'),
                showscale=True,
                colorbar=dict(title="Avg interactions<br>per post")
            )
        ))
        fig.update_layout(
            template='plotly_white',
            title={'text': 'Keywords: number of posts vs. average interactions per post',
                   'x': 0.5, 'xanchor': 'center'},
            xaxis_title='Number of posts containing keyword' + (' (log scale)' if log_x else ''),
            yaxis_title='Average interactions per post',
            height=700,
            margin=dict(t=90, b=90),
            font=dict(family="Noto Naskh Arabic, DejaVu Sans, Arial", size=16),
            annotations=[dict(text="Bubble size = total interactions",
                              xref='paper', yref='paper', x=1, y=-0.14,
                              showarrow=False, font=dict(size=13, color='#666'))]
        )
        fig.update_xaxes(type='log' if log_x else 'linear',
                         tickformat='~s', gridcolor='#E5E5E5')
        fig.update_yaxes(gridcolor='#E5E5E5')
        fig.show()

    # ------------------------------ Grouped Bars: By Category ------------------------------
    @staticmethod
    def plot_keywords_by_category(category_keywords, top_n_per_category=10,
                                  cols=3, colors=None, share_x=False):
        import math
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots

        if not category_keywords:
            print("❌ No category keywords to visualize"); return

        cats = [(str(c), kdf.head(top_n_per_category))
                for c, kdf in category_keywords.items()
                if kdf is not None and len(kdf) > 0]
        if not cats:
            print("❌ No data to display"); return

        # One solid color per category, no yellow
        colors = colors or ["#1B7F79", "#8C2D4B", "#2E6F9E", "#5B4B8A",
                            "#2F6B4F", "#C0562F", "#4A5568", "#8A5A9E"]

        n = len(cats)
        cols = min(cols, n)
        rows = math.ceil(n / cols)
        v_space = 0 if rows == 1 else min(0.12, 0.5 / (rows - 1))

        fig = make_subplots(
            rows=rows, cols=cols,
            subplot_titles=[c for c, _ in cats],
            horizontal_spacing=0.12, vertical_spacing=v_space,
            shared_xaxes=share_x
        )

        global_max = max(d['Frequency'].max() for _, d in cats)

        for i, (cat, d) in enumerate(cats):
            r, c = i // cols + 1, i % cols + 1
            fig.add_trace(go.Bar(
                y=d['Keyword'], x=d['Frequency'],
                orientation='h',
                text=d['Frequency'], texttemplate='%{x:,.0f}',
                textposition='outside', cliponaxis=False,
                marker=dict(color=colors[i % len(colors)]),
                hovertemplate="<b>%{y}</b><br>Total mentions: %{x:,.0f}<extra></extra>",
                showlegend=False
            ), row=r, col=c)

            top = global_max if share_x else d['Frequency'].max()
            fig.update_xaxes(range=[0, top * 1.3], row=r, col=c)

        fig.update_layout(
            template='plotly_white',
            title={'text': f'Top {top_n_per_category} keywords by category (total mentions)',
                   'x': 0.5, 'xanchor': 'center'},
            height=rows * max(380, top_n_per_category * 34) + 80,
            width=cols * 480,
            margin=dict(t=100, l=20, r=20),
            font=dict(family="Noto Naskh Arabic, DejaVu Sans, Arial", size=16)
        )
        fig.update_yaxes(categoryorder='total ascending', automargin=True)
        fig.update_xaxes(gridcolor='#E5E5E5', tickformat='~s')
        fig.update_xaxes(title_text='Total mentions', row=rows)
        fig.show()

    # ------------------------------ Orchestrator ------------------------------
    @staticmethod
    def analyze_arabic_keywords(df,
                                message_column='Message',
                                top_n=50,
                                precomputed_keywords=None,
                                save_prefix=None):
        print("=" * 80)
        print("🔍 ARABIC KEYWORD ANALYSIS")
        print("=" * 80)

        extractor = ArabicKeywordExtractor(df, message_column)

        print("\n1️⃣ Extracting top keywords...")
        if precomputed_keywords is not None and not precomputed_keywords.empty:
            keywords = precomputed_keywords
            print(f"✅ Reusing precomputed {len(keywords)} keywords")
        else:
            keywords = extractor.extract_keywords_quick(max_keywords=top_n)

        if keywords is not None and not keywords.empty:
            print("\nTop 10 Keywords:")
            print(keywords.head(10).to_string(index=False))
            ArabicKeywordVisualizer.plot_top_keywords(keywords, top_n=20, title="Top Keywords")
            ArabicKeywordVisualizer.plot_wordcloud(
                keywords, title="Top Keywords Word Cloud", save_prefix=save_prefix)

        print("\n2️⃣ Extracting keywords with engagement...")
        kw_eng = extractor.extract_keywords_with_engagement(top_n=top_n)
        if kw_eng is not None and not kw_eng.empty:
            print("\nTop 10 Keywords by Engagement:")
            print(kw_eng.head(10).to_string(index=False))
            ArabicKeywordVisualizer.plot_keywords_with_engagement(kw_eng)

        print("\n3️⃣ Extracting common phrases (bigrams)...")
        bigrams = extractor.extract_ngrams(n=2, min_freq=3, top_n=30)
        if bigrams is not None and not bigrams.empty:
            print("\nTop 10 Phrases:")
            print(bigrams.head(10).to_string(index=False))

        if 'Type' in df.columns:
            print("\n4️⃣ Extracting keywords by content type...")
            by_type = extractor.extract_keywords_by_category('Type', top_n=15)
            if by_type:
                ArabicKeywordVisualizer.plot_keywords_by_category(by_type)
        else:
            by_type = None

        print("\n✅ KEYWORD ANALYSIS COMPLETE!")
        return {
            'keywords': keywords,
            'keywords_engagement': kw_eng,
            'bigrams': bigrams,
            'keywords_by_type': by_type
        }
results = ArabicKeywordVisualizer.analyze_arabic_keywords(
    df1,                           # your posts DataFrame
    message_column='Message',
    top_n=50,
    save_prefix="figures/wc"      # optional: saves wc_arabic.png / wc_english.png
)

# Re-draw only the word cloud later, reusing the extracted keywords
#ArabicKeywordVisualizer.plot_wordcloud(results['keywords'], max_words=100)

# Skip re-extraction on later runs
#results = ArabicKeywordVisualizer.analyze_arabic_keywords(
 #   df, precomputed_keywords=results['keywords'])

In [ ]:
# Make sure interactions are numeric (skip if already done)
df['Total Interactions'] = pd.to_numeric(
    df['Total Interactions'].astype(str).str.replace(',', '', regex=False),
    errors='coerce').fillna(0).astype(int)

# Extract keywords with engagement, then plot
extractor = ArabicKeywordExtractor(df, 'Message')
kw_eng = extractor.extract_keywords_with_engagement(top_n=50)

ArabicKeywordVisualizer.plot_keywords_with_engagement(kw_eng, top_n=30)

In [ ]:
df.columns

In [ ]:
import re
import numpy as np
import pandas as pd

# ---------- 0. clean engagement column once ----------
ti_raw = df['Total Interactions']
print("dtype of Total Interactions:", ti_raw.dtype)
df['TI'] = pd.to_numeric(ti_raw.astype(str).str.replace(',', '', regex=False),
                         errors='coerce')
print("non-numeric / missing after cleaning:", df['TI'].isna().sum(), "\n")

N = len(df)

# ---------- 1. corpus overview ----------
print("=== 1. CORPUS OVERVIEW ===")
print(f"Posts:              {N:,}")
print(f"Unique pages:       {df['Facebook Id'].nunique():,}  (by Facebook Id)")
print(f"Unique page names:  {df['Page Name'].nunique():,}")
print(f"Total interactions: {df['TI'].sum():,.0f}")
print(f"Mean / post:        {df['TI'].mean():.1f}")
print(f"Median / post:      {df['TI'].median():.1f}")
top1 = df['TI'].nlargest(int(np.ceil(0.01 * N))).sum()
print(f"Top 1% share:       {100 * top1 / df['TI'].sum():.1f}%\n")

# ---------- 2. geographic ----------
print("=== 2. GEOGRAPHY ===")
country = df['Page Admin Top Country'].astype(str).str.strip()
country = country.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
df['country'] = country

has_c = df['country'].notna()
print(f"Posts with country:    {100 * has_c.mean():.1f}%")
print(f"Posts without country: {(~has_c).sum():,} from "
      f"{df.loc[~has_c, 'Facebook Id'].nunique():,} pages")
print(f"Distinct countries:    {df['country'].nunique()}")

ARAB_LEAGUE = {'DZ','BH','KM','DJ','EG','IQ','JO','KW','LB','LY','MR','MA',
               'OM','PS','QA','SA','SO','SD','SY','TN','AE','YE'}
arab = df['country'].isin(ARAB_LEAGUE)
print(f"Arab League share:     {100 * arab.sum() / has_c.sum():.1f}% of posts with country")

geo = (df[has_c].groupby('country')
       .agg(Posts=('TI', 'size'), Pages=('Facebook Id', 'nunique'),
            Mean=('TI', 'mean'), Median=('TI', 'median'))
       .sort_values('Posts', ascending=False))
geo['Pct'] = 100 * geo['Posts'] / has_c.sum()
print(f"Top-10 share:          {geo['Pct'].head(10).sum():.1f}%")
print(f"Egypt share:           {geo.loc['EG', 'Pct']:.1f}%\n")

MIN_POSTS = 500      # pick your threshold; report it in the paper
geo_rep = geo[geo['Posts'] >= MIN_POSTS]
print(f"Countries with >= {MIN_POSTS} posts: {len(geo_rep)}")
print(geo_rep.round(2).to_string(), "\n")
print("Mean range:", geo_rep['Mean'].idxmin(), round(geo_rep['Mean'].min(), 1),
      "to", geo_rep['Mean'].idxmax(), round(geo_rep['Mean'].max(), 1))
print("Median range:", geo_rep['Median'].idxmin(), geo_rep['Median'].min(),
      "to", geo_rep['Median'].idxmax(), geo_rep['Median'].max(), "\n")

# ---------- 3. content characteristics ----------
print("=== 3. CONTENT ===")
msg = df['Message'].fillna('').astype(str)
df['n_chars']    = msg.str.len()
df['n_hashtags'] = msg.str.count(r'#[\w\u0600-\u06FF_]+')
df['has_url']    = msg.str.contains(r'https?://|www\.', regex=True)

print(f"Length mean / median: {df['n_chars'].mean():.0f} / {df['n_chars'].median():.0f}")
print(f"Hashtags mean:        {df['n_hashtags'].mean():.2f}")
print(f"Posts with 0 hashtags:{100 * (df['n_hashtags'] == 0).mean():.1f}%")
print(f"Posts with URL:       {100 * df['has_url'].mean():.1f}%\n")

df['hashtag_bin'] = pd.cut(df['n_hashtags'], [-1, 0, 3, np.inf], labels=['0', '1-3', '4+'])
hb = df.groupby('hashtag_bin', observed=True)['TI'].agg(['size', 'mean', 'median'])
print(hb.round(1).to_string(), "\n")

# ---------- 4. consistency checks ----------
print("=== 4. CONSISTENCY CHECKS ===")
w_mean = (hb['size'] * hb['mean']).sum() / hb['size'].sum()
print(f"Weighted mean of hashtag bins = {w_mean:.1f}  (must equal {df['TI'].mean():.1f})")
g_all = df.groupby(df['country'].fillna('MISSING'))['TI'].agg(['size', 'mean'])
w_geo = (g_all['size'] * g_all['mean']).sum() / g_all['size'].sum()
print(f"Weighted mean of countries    = {w_geo:.1f}  (must equal {df['TI'].mean():.1f})")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.alpha': 0.3,
    'grid.linewidth': 0.5, 'axes.axisbelow': True,
})

# ---- bins ----
df['hashtag_bin'] = pd.cut(df['n_hashtags'], [-1, 0, 3, np.inf],
                           labels=['0', '1–3', '4+'])
df['length_bin'] = pd.cut(df['n_chars'], [-1, 100, 300, 500, 1000, np.inf],
                          labels=['0–100', '100–300', '300–500', '500–1k', '1k+'])


def panel(ax, bincol, xlabel):
    g = df.groupby(bincol, observed=True)['TI'].agg(
        n='size', mean='mean', median='median',
        q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75))
    x = np.arange(len(g))

    ax.bar(x, g['median'], width=0.62, color='#4C72B0',
           edgecolor='black', linewidth=0.5, zorder=2, label='Median')
    ax.vlines(x, g['q1'], g['q3'], color='black', linewidth=1.0, zorder=3)
    ax.scatter(x, g['mean'], marker='D', s=22, color='#C44E52',
               edgecolor='black', linewidth=0.5, zorder=4, label='Mean')

    for xi, (m, q3) in enumerate(zip(g['median'], g['q3'])):
        ax.text(xi, q3 * 1.04, f'{m:.0f}', ha='center', va='bottom', fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels([f'{lab}\n($n$={n:,})' for lab, n in zip(g.index, g['n'])],
                       fontsize=8)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylim(0, max(g['q3'].max(), g['mean'].max()) * 1.18)
    return g


fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9))

g1 = panel(axes[0], 'hashtag_bin', 'Hashtags per post')
g2 = panel(axes[1], 'length_bin',  'Message length (characters)')

axes[0].set_ylabel('Interactions per post', fontsize=9)
axes[0].legend(frameon=False, fontsize=8, loc='upper left')
axes[0].text(-0.16, 1.04, '(a)', transform=axes[0].transAxes,
             fontweight='bold', fontsize=10)
axes[1].text(-0.10, 1.04, '(b)', transform=axes[1].transAxes,
             fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('figure/content_characteristics.pdf', bbox_inches='tight')
plt.show()

print(g1.round(1).to_string(), '\n')
print(g2.round(1).to_string())

In [ ]:
len(df)

In [ ]:
import re
import numpy as np

_AR = re.compile(r'[\u0621-\u064A]')
_EN = re.compile(r'[A-Za-z]')

def detect_language(text):
    if not isinstance(text, str) or not text.strip():
        return 'Unknown'
    ar = len(_AR.findall(text))
    en = len(_EN.findall(text))
    if ar == 0 and en == 0:
        return 'Unknown'
    if en == 0:
        return 'Arabic'
    if ar == 0:
        return 'English'
    return 'Mixed'

df['Language'] = df['Message'].apply(detect_language)

print(df['Language'].value_counts())
print((df['Language'].value_counts(normalize=True) * 100).round(2))

In [ ]:
import re
import pandas as pd

def detect_language(text):
    if pd.isna(text) or not str(text).strip():
        return "Unknown"

    text = str(text)

    # Remove URLs, mentions, and hashtags symbols
    text = re.sub(r"http\S+|www\.\S+|@\w+", " ", text)
    text = text.replace("#", "")

    arabic_chars = len(re.findall(r"[\u0600-\u06FF]", text))
    english_chars = len(re.findall(r"[A-Za-z]", text))
    total_chars = arabic_chars + english_chars

    if total_chars == 0:
        return "Unknown"

    arabic_ratio = arabic_chars / total_chars
    english_ratio = english_chars / total_chars

    if arabic_ratio >= 0.80:
        return "Arabic"
    elif english_ratio >= 0.80:
        return "English"
    else:
        return "Mixed"

df["language"] = df["Message"].apply(detect_language)

language_counts = (
    df["language"]
    .value_counts(dropna=False)
    .rename_axis("Language")
    .reset_index(name="Post_Count")
)

language_counts["Percentage"] = (
    language_counts["Post_Count"] / len(df) * 100
).round(2)

print(language_counts)

In [ ]:
len(df_filtered)

In [ ]:
text_col = "Message"
import re

# Define patterns
arabic_pattern = re.compile(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF]')
english_pattern = re.compile(r'[A-Za-z]')

def detect_language(text):
    if not isinstance(text, str) or not text.strip():
        return "other"

    has_arabic = bool(arabic_pattern.search(text))
    has_english = bool(english_pattern.search(text))

    if has_arabic and has_english:
        return "mixed"
    elif has_arabic:
        return "arabic"
    elif has_english:
        return "english"
    else:
        return "other"

# Apply language detection (use .copy() to avoid warning)
df_filtered = df_filtered.copy()
df_filtered["language"] = df_filtered[text_col].apply(detect_language)

# Detailed language analysis
print("="*60)
print("LANGUAGE DISTRIBUTION ANALYSIS".center(60))
print("="*60)

language_counts = df_filtered["language"].value_counts()
total = len(df_filtered)

print(f"\n{'Language':<15} {'Posts':<15} {'Percentage':<15} {'Bar'}")
print("-"*60)
for lang, count in language_counts.items():
    pct = (count / total) * 100
    bar = "█" * int(pct / 2)  # Visual bar
    print(f"{lang:<15} {count:<15,} {pct:>6.2f}%        {bar}")

print("="*60)
print(f"Total Posts: {total:,}")

# Optional: Save language-specific subsets
arabic_only = df_filtered[df_filtered["language"] == "arabic"]
english_only = df_filtered[df_filtered["language"] == "english"]
mixed_lang = df_filtered[df_filtered["language"] == "mixed"]

print(f"\n📊 Subset Sizes:")
print(f"  • Arabic-only posts: {len(arabic_only):,}")
print(f"  • English-only posts: {len(english_only):,}")
print(f"  • Mixed-language posts: {len(mixed_lang):,}")
print(f"  • Other (no text/numbers only): {len(df_filtered[df_filtered['language'] == 'other']):,}")


In [ ]:
df=pd.read_csv('/content/drive/MyDrive/Facebook Crowdtangle Data/All Zied Download Folder/Renamed by Rafi/Categorized by Maryam/Environment/Environment/df_with_topics.csv')

In [ ]:
df.columns

In [ ]:
df['Assigned_Topic'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

# Exclude UNASSIGNED
df_valid = df[df["Assigned_Topic"] != "UNASSIGNED"].copy()

# Exactly 1000 samples, preserving topic proportions
df_sampled, _ = train_test_split(
    df_valid,
    train_size=2000,
    stratify=df_valid["Assigned_Topic"],
    random_state=42
)

df_sampled = df_sampled.reset_index(drop=True)

print("Total:", len(df_sampled))
print(df_sampled["Assigned_Topic"].value_counts())

In [ ]:
df_sampled.to_csv('/content/drive/MyDrive/Facebook Crowdtangle Data/All Zied Download Folder/Renamed by Rafi/Categorized by Maryam/Environment/Environment/df_sampled.csv')

In [ ]:
import pandas as pd

def stratified_sample_equal(
    df,
    topic_col='Assigned_Topic',
    n_per_topic=150,          # target sample size per topic; adjust to your annotation budget
    random_state=42
):
    """
    Equal/targeted allocation stratified sample: draws min(n_per_topic, group_size)
    rows per topic. Small topics (below n_per_topic) return all their rows.
    Adds N_j (true population count) and weight_j = N_j / n_j for later
    reweighting when computing pooled/overall metrics.
    """
    def sample_group(g):
        n = min(n_per_topic, len(g))
        return g.sample(n=n, random_state=random_state)

    sample_df = (
        df.groupby(topic_col, group_keys=False)[df.columns.tolist()]
          .apply(sample_group)
          .reset_index(drop=True)
    )

    # population size per topic (from the FULL df, not the sample)
    pop_counts = df[topic_col].value_counts()
    samp_counts = sample_df[topic_col].value_counts()

    sample_df['N_topic'] = sample_df[topic_col].map(pop_counts)
    sample_df['n_topic'] = sample_df[topic_col].map(samp_counts)
    sample_df['reweight'] = sample_df['N_topic'] / sample_df['n_topic']

    # allocation summary — check this before you start annotating
    summary = pd.DataFrame({
        'population_N': pop_counts,
        'sample_n': samp_counts,
        'coverage_%': (samp_counts / pop_counts * 100).round(2),
        'reweight_factor': (pop_counts / samp_counts).round(2)
    }).sort_values('population_N', ascending=False)

    print(summary)
    print(f"\nTotal sampled: {len(sample_df)} / {len(df)}")

    return sample_df, summary

sample_df, allocation_summary = stratified_sample_equal(df, n_per_topic=150)
sample_df.to_csv('sample_for_annotation.csv', index=False)

In [ ]:
sample_df.to_csv('sample_for_annotation.csv', index=False)

In [ ]:

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/Facebook Crowdtangle Data/All Zied Download Folder/Renamed by Rafi/Categorized by Maryam/Environment/Environment"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "invalid_llmaa_checkpoint.csv")

print(f"Checkpoint will be saved to: {CHECKPOINT_PATH}")

In [ ]:
df_sampled=pd.read_csv('/content/drive/MyDrive/Facebook Crowdtangle Data/All Zied Download Folder/Renamed by Rafi/Categorized by Maryam/Environment/Environment/df_sampled_2000.csv')

In [ ]:

import os
import time
import pandas as pd
import requests
import re

CHECKPOINT_DIR = "/content/drive/MyDrive/Facebook Crowdtangle Data/All Zied Download Folder/Renamed by Rafi/Categorized by Maryam/Environment/Environment"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "invalid_llmaa_checkpoint.csv")

print(f"Checkpoint will be saved to: {CHECKPOINT_PATH}")


OPENROUTER_API_KEY = "sk----"

# Pin the dated snapshot, not the floating alias — makes results reproducible for the methods section
MODEL = "meta-llama/llama-3.1-70b-instruct"

TOPICS = [

    "Captivity, Zoos & Ethical Wildlife Management",
    "Climate Change Impact on Wildlife",
    "Habitat Loss, Deforestation & Ecosystem Degradation",
    "Wildlife Rehabilitation & Rescue",
    "Wildlife Awareness, Media & Activism",
    "Human-Wildlife Conflict",
    "Marine & Ocean Conservation",
    "Endangered Species & Extinction Risk",
    "Wildlife Crime, Poaching & Enforcement",
    "Conservation Policy, Law & Funding",
    "Indigenous & Community-Based Conservation",
    "National Parks & Protected Areas",
    "UNASSIGNED",  # escape valve — required, or every ambiguous post is force-fit
]


SYSTEM_PROMPT = """
You are an expert annotator classifying social media messages
about wildlife, biodiversity, nature, and conservation.

Most messages are written in Arabic, including Modern Standard
Arabic (MSA), Gulf, Egyptian, Levantine, and Maghrebi dialects.
Some messages may contain English, code-switching, informal
spelling, or Arabizi (Arabic transliterated into Latin letters
and numbers, such as "7" for ح and "3" for ع).

Classify the meaning of each message regardless of its dialect,
script, or writing style. Do not classify a message as UNASSIGNED
merely because it contains unfamiliar dialectal vocabulary,
informal expressions, or spelling variations.

Your task is to assign each message to exactly ONE of the
following 12 predefined topics based on its primary meaning.

Each topic has an English and Arabic definition. Both definitions
describe the same topic and should be interpreted together.

TOPIC DEFINITIONS

1. Captivity, Zoos & Ethical Wildlife Management

Wild animals kept in zoos, aquariums, breeding facilities,
or captivity, including animal welfare, living conditions,
captive breeding, management, and ethical treatment.

الحيوانات البرية الموجودة في حدائق الحيوان أو الأحواض المائية
أو مراكز التكاثر أو الأسر، بما في ذلك رفاهية الحيوانات
وظروف الاحتجاز والتكاثر في الأسر والإدارة والمعاملة الأخلاقية.


2. Climate Change Impact on Wildlife

Effects of climate change, global warming, extreme heat,
climate-related drought, changing weather patterns, and
other climatic changes on wild animals, their habitats,
migration, reproduction, and survival.

تأثير تغير المناخ والاحتباس الحراري وارتفاع درجات الحرارة
والجفاف المرتبط بالمناخ وتغير أنماط الطقس على الحيوانات
البرية وموائلها وهجرتها وتكاثرها وبقائها.


3. Habitat Loss, Deforestation & Ecosystem Degradation

Destruction, fragmentation, or degradation of natural
habitats and ecosystems affecting wildlife, including
deforestation, land conversion, urban expansion,
terrestrial pollution, and loss of biodiversity.

تدمير الموائل الطبيعية وتجزئتها وتدهور النظم البيئية
التي تعتمد عليها الحياة البرية، بما في ذلك إزالة الغابات
وتغيير استخدام الأراضي والتوسع العمراني والتلوث البري
وفقدان التنوع البيولوجي.


4. Wildlife Rehabilitation & Rescue

Rescue, veterinary treatment, rehabilitation, recovery,
and release of individual wild animals that are injured,
stranded, orphaned, trapped, or in distress.

إنقاذ الحيوانات البرية المصابة أو العالقة أو اليتيمة
أو المعرضة للخطر، وعلاجها بيطرياً وإعادة تأهيلها
وتعافيها وإطلاقها مجدداً في البرية.


5. Wildlife Awareness, Media & Activism

Public education, awareness campaigns, environmental
advocacy, wildlife documentaries, photography, observation,
species information, animal behavior, and general
appreciation or promotion of wildlife and nature.

التوعية العامة والتثقيف البيئي وحملات حماية الحياة البرية
والنشاط البيئي والأفلام الوثائقية وتصوير الحيوانات
ومشاهدتها والمعلومات عن الأنواع وسلوك الحيوانات
والتقدير العام للحياة البرية والطبيعة.


6. Human-Wildlife Conflict

Harmful or disputed interactions between wild animals
and humans, including attacks on people, crop damage,
livestock predation, retaliatory killing, property
damage, and efforts to improve human-wildlife coexistence.

الصراعات والتفاعلات الضارة بين الإنسان والحيوانات البرية،
بما في ذلك الهجمات على البشر وإتلاف المحاصيل وافتراس
الماشية والقتل الانتقامي والأضرار بالممتلكات والجهود
المبذولة لتحسين التعايش بين الإنسان والحياة البرية.


7. Marine & Ocean Conservation

Protection and management of marine wildlife and aquatic
ecosystems, including whales, dolphins, sea turtles,
coral reefs, marine pollution, plastic waste, overfishing,
bycatch, and threats to marine biodiversity.

حماية وإدارة الحياة البرية البحرية والنظم البيئية المائية،
بما في ذلك الحيتان والدلافين والسلاحف البحرية والشعاب
المرجانية والتلوث البحري والنفايات البلاستيكية والصيد
الجائر والصيد العرضي والتهديدات للتنوع البيولوجي البحري.


8. Endangered Species & Extinction Risk

Conservation status and survival of wildlife species,
including endangered and threatened species, declining
populations, extinction risk, species-level recovery,
and protection or reintroduction of threatened species.

حالة حفظ الأنواع البرية وبقائها، بما في ذلك الأنواع
المهددة بالانقراض وتناقص أعدادها وخطر انقراضها
وبرامج استعادة الأنواع وحمايتها وإعادة توطينها.


9. Wildlife Crime, Poaching & Enforcement

Illegal hunting, killing, capturing, trading, or
trafficking of wild animals and wildlife products,
including poaching, illegal wildlife trade, investigations,
seizures, arrests, and enforcement against wildlife crime.

الصيد غير القانوني وقتل الحيوانات البرية وأسرها
والاتجار غير المشروع بها أو بمنتجاتها، بما في ذلك
الصيد الجائر وتهريب الحيوانات والتحقيقات والمصادرات
والاعتقالات وإنفاذ قوانين مكافحة جرائم الحياة البرية.


10. Conservation Policy, Law & Funding

Government policies, legislation, regulations, funding,
donations, institutional programs, and activities by
conservation organizations intended to protect wildlife,
natural resources, and biodiversity.

السياسات الحكومية والتشريعات واللوائح والتمويل
والتبرعات والبرامج المؤسسية وأنشطة منظمات الحفاظ
على البيئة الهادفة إلى حماية الحياة البرية والموارد
الطبيعية والتنوع البيولوجي.


11. Indigenous & Community-Based Conservation

Wildlife conservation involving indigenous peoples
or local communities, including traditional ecological
knowledge, community-led protection, local natural
resource management, and land rights connected to
biodiversity conservation.

حماية الحياة البرية بمشاركة السكان الأصليين أو المجتمعات
المحلية، بما في ذلك المعرفة البيئية التقليدية ومبادرات
الحماية المجتمعية وإدارة الموارد الطبيعية محلياً
وحقوق الأراضي المرتبطة بالحفاظ على التنوع البيولوجي.


12. National Parks & Protected Areas

Establishment, management, conservation, and visitation
of national parks, nature reserves, wildlife sanctuaries,
and designated protected areas, including their ecological
importance and role in protecting wildlife.

إنشاء وإدارة وحماية وزيارة الحدائق الوطنية والمحميات
الطبيعية ومحميات الحياة البرية والمناطق المحمية،
بما في ذلك أهميتها البيئية ودورها في حماية الحيوانات البرية.


CLASSIFICATION RULES

1. PRIMARY MEANING
Identify the primary subject of the message based on its
overall meaning. Do not classify based solely on the
presence of individual keywords.

2. SINGLE TOPIC
Assign exactly ONE topic to each message. Do not return
multiple topics or combine category names.

3. SPECIFICITY
When multiple topics are relevant, select the topic that
best represents the main focus of the message.

Prefer a specific conservation issue or event over a
general topic when the specific issue is clearly the
primary focus.

4. GENERAL WILDLIFE CONTENT
Messages about wildlife education, animal behavior,
species facts, nature appreciation, wildlife photography,
documentaries, and wildlife observation can be classified
as Wildlife Awareness, Media & Activism when no more
specific topic applies.

An explicit mention of conservation or environmental
protection is not required for this category.

However, do not use this category as a fallback for
unrelated, unclear, or insufficient content.

5. OVERLAPPING TOPICS
Distinguish between categories based on the main issue
being discussed.

Examples:

- Illegal wildlife trafficking or poaching belongs to
  Wildlife Crime, Poaching & Enforcement.

- The living conditions of animals in zoos belong to
  Captivity, Zoos & Ethical Wildlife Management.

- Rescue and veterinary treatment of an individual
  injured animal belong to Wildlife Rehabilitation & Rescue.

- Species-level population decline or extinction risk
  belongs to Endangered Species & Extinction Risk.

- Wildlife attacks on people or livestock belong to
  Human-Wildlife Conflict.

- Habitat destruction caused by deforestation or
  land conversion belongs to Habitat Loss, Deforestation
  & Ecosystem Degradation.

- Wildlife impacts explicitly attributed to climate
  change belong to Climate Change Impact on Wildlife.

- A poaching incident occurring inside a national park
  belongs to Wildlife Crime, Poaching & Enforcement
  when the incident is the primary focus.

- Community participation alone is insufficient for
  Indigenous & Community-Based Conservation unless
  the message concerns a community role in conservation.

- Marine wildlife and marine ecosystem protection
  belong to Marine & Ocean Conservation when the
  marine conservation issue is the primary focus.

6. CONTEXTUAL INTERPRETATION
Interpret dialectal expressions, informal language,
code-switching, and Arabizi based on their intended
meaning rather than exact vocabulary.

Do not assume that a message is unrelated simply
because its wording differs from the definitions.

7. INSUFFICIENT EVIDENCE
Do not infer a specific conservation event, crime,
policy, or environmental threat without sufficient
support from the message.

For example, mentioning a national park is not
automatically evidence of poaching, and mentioning
an endangered animal does not necessarily mean that
the message discusses extinction risk.

8. UNASSIGNED
Return UNASSIGNED only when one of the following applies:

- The message does not reasonably fit any of the
  12 predefined topics.

- The message is unrelated to wildlife, biodiversity,
  nature, or conservation.

- The message contains too little meaningful information
  to identify a relevant topic.

- The message is too ambiguous to determine its
  primary topic, even after considering its context.

Do not force a message into an unrelated category
just to avoid UNASSIGNED.

However, do not choose UNASSIGNED merely because
the post does not explicitly mention conservation.
General wildlife-related content may still fit
Wildlife Awareness, Media & Activism.

9. CONSISTENCY
Apply the same topic definitions and classification
principles to all messages, regardless of language,
dialect, length, writing style, or emotional tone.

10. INTERNAL DECISION
Consider the message meaning, relevant definitions,
and potential category overlaps before selecting
the most appropriate label.

Do not output your reasoning.

OUTPUT REQUIREMENTS

Return ONLY the exact English topic name from the
following list:

Captivity, Zoos & Ethical Wildlife Management
Climate Change Impact on Wildlife
Habitat Loss, Deforestation & Ecosystem Degradation
Wildlife Rehabilitation & Rescue
Wildlife Awareness, Media & Activism
Human-Wildlife Conflict
Marine & Ocean Conservation
Endangered Species & Extinction Risk
Wildlife Crime, Poaching & Enforcement
Conservation Policy, Law & Funding
Indigenous & Community-Based Conservation
National Parks & Protected Areas
UNASSIGNED

Do not return explanations, translations, numbering,
quotation marks, punctuation, or additional text.
""".strip()



RETRY_LABELS = {"API_ERROR", "EMPTY_RESPONSE"}   # re-attempted on the next run
MAX_TOKENS = 300                                 # 30 is too low for reasoning models


def classify_message(message, max_retries=4):
    failure = "API_ERROR"
    problem = "no attempt made"

    for attempt in range(max_retries):
        if attempt:
            time.sleep(2 ** attempt)
        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": MODEL,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": str(message)},
                    ],
                    "temperature": 0,
                    "max_tokens": MAX_TOKENS,
                },
                timeout=60,
            )

            if response.status_code == 429 or response.status_code >= 500:
                failure, problem = "API_ERROR", f"HTTP {response.status_code}"
                continue

            response.raise_for_status()
            data = response.json()

            # OpenRouter can return HTTP 200 with an error body and no choices
            if not data.get("choices"):
                failure, problem = "API_ERROR", f"no choices: {data.get('error')}"
                continue

            choice = data["choices"][0]
            content = (choice.get("message") or {}).get("content")

            # content is None/empty when the model spent its tokens on
            # reasoning, refused, or was cut off by a content filter
            if not isinstance(content, str) or not content.strip():
                failure = "EMPTY_RESPONSE"
                problem = f"empty content, finish_reason={choice.get('finish_reason')!r}"
                continue

            return normalize_topic(content)

        except (requests.exceptions.RequestException, ValueError) as e:
            failure, problem = "API_ERROR", str(e)

    print(f"Failed after {max_retries} attempts: {problem}")
    return failure


def normalize_for_compare(s):
    s = s.replace('&', 'and')
    s = re.sub(r'[^\w\s]', '', s)      # strip commas, periods, quotes
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s

TOPIC_LOOKUP = {normalize_for_compare(t): t for t in TOPICS}

def normalize_topic(raw):
    cleaned = raw.strip().strip('."\'').strip()
    if cleaned in TOPICS:
        return cleaned
    key = normalize_for_compare(cleaned)
    if key in TOPIC_LOOKUP:
        return TOPIC_LOOKUP[key]
    if key:
        for norm_key, original in TOPIC_LOOKUP.items():   # loose substring fallback
            if norm_key in key or key in norm_key:
                return original
    print(f"UNRECOGNIZED OUTPUT: {raw!r}")
    return "INVALID_OUTPUT"


def needs_label(value):
    # NaN covers rows reloaded from a checkpoint CSV, where None becomes NaN
    return pd.isna(value) or value in RETRY_LABELS


CHECKPOINT_EVERY = 50

if "Human_Topic" in df.columns:
    predictions = df["Human_Topic"].astype(object).tolist()
else:
    predictions = [None] * len(df)

todo = [i for i, value in enumerate(predictions) if needs_label(value)]
print(f"{len(todo):,} of {len(df):,} rows to classify")

for done, i in enumerate(todo, start=1):
    message = df["Message"].iloc[i]

    if pd.isna(message) or not str(message).strip():
        predictions[i] = "EMPTY_MESSAGE"
    else:
        predictions[i] = classify_message(message)

    if done % CHECKPOINT_EVERY == 0:
        df["Human_Topic"] = predictions
        df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")
        print(f"Checkpointed at {done}/{len(todo)}")

df["Human_Topic"] = predictions
df.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8-sig")

# sanity checks before moving to agreement analysis
print(df["Human_Topic"].value_counts(dropna=False))
problem_labels = ["INVALID_OUTPUT", "API_ERROR", "EMPTY_RESPONSE", "EMPTY_MESSAGE"]
print("Invalid/error rows:", df["Human_Topic"].isin(problem_labels).sum())

In [ ]:
df=df1_invalid_messages.copy()

In [ ]:
sample_disagreement = (
    df_sampled.loc[
        (df_sampled["LLM_Topic"] == "UNASSIGNED") &
        (df_sampled["Assigned_Topic"] != "UNASSIGNED")
    ]
    .sample(n=50, random_state=42)
)

sample_disagreement[
    ["Message", "Assigned_Topic", "LLM_Topic"]
]

In [ ]:
df_sampled.to_csv('/content/drive/MyDrive/Facebook Crowdtangle Data/All Zied Download Folder/Renamed by Rafi/Categorized by Maryam/Environment/Environment/df_sampled_annotated_Qwen_8_8Oct.csv')

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Use your original SBERT model for a controlled comparison
model = SentenceTransformer("YOUR_ORIGINAL_SBERT_MODEL")

topic_names = list(TOPIC_DEFINITIONS.keys())
topic_texts = list(TOPIC_DEFINITIONS.values())

topic_embeddings = model.encode(
    topic_texts,
    normalize_embeddings=True,
    convert_to_numpy=True
)

post_embeddings = model.encode(
    df["Message"].fillna("").astype(str).tolist(),
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Cosine similarity because embeddings are normalized
similarities = post_embeddings @ topic_embeddings.T

df["SBERT_score"] = similarities.max(axis=1)
df["SBERT_best_topic"] = [
    topic_names[i] for i in similarities.argmax(axis=1)
]

In [ ]:
df_sampled.head(2)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import entropy

def compute_reaction_scores(df):
    """
    Adds 3 new columns to df:
      - sentiment_score  (−1 to +1)
      - polarization_score (0 to 1)
      - cohesion_score  (0 to 1)
    Assumes reaction columns:
      Likes, Love, Care, Haha, Wow, Angry, Sad
    """
    # Make a copy to avoid modifying original
    df = df.copy()

    # --- 0. Clean / ensure numeric ---
    reaction_cols = [
        'Likes', 'Love', 'Care',
        'Haha', 'Wow',
        'Angry', 'Sad'
    ]
    df[reaction_cols] = df[reaction_cols].fillna(0).astype(float)
    like  = df['Likes'].to_numpy()
    love  = df['Love'].to_numpy()
    care  = df['Care'].to_numpy()
    haha  = df['Haha'].to_numpy()
    wow   = df['Wow'].to_numpy()
    angry = df['Angry'].to_numpy()
    sad   = df['Sad'].to_numpy()

    # --- 1. Sentiment score (−1 to +1) ---
    positive_base = like*1 + love*2 + care*0.5
    negative_base = angry*2 + sad*1
    sign = np.where(positive_base >= negative_base, 1.0, -1.0)

    haha_adj = sign * haha
    wow_adj  = sign * wow

    haha_pos = np.where(haha_adj > 0,  haha_adj, 0.0)
    haha_neg = np.where(haha_adj < 0, -haha_adj, 0.0)
    wow_pos  = np.where(wow_adj  > 0,  wow_adj,  0.0)
    wow_neg  = np.where(wow_adj  < 0, -wow_adj,  0.0)

    weighted_positive = (
        like*1    +
        love*2    +
        care*0.5  +
        haha_pos  +
        wow_pos
    )
    weighted_negative = (
        angry*2   +
        sad*1     +
        haha_neg  +
        wow_neg
    )
    total_weighted = weighted_positive + weighted_negative
    sentiment = np.where(
        total_weighted > 0,
        (weighted_positive - weighted_negative) / total_weighted,
        np.nan
    )
    df['sentiment_score'] = sentiment

    # --- 2. Polarization score (0 to 1) ---
    pos_count = (
        like +
        love +
        care +
        haha_pos +
        wow_pos
    )
    neg_count = (
        angry +
        sad +
        haha_neg +
        wow_neg
    )
    total_posneg = pos_count + neg_count
    pos_prop = np.divide(
        pos_count, total_posneg,
        out=np.zeros_like(pos_count),
        where=total_posneg > 0
    )
    neg_prop = np.divide(
        neg_count, total_posneg,
        out=np.zeros_like(neg_count),
        where=total_posneg > 0
    )
    polarization = 2 * np.minimum(pos_prop, neg_prop)
    polarization[total_posneg == 0] = np.nan
    df['polarization_score'] = polarization

    # --- 3. Cohesion score (0 to 1) ---
    counts = np.vstack([like, love, haha, wow, angry, sad, care]).T
    sums = counts.sum(axis=1)
    probs = np.divide(
        counts,
        sums[:, None],
        out=np.zeros_like(counts),
        where=sums[:, None] > 0
    )
    H = entropy(probs, base=2, axis=1)
    H_max = np.log2(counts.shape[1])
    H_norm = np.where(
        sums > 0,
        H / H_max,
        np.nan
    )
    cohesion = np.where(
        sums > 0,
        1.0 - H_norm,
        np.nan
    )
    df['cohesion_score'] = cohesion
    return df


def label_scores(df):
    """
    Converts numeric scores to categorical labels with 3-4 categories each.
    Adds 3 new columns: sentiment_label, cohesion_label, polarization_label
    """

    # Make a copy to avoid modifying original
    df = df.copy()

    # Sentiment: -1 to +1 (4-point scale)
    def sentiment_label(score):
        if pd.isna(score):
            return 'No Reaction'
        elif score >= 0.3:
            return 'Positive'
        elif score <= -0.3:
            return 'Negative'
        else:
            return 'Neutral'

    # Cohesion: 0 to 1 (3-point scale)
    def cohesion_label(score):
        if pd.isna(score):
            return 'No Reaction'
        elif score >= 0.65:
            return 'Unified'
        elif score >= 0.35:
            return 'Mixed'
        else:
            return 'Fragmented'

    # Polarization: 0 to 1 (3-point scale)
    def polarization_label(score):
        if pd.isna(score):
            return 'No Reaction'
        elif score >= 0.65:
            return 'Polarized'
        elif score >= 0.25:
            return 'Divided'
        else:
            return 'Consensus'

    df['sentiment_label'] = df['sentiment_score'].apply(sentiment_label)
    df['cohesion_label'] = df['cohesion_score'].apply(cohesion_label)
    df['polarization_label'] = df['polarization_score'].apply(polarization_label)

    return df


def process_dataframe(df):
    """
    Process existing dataframe with reaction scores and labels

    Parameters:
    -----------
    df : pandas.DataFrame
        Your dataframe with reaction columns (Likes, Love, Care, Haha, Wow, Angry, Sad)

    Returns:
    --------
    df_processed : pandas.DataFrame
        Original dataframe with added score and label columns
    """

    print(f"✓ Processing {len(df):,} rows...")

    # Compute scores
    df_processed = compute_reaction_scores(df)

    # Add labels
    df_processed = label_scores(df_processed)

    print(f"✓ Added 6 new columns: sentiment_score, sentiment_label, cohesion_score, cohesion_label, polarization_score, polarization_label")

    # Print summary statistics
    print("\n📊 Score Summary Statistics:")
    print(df_processed[['sentiment_score', 'cohesion_score', 'polarization_score']].describe())

    print("\n📊 Label Distributions:")
    print("\nSentiment:")
    print(df_processed['sentiment_label'].value_counts())
    print("\nCohesion:")
    print(df_processed['cohesion_label'].value_counts())
    print("\nPolarization:")
    print(df_processed['polarization_label'].value_counts())

    return df_processed


# Usage
if __name__ == "__main__":
    # Assuming you already have df loaded
    # df = your existing dataframe

    # Process it
    df_processed = process_dataframe(df)

    # Now df_processed has all the new columns
    print("\n✓ Processing complete!")
    print(f"✓ Dataframe now has {len(df_processed.columns)} columns")

In [ ]:
len(df)

In [ ]:
import numpy as np
import pandas as pd


def compute_reaction_scores(
    df,
    min_valenced_reactions=5,
    min_total_reactions=5,
    include_like_in_sentiment=False
):
    """
    Computes:
      - sentiment_score: relative positive/negative valence
      - polarization_score: balance between valence directions
      - cohesion_score: concentration across seven reaction types

    Haha and Wow are excluded from sentiment and polarization because
    their valence is context-dependent. They remain in cohesion.

    Scores based on fewer than the specified minimum number of
    reactions are returned as NaN.
    """

    df = df.copy()

    reaction_cols = [
        "Likes", "Love", "Care", "Haha",
        "Wow", "Angry", "Sad"
    ]

    # Convert invalid and missing entries to zero
    df[reaction_cols] = (
        df[reaction_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
    )

    # Reaction counts cannot be negative
    if (df[reaction_cols] < 0).any().any():
        raise ValueError("Reaction columns contain negative values.")

    like = df["Likes"].to_numpy(dtype=float)
    love = df["Love"].to_numpy(dtype=float)
    care = df["Care"].to_numpy(dtype=float)
    haha = df["Haha"].to_numpy(dtype=float)
    wow = df["Wow"].to_numpy(dtype=float)
    angry = df["Angry"].to_numpy(dtype=float)
    sad = df["Sad"].to_numpy(dtype=float)

    # ------------------------------------------------------------
    # 1. Sentiment and polarization
    # ------------------------------------------------------------

    positive = love + care
    negative = sad + angry

    # Optional Like-inclusive robustness specification
    if include_like_in_sentiment:
        positive = positive + like

    valenced_total = positive + negative

    sentiment = np.full(len(df), np.nan, dtype=float)
    polarization = np.full(len(df), np.nan, dtype=float)

    valid_valence = valenced_total >= min_valenced_reactions

    np.divide(
        positive - negative,
        valenced_total,
        out=sentiment,
        where=valid_valence
    )

    np.divide(
        2.0 * np.minimum(positive, negative),
        valenced_total,
        out=polarization,
        where=valid_valence
    )

    # ------------------------------------------------------------
    # 2. Cohesion across all seven reaction types
    # ------------------------------------------------------------

    counts = np.column_stack([
        like, love, care, haha, wow, sad, angry
    ])

    total_reactions = counts.sum(axis=1)

    probabilities = np.zeros_like(counts, dtype=float)

    np.divide(
        counts,
        total_reactions[:, None],
        out=probabilities,
        where=total_reactions[:, None] > 0
    )

    # Calculate entropy without log2(0) warnings
    log_probabilities = np.zeros_like(probabilities)

    np.log2(
        probabilities,
        out=log_probabilities,
        where=probabilities > 0
    )

    entropy_score = -(
        probabilities * log_probabilities
    ).sum(axis=1)

    max_entropy = np.log2(counts.shape[1])

    cohesion = np.full(len(df), np.nan, dtype=float)
    valid_cohesion = total_reactions >= min_total_reactions

    cohesion[valid_cohesion] = (
        1.0
        - entropy_score[valid_cohesion] / max_entropy
    )

    # Protect against minor floating-point deviations
    sentiment = np.clip(sentiment, -1.0, 1.0)
    polarization = np.clip(polarization, 0.0, 1.0)
    cohesion = np.clip(cohesion, 0.0, 1.0)

    df["valenced_reaction_count"] = valenced_total
    df["total_reaction_count"] = total_reactions
    df["sentiment_score"] = sentiment
    df["polarization_score"] = polarization
    df["cohesion_score"] = cohesion

    return df

In [ ]:
def label_scores(
    df,
    min_valenced_reactions=5,
    min_total_reactions=5
):
    """
    Adds descriptive labels while distinguishing:
      - posts with no reactions;
      - posts without clearly valenced reactions;
      - posts with insufficient reaction evidence;
      - posts with valid scores.
    """

    df = df.copy()

    def sentiment_label(row):
        score = row["sentiment_score"]
        total = row["total_reaction_count"]
        valenced = row["valenced_reaction_count"]

        if total == 0:
            return "No Reaction"
        elif valenced == 0:
            return "No Valenced Reaction"
        elif valenced < min_valenced_reactions:
            return "Insufficient Valenced Reactions"
        elif score >= 0.3:
            return "Positive"
        elif score <= -0.3:
            return "Negative"
        else:
            return "Neutral"

    def polarization_label(row):
        score = row["polarization_score"]
        total = row["total_reaction_count"]
        valenced = row["valenced_reaction_count"]

        if total == 0:
            return "No Reaction"
        elif valenced == 0:
            return "No Valenced Reaction"
        elif valenced < min_valenced_reactions:
            return "Insufficient Valenced Reactions"
        elif score >= 0.65:
            return "Polarized"
        elif score >= 0.25:
            return "Divided"
        else:
            return "Consensus"

    def cohesion_label(row):
        score = row["cohesion_score"]
        total = row["total_reaction_count"]

        if total == 0:
            return "No Reaction"
        elif total < min_total_reactions:
            return "Insufficient Reactions"
        elif score >= 0.65:
            return "Unified"
        elif score >= 0.35:
            return "Mixed"
        else:
            return "Fragmented"

    df["sentiment_label"] = df.apply(
        sentiment_label, axis=1
    )

    df["polarization_label"] = df.apply(
        polarization_label, axis=1
    )

    df["cohesion_label"] = df.apply(
        cohesion_label, axis=1
    )

    return df

In [ ]:
df_processed = compute_reaction_scores(
    df,
    min_valenced_reactions=5,
    min_total_reactions=5,
    include_like_in_sentiment=False
)

# Apply categorical labels
df_processed = label_scores(df_processed)

# Print example results
print("\nSample reaction scores:")
print(
    df_processed[
        [
            "Likes", "Love", "Care", "Haha", "Wow", "Sad", "Angry",
            "valenced_reaction_count",
            "total_reaction_count",
            "sentiment_score",
            "sentiment_label",
            "polarization_score",
            "polarization_label",
            "cohesion_score",
            "cohesion_label"
        ]
    ].head(20).to_string(index=False)
)

# Print score statistics
print("\nScore summary:")
print(
    df_processed[
        [
            "sentiment_score",
            "polarization_score",
            "cohesion_score"
        ]
    ].describe()
)

# Print label distributions
for column in [
    "sentiment_label",
    "polarization_label",
    "cohesion_label"
]:
    print(f"\n{column}:")

    counts = df_processed[column].value_counts(dropna=False)
    percentages = (
        df_processed[column]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages
    })

    print(summary)

In [ ]:
df_processed.head(2)

In [ ]:
df_processed.columns

In [ ]:
!wget -q https://github.com/google/fonts/raw/main/ofl/notosansarabic/NotoSansArabic%5Bwdth%2Cwght%5D.ttf -O NotoSansArabic.ttf

!pip install tashaphyne

In [ ]:
!pip install sentence-transformers

In [ ]:
"""
SBERT Topic Modeling - 12 Merged Topics for Better Readability
Refined from original 20 topics based on semantic overlap and discourse patterns
"""

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')


"""
SBERT Topic Modeling - 12 Merged Topics with Comprehensive Dialectal Arabic
Includes Egyptian, Levantine, Gulf, and Maghrebi dialects
"""

TOPICS_MERGED = {
    "Captivity, Zoos & Ethical Wildlife Management": """
        zoo, captive, cage, freedom, breeding program, born in captivity, sad, prison,
        released, belongs in wild, conservation breeding, shouldn't be captive, animal welfare,
        ethical treatment, captive animals, zoo animals, animal rights, wild animals in captivity,
        حديقة حيوان, أسير, قفص, حرية, برنامج تربية, ولد في الأسر, حزين, سجن,
        أطلق سراحه, ينتمي للبرية, تكاثر, حديقة الحيوانات, اتولد في الأسر, طلعوه,
        مكانه البرية, ولد بالأسر, محله بالبرية, مكانه البر, جاردة الحيوانات,
        تولد في الأسر, السجن, خرجوه, بلاصتو البرية, حقوق الحيوان, رعاية الحيوانات,
        Egyptian: حديقة الحيوانات, اتولد في الأسر, طلعوه, مكانه البرية, قفص, حزين قوي,
        سجن, مش حر, لازم يطلع, حرام يكون في قفص, المفروض في البرية,
        Levantine: حديقة الحيوانات, ولد بالأسر, محله بالبرية, قفص, حزين كتير,
        سجن, مش حر, لازم يطلع, حرام يكون بقفص, المفروض بالبرية,
        Gulf: حديقة الحيوانات, ولد بالأسر, مكانه البر, قفص, حزين مرة,
        سجن, مش حر, لازم يطلع, حرام يكون بقفص, المفروض بالبرية,
        Maghrebi: جاردة الحيوانات, تولد في الأسر, بلاصتو البرية, قفص, حزين بزاف,
        الحبس, ماشي حر, خاصو يخرج, حرام يكون في القفص, خاصو يكون في البرية
    """,

    "Climate Change Impact on Wildlife": """
        warming, hotter, melting ice, starving, dying from heat, no food, changing weather,
        suffering, struggling, climate crisis, ice caps, polar bears, drying up, global warming,
        temperature rise, extreme weather, climate adaptation, habitat shift, migration patterns,
        احترار, حرارة, ذوبان الجليد, جوع, يموت من الحر, طقس, معاناة, أزمة مناخية,
        دببة قطبية, جفاف, الحر, سخونة, الجليد بيدوب, جعان, بيموت من الحر, مافيش أكل,
        الجو اتغير, بيعاني, أزمة المناخ, جوعان, عم يدوب, عم يموت من الحر, ما في أكل,
        الجو تغير, عم يعاني, يذوب, ما فيه أكل, كيدوب, جيعان, ما كاين ماكلة,
        الجو تبدل, كيعاني, السخانة, الاحتباس الحراري, التغير المناخي,
        Egyptian: الحر الشديد, سخونة, الجليد بيدوب, جعان, بيموت من الحر, مافيش أكل,
        الجو اتغير, بيعاني, الدنيا بتسخن, درجة الحرارة عالية, الطقس غريب,
        Levantine: الحر الشديد, سخونة, الجليد عم يدوب, جوعان, عم يموت من الحر, ما في أكل,
        الجو تغير, عم يعاني, الدنيا عم تسخن, درجة الحرارة عالية, الطقس غريب,
        Gulf: الحر الشديد, حرارة, الجليد يذوب, جوعان, يموت من الحر, ما فيه أكل,
        الجو تغير, يعاني, الدنيا تسخن, درجة الحرارة عالية, الطقس غريب,
        Maghrebi: السخانة, الجليد كيدوب, جيعان, كيموت من الحر, ما كاين ماكلة,
        الجو تبدل, كيعاني, الدنيا كتسخن, درجة الحرارة عالية, الطقس غريب
    """,

    "Habitat Loss, Deforestation & Ecosystem Degradation": """
        destroying, cutting down, lost home, nowhere to go, bulldozers, trees gone, homeless,
        razed, cleared, developers, greedy, profits, destroyed habitat, building over, deforestation,
        land clearing, forest loss, ecosystem, nature, balance, biodiversity, everything connected,
        food chain, nature's balance, ecological, crucial role, habitat destruction, environmental degradation,
        تدمير, قطع الأشجار, فقد بيته, تشرد, جرافات, غابات, تجريف, طمع, أرباح,
        مقاولون, تدمير البيئة, قطعوا الشجر, ضاع بيته, مشرد, معندوش مكان, الشجر راح,
        فلوس, مقاولين, ما عندو مكان, ما عنده مكان, بولدوزر, تهديم, داره,
        ما عندو فين يمشي, الفلوس, نظام بيئي, طبيعة, توازن, تنوع بيولوجي,
        كل شيء مترابط, سلسلة غذائية, توازن الطبيعة, بيئي, كله مترابط, كلو مترابط,
        Egyptian: قطعوا الشجر, تدمير, ضاع بيته, مشرد, معندوش مكان, الشجر راح,
        جرافات, البلدوزر, المقاولين, طمع في الفلوس, البيئة اتدمرت, التوازن البيئي,
        Levantine: قطعوا الشجر, تدمير, ضاع بيته, مشرد, ما عندو مكان, الشجر راح,
        جرافات, البلدوزر, المقاولين, طمع بالمصاري, البيئة اندمرت, التوازن البيئي,
        Gulf: قطعوا الشجر, تدمير, ضاع بيته, مشرد, ما عنده مكان, الشجر راح,
        جرافات, البلدوزر, المقاولين, طمع بالفلوس, البيئة تدمرت, التوازن البيئي,
        Maghrebi: قطعو الشجر, تهديم, ضاع داره, مشرد, ما عندو فين يمشي, الشجر راح,
        بولدوزر, المقاولين, طماع في الفلوس, البيئة تهدمت, التوازن البيئي
    """,

    "Wildlife Rehabilitation & Rescue": """
        rescued, saved, baby, orphan, cute, adorable, amazing, recovery, miracle, heartwarming,
        released back, second chance, caring for, nursing, happy ending, wildlife rescue,
        animal rescue, veterinary care, sanctuary, healing, rehabilitation center, foster care,
        إنقاذ, إنقذوه, طفل, يتيم, لطيف, جميل, رائع, تعافي, معجزة, مؤثر,
        أطلق سراحه, فرصة ثانية, رعاية, نهاية سعيدة, انقذوه, صغير, كيوت, تعافى,
        طلعوه, فرصة تانية, بيرعوه, نهاية حلوة, أنقذوه, حلو كتير, رجعوه, عم يرعوه,
        حلو مرة, يرعونه, نقذوه, زوين بزاف, تشافى, كيرعيوه,
        Egyptian: انقذوه, صغير, يتيم, كيوت, جميل قوي, روعة, تعافى, معجزة,
        مؤثر جدا, طلعوه, فرصة تانية, بيرعوه, نهاية حلوة, حبيب, لطيف قوي,
        Levantine: أنقذوه, صغير, يتيم, حلو كتير, جميل كتير, رائع, تعافى, معجزة,
        مؤثر كتير, رجعوه, فرصة تانية, عم يرعوه, نهاية حلوة, حبوب, لطيف كتير,
        Gulf: أنقذوه, صغير, يتيم, حلو مرة, جميل مرة, رائع, تعافى, معجزة,
        مؤثر مرة, رجعوه, فرصة ثانية, يرعونه, نهاية حلوة, حبوب, لطيف مرة,
        Maghrebi: نقذوه, صغير, يتيم, زوين بزاف, جميل بزاف, رائع, تشافى, معجزة,
        مؤثر بزاف, رجعوه, فرصة ثانية, كيرعيوه, نهاية مزيانة, حبوب, لطيف بزاف
    """,

    "Wildlife Awareness, Media & Activism": """
        share, raise awareness, spread the word, repost, make this viral, sign petition,
        take action, join us, stand up, fight for, voice, activism, movement, hashtag, campaign,
        amazing photo, incredible, beautiful, look at this, stunning, captured, photographer,
        video, footage, viral, must see, breathtaking, wow, shares, wildlife photography,
        safari, amazing experience, bucket list, trip of lifetime, guide, tour, traveling,
        adventure, photos, incredible, must see, visit, tourism, ecotourism,
        شارك, رفع الوعي, انشر الكلمة, أعد النشر, فيروسي, وقع العريضة, اتخذ إجراء,
        انضم, قف, صوت, نشاط, حركة, هاشتاغ, حملة, شير, ريبوست, خليها فيرال,
        صورة رائعة, جميل, شاهد, مذهل, مصور, فيديو, لقطات, يجب أن تشاهد, مشاركات,
        صورة جميلة, روعة, بص, فيرال, لازم تشوف, واو, سفاري, تجربة رائعة,
        رحلة العمر, دليل, جولة, سفر, مغامرة, صور, سياحة, زيارة, تصاور,
        Egyptian: شير, شارك, انشر, ريبوست, خليها فيرال, اعمل حاجة, انضم لينا,
        قوم, صوتك, حملة, بص, لازم تشوف, واو, صورة جميلة قوي, روعة, فيديو,
        رحلة, سفر, مغامرة, تجربة روعة, شوفت, سياحة,
        Levantine: شير, شارك, انشر, ريبوست, خليها فيرال, اعمل شي, انضم معنا,
        قوم, صوتك, حملة, شوف, لازم تشوف, واو, صورة حلوة كتير, روعة, فيديو,
        رحلة, سفر, مغامرة, تجربة حلوة كتير, شفت, سياحة,
        Gulf: شير, شارك, انشر, ريبوست, خليها فيرال, سوي شي, انضم لنا,
        قوم, صوتك, حملة, شوف, لازم تشوف, واو, صورة حلوة مرة, روعة, فيديو,
        رحلة, سفر, مغامرة, تجربة حلوة مرة, شفت, سياحة,
        Maghrebi: شير, شارك, انشر, ريبوست, ديرها فيرال, دير شي, انضم معانا,
        قوم, صوتك, الحملة, شوف, خاصك تشوف, واو, تصويرة زوينة بزاف, روعة, فيديو,
        رحلة, سفر, مغامرة, تجربة زوينة بزاف, شفت, السياحة
    """,

    "Human-Wildlife Conflict": """
        attack, killed person, dangerous, fear, scared, village, crops destroyed, livestock killed,
        elephants raiding, living with, conflict, compensation, problem, human wildlife coexistence,
        crop raiding, property damage, mitigation, fencing, buffer zones, farmer conflict,
        هجوم, قتل شخص, خطير, خوف, قرية, دمر المحاصيل, قتل الماشية, فيلة, صراع,
        تعويض, مشكلة, قتل حد, خطر, خايف, دمر الزرع, قتل المواشي, قتل حدا, ضيعة,
        قتل واحد, خرب الزرع, المشكل, الصراع, التعويض, الفيلة,
        Egyptian: هجوم, قتل حد, خطر, خايف, قرية, دمر الزرع, قتل المواشي,
        الفيلة, صراع, تعويض, مشكلة كبيرة, خرب المحصول, البهايم ماتت,
        Levantine: هجوم, قتل حدا, خطر, خايف, ضيعة, دمر الزرع, قتل المواشي,
        الفيلة, صراع, تعويض, مشكلة كبيرة, خرب المحصول, البهايم ماتت,
        Gulf: هجوم, قتل واحد, خطر, خايف, قرية, دمر الزرع, قتل البهايم,
        الفيلة, صراع, تعويض, مشكلة كبيرة, خرب المحصول, البهايم ماتت,
        Maghrebi: هجوم, قتل واحد, خطر, خايف, قرية, خرب الزرع, قتل البهايم,
        الفيلة, الصراع, التعويض, المشكل, خرب المحصول, البهايم ماتو
    """,

    "Marine & Ocean Conservation": """
        ocean, sea, beach, plastic, trash, fishing nets, caught, tangled, dolphins, whales,
        turtles, sharks, coral, pollution, stranded, washed up, drowning in plastic, marine life,
        ocean pollution, plastic pollution, garbage, killing, choking, toxic, poisoned, contaminated,
        oil spill, dirty, suffering, waste, rubbish, littering, marine debris,
        محيط, بحر, شاطئ, بلاستيك, قمامة, شباك صيد, دلافين, حيتان, سلاحف,
        أسماك قرش, شعاب مرجانية, تلوث, جنح, زبالة, اتمسك, قروش, شط, انمسك,
        شاطي, مرجان, البحر, الشط, الزبل, تشد, فكرون, المرجان, التلوث, نفايات,
        Egyptian: البحر, الشاطئ, بلاستيك, زبالة, شباك صيد, دلافين, حيتان,
        سلاحف, قروش, مرجان, تلوث, اتمسك في الشبكة, الزبالة في البحر,
        Levantine: البحر, الشط, بلاستيك, زبالة, شباك صيد, دلافين, حيتان,
        سلاحف, قروش, مرجان, تلوث, انمسك بالشبكة, الزبالة بالبحر,
        Gulf: البحر, الشاطي, بلاستيك, زبالة, شباك صيد, دلافين, حيتان,
        سلاحف, قروش, مرجان, تلوث, انمسك بالشبكة, الزبالة بالبحر,
        Maghrebi: البحر, الشط, البلاستيك, الزبل, شباك الصيد, دلافين, حيتان,
        فكرون, قروش, المرجان, التلوث, تشد في الشبكة, الزبل في البحر
    """,

    "Endangered Species & Extinction Risk": """
        save, help, dying, disappearing, gone forever, last one, extinction, running out of time,
        losing, vanishing, rare, protect them, don't let them die, almost extinct, few left,
        endangered, threatened, IUCN, red list, vulnerable, critically endangered,
        انقراض, مهدد بالانقراض, نادر, احمي, ينقرض, يختفي, آخر واحد, نفقد, حماية,
        إنقاذ, مساعدة, ينتهي, يموت, بيموت, بيختفي, ضاع, خلاص راح, نحميه, ساعدوه,
        نادر قوي, عم ينقرض, عم يموت, لازم نحميه, نادر كتير, راح, ضايع, نادر مرة,
        كيموت, كيختافي, الاخير, نحميوه, عاونوه, نادر بزاف,
        Egyptian: بيموت, بيختفي, انقراض, نادر قوي, آخر واحد, ضاع, خلاص راح,
        نحميه, ساعدوه, مهدد بالانقراض, لازم نحميه, مش هيفضل منه حاجة,
        Levantine: عم يموت, عم يختفي, انقراض, نادر كتير, آخر واحد, راح, ضايع,
        لازم نحميه, ساعدوه, مهدد بالانقراض, مش رح يبقى منه شي,
        Gulf: يموت, يختفي, انقراض, نادر مرة, آخر واحد, راح, ضايع,
        نحميه, ساعدوه, مهدد بالانقراض, ما رح يبقى منه شي,
        Maghrebi: كيموت, كيختافي, الانقراض, نادر بزاف, الاخير, ضاع, راح,
        نحميوه, عاونوه, مهدد بالانقراض, ماشي غادي يبقى منه والو
    """,

    "Wildlife Crime, Poaching & Enforcement": """
        killed, murdered, slaughtered, poachers, hunters, trophy, ivory, horns, tusks, caught,
        arrested, cruel, barbaric, senseless killing, blood, butchered, shot dead,
        rangers, heroes, bust, seized, confiscated, brave, risking lives, patrol, stopped,
        intercepted, justice, prosecution, anti-poaching,
        trophy hunter, disgusting, sick, coward, killed for fun, shame, outrage, ban hunting,
        صيد جائر, قتل, ذبح, صيادون, عاج, قرون, أنياب, اعتقال, قبض, قسوة, وحشية,
        دماء, ظلم, جريمة, صيادين, دم, اتقبض عليه, اتمسك, حرام, قتلوه, ذبحوه,
        انمسك, انقبض عليه, جريمة قتل, مسكوه, قبضوا عليه, حراس, أبطال, مصادرة,
        شجاع, دورية, توقيف, عدالة, محاكمة, ضبط, صادروا, وقفوه, ضبطوه,
        Egyptian: قتلوه, ذبحوه, صيادين, عاج, قرون, اتقبض عليه, اتمسك, حرام,
        وحشية, دم, جريمة, قبضوا عليه, حراس, أبطال, صادروا, وقفوه, ضبطوه,
        مقرف, جبان, قتل عشان اللعب, عيب, منع الصيد,
        Levantine: قتلوه, ذبحوه, صيادين, عاج, قرون, انقبض عليه, انمسك, حرام,
        وحشية, دم, جريمة, قبضوا عليه, حراس, أبطال, صادروا, وقفوه,
        مقرف, جبان, قتل للمتعة, عيب, منع الصيد,
        Gulf: قتلوه, ذبحوه, صيادين, عاج, قرون, انقبض عليه, مسكوه, حرام,
        وحشي, دم, جريمة, قبضوا عليه, حراس, أبطال, صادروا, وقفوه,
        مقرف, جبان, قتل للمتعة, عيب, منع الصيد,
        Maghrebi: قتلوه, ذبحوه, صيادة, العاج, القرون, تشد عليه, حرام,
        وحشي, الدم, جريمة, شدو عليه, حراس, الأبطال, صادرو, وقفوه,
        مقرف, جبان, قتل للعب, العار, منع الصيد
    """,

    "Conservation Policy, Law & Funding": """
        ban, petition, sign, government, law, illegal, make it illegal, change the law, lawmakers,
        vote, policy, protect by law, legislation needed,
        donate, please help, support, give, every dollar helps, share, fundraiser, sponsor,
        adopt, contribute, conservation funding, NGO, charity, campaign,
        حظر, عريضة, وقع, حكومة, قانون, غير قانوني, تشريع, تصويت, سياسة,
        حماية قانونية, منع, ممنوع, حماية بالقانون, تبرع, ساعد, دعم, كل قرش يساعد,
        شارك, انشر, حملة تبرعات, رعاية, تبني, ساهم, ادعم,
        Egyptian: منع, عريضة, وقع, حكومة, قانون, ممنوع, تشريع, تصويت,
        حماية بالقانون, تبرع, ساعد, ادعم, كل قرش بيساعد, شير, حملة تبرعات,
        رعاية, تبني, ساهم,
        Levantine: منع, عريضة, وقع, حكومة, قانون, ممنوع, تشريع, تصويت,
        حماية بالقانون, تبرع, ساعد, ادعم, كل قرش بساعد, شير, حملة تبرعات,
        رعاية, تبني, ساهم,
        Gulf: منع, عريضة, وقع, حكومة, قانون, ممنوع, تشريع, تصويت,
        حماية بالقانون, تبرع, ساعد, ادعم, كل ريال يساعد, شير, حملة تبرعات,
        رعاية, تبني, ساهم,
        Maghrebi: المنع, العريضة, وقع, الحكومة, القانون, ممنوع, التشريع, التصويت,
        الحماية بالقانون, تبرع, عاون, ادعم, كل درهم كيعاون, شير, حملة التبرعات,
        الرعاية, التبني, ساهم
    """,

    "Indigenous & Community-Based Conservation": """
        tribes, native, local people, land rights, ancestral, traditional, community, respect,
        indigenous knowledge, forced out, displacement, guardians, indigenous peoples,
        قبائل, سكان أصليون, أهل البلد, حقوق الأرض, تقليدي, مجتمع, احترام,
        معرفة أصلية, تهجير, أوصياء, هجروهم, حماة, قبايل, القبايل, أهل البلاد,
        السكان الأصليين, المجتمع, الاحترام,
        Egyptian: قبائل, أهل البلد, سكان أصليين, حقوق الأرض, تقليدي, مجتمع,
        احترام, هجروهم, حماة, المجتمعات المحلية, الناس الأصليين,
        Levantine: قبائل, أهل البلد, سكان أصليين, حقوق الأرض, تقليدي, مجتمع,
        احترام, هجروهم, حماة, المجتمعات المحلية, الناس الأصليين,
        Gulf: قبايل, أهل البلد, سكان أصليين, حقوق الأرض, تقليدي, مجتمع,
        احترام, هجروهم, حماة, المجتمعات المحلية, الناس الأصليين,
        Maghrebi: القبايل, أهل البلاد, السكان الأصليين, حقوق الأرض, تقليدي, المجتمع,
        الاحترام, هجروهم, المجتمعات المحلية, الناس الأصليين
    """,

    "National Parks & Protected Areas": """
        park, reserve, sanctuary, visit, amazing, beautiful, saw, spotted, trip, experience,
        preserve, protect, safe haven, wildlife haven, national park, protected area,
        محمية, حديقة وطنية, محمية طبيعية, زيارة, رائع, جميل, شاهد, رحلة, تجربة,
        حماية, ملاذ آمن, حديقة, شفت, شوفت, مكان آمن, شفته, محل آمن, جاردة, بلاصة آمنة,
        Egyptian: محمية, حديقة, زيارة, رائع, جميل قوي, شوفت, رحلة, تجربة,
        حماية, مكان آمن, محمية طبيعية, حديقة وطنية,
        Levantine: محمية, حديقة, زيارة, رائع, حلو كتير, شفت, رحلة, تجربة,
        حماية, محل آمن, محمية طبيعية, حديقة وطنية,
        Gulf: محمية, حديقة, زيارة, رائع, حلو مرة, شفت, رحلة, تجربة,
        حماية, مكان آمن, محمية طبيعية, حديقة وطنية,
        Maghrebi: محمية, جاردة, زيارة, رائع, زوين بزاف, شفت, رحلة, تجربة,
        حماية, بلاصة آمنة, محمية طبيعية, جاردة وطنية
    """
}

# =============================================================================
# SBERT TOPIC MODELING FUNCTION (SAME AS BEFORE)
# =============================================================================

def apply_sbert_topic_modeling(df, message_column='Message',
                               similarity_threshold=0.3,
                               model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
                               batch_size=32):
    """
    Apply SBERT topic modeling to dataframe with 12 merged topics

    Parameters:
    -----------
    df : pandas.DataFrame
        Your dataframe with posts
    message_column : str
        Column containing post text
    similarity_threshold : float
        Minimum cosine similarity for topic assignment (default: 0.3)
    model_name : str
        SBERT model to use
    batch_size : int
        Batch size for encoding

    Returns:
    --------
    df with new columns: 'Assigned_Topic', 'Topic_Similarity', 'Topic_Confidence'
    """

    print("="*100)
    print("SBERT TOPIC MODELING - 12 MERGED TOPICS".center(100))
    print("="*100)

    # Step 1: Prepare data
    print(f"\n📊 Dataset info:")
    print(f"   Total posts: {len(df):,}")
    print(f"   Message column: '{message_column}'")
    print(f"   Number of topics: {len(TOPICS_MERGED)}")

    df = df.copy()

    # Handle missing messages
    df[message_column] = df[message_column].fillna('')
    df[message_column] = df[message_column].astype(str)

    # Filter out empty messages
    df_valid = df[df[message_column].str.strip() != ''].copy()
    print(f"   Posts with valid messages: {len(df_valid):,}")

    # Step 2: Load SBERT model
    print(f"\n🧠 Loading SBERT model: {model_name}")
    model = SentenceTransformer(model_name)
    print("   ✅ Model loaded successfully")

    # Step 3: Encode topic descriptions
    print(f"\n📝 Encoding {len(TOPICS_MERGED)} topic descriptions...")
    topic_names = list(TOPICS_MERGED.keys())
    topic_descriptions = list(TOPICS_MERGED.values())

    topic_embeddings = model.encode(topic_descriptions,
                                    show_progress_bar=True,
                                    batch_size=batch_size)
    print("   ✅ Topic embeddings created")

    # Step 4: Encode post messages
    print(f"\n📝 Encoding {len(df_valid):,} post messages...")
    print("   (This may take several minutes depending on dataset size)")

    post_embeddings = model.encode(df_valid[message_column].tolist(),
                                   show_progress_bar=True,
                                   batch_size=batch_size)
    print("   ✅ Post embeddings created")

    # Step 5: Compute similarities
    print("\n🔍 Computing cosine similarities...")
    similarities = cosine_similarity(post_embeddings, topic_embeddings)
    print("   ✅ Similarities computed")

    # Step 6: Assign topics
    print(f"\n🎯 Assigning topics (threshold: {similarity_threshold})...")

    # Get best topic for each post
    max_similarities = similarities.max(axis=1)
    best_topic_indices = similarities.argmax(axis=1)

    # Assign topics
    assigned_topics = []
    topic_similarities = []
    topic_confidences = []

    for idx, (best_idx, max_sim) in enumerate(zip(best_topic_indices, max_similarities)):
        if max_sim >= similarity_threshold:
            assigned_topics.append(topic_names[best_idx])
            topic_similarities.append(max_sim)

            # Calculate confidence (normalized difference from second-best)
            sims = similarities[idx]
            sorted_sims = np.sort(sims)[::-1]
            if len(sorted_sims) > 1:
                confidence = (sorted_sims[0] - sorted_sims[1]) / sorted_sims[0]
            else:
                confidence = 1.0
            topic_confidences.append(confidence)
        else:
            assigned_topics.append('UNASSIGNED')
            topic_similarities.append(max_sim)
            topic_confidences.append(0.0)

    # Add to dataframe
    df_valid['Assigned_Topic'] = assigned_topics
    df_valid['Topic_Similarity'] = topic_similarities
    df_valid['Topic_Confidence'] = topic_confidences

    # Step 7: Merge back to original dataframe
    df['Assigned_Topic'] = 'UNASSIGNED'
    df['Topic_Similarity'] = 0.0
    df['Topic_Confidence'] = 0.0

    df.loc[df_valid.index, 'Assigned_Topic'] = df_valid['Assigned_Topic']
    df.loc[df_valid.index, 'Topic_Similarity'] = df_valid['Topic_Similarity']
    df.loc[df_valid.index, 'Topic_Confidence'] = df_valid['Topic_Confidence']

    # Step 8: Print summary
    print("\n" + "="*100)
    print("TOPIC ASSIGNMENT SUMMARY - 12 MERGED TOPICS".center(100))
    print("="*100)

    topic_counts = df['Assigned_Topic'].value_counts()
    print(f"\n{'Topic':<55} {'Count':>10} {'%':>8}")
    print("-"*75)
    for topic, count in topic_counts.items():
        pct = (count / len(df)) * 100
        print(f"{topic:<55} {count:>10,} {pct:>7.2f}%")

    print("\n" + "="*100)

    return df


# Apply topic modeling directly
df_with_topics = apply_sbert_topic_modeling(
    df_processed,
    message_column='Message',
    similarity_threshold=0.3,
    batch_size=32
)

# Save to CSV
df_with_topics.to_csv('Wildlife_12_topics.csv', index=False, encoding='utf-8-sig')

print("✅ Topic modeling complete!")
print(f"✅ Saved {len(df_with_topics):,} posts with topic assignments")

In [ ]:
df_with_topics.columns

In [ ]:
df_with_topics.head(1)

In [ ]:
df_with_topics.to_csv('/content/drive/MyDrive/Facebook Crowdtangle Data/All Zied Download Folder/Renamed by Rafi/Categorized by Maryam/Environment/Environment/df_with_topics.csv')

In [ ]:
"""
Comprehensive Analysis for ICWSM Paper
Deep statistical analysis, visualizations, and research question answers
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set publication-quality style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11


# =============================================================================
# DATA PREPARATION
# =============================================================================

def prepare_analysis_data(df):
    """
    Prepare data for analysis with proper type conversions
    """

    print("="*100)
    print("DATA PREPARATION".center(100))
    print("="*100)

    df = df.copy()

    # Convert numeric columns
    numeric_cols = ['Total Interactions', 'Likes', 'Comments', 'Shares',
                   'Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care',
                   'sentiment_score', 'polarization_score', 'cohesion_score',
                   'Topic_Similarity', 'Topic_Confidence']

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Calculate engagement metrics
    df['engagement_rate'] = df['Total Interactions'] / (df['Followers at Posting'] + 1)
    df['comment_rate'] = df['Comments'] / (df['Total Interactions'] + 1)
    df['share_rate'] = df['Shares'] / (df['Total Interactions'] + 1)
    df['share_to_like_ratio'] = df['Shares'] / (df['Likes'] + 1)

    # Virality classification
    virality_threshold = df['Total Interactions'].quantile(0.90)
    df['is_viral'] = (df['Total Interactions'] > virality_threshold).astype(int)

    # Date processing
    df['Post Created Date'] = pd.to_datetime(df['Post Created Date'], errors='coerce')
    df['day_of_week'] = df['Post Created Date'].dt.day_name()
    df['week'] = df['Post Created Date'].dt.isocalendar().week

    print(f"✅ Data prepared: {len(df):,} posts")
    print(f"✅ Virality threshold: {virality_threshold:.0f} interactions")

    return df


# =============================================================================
# RQ1: TOPIC DISTRIBUTION AND PREVALENCE
# =============================================================================

def analyze_rq1_topic_distribution(df):
    """
    RQ1: What are the dominant themes and topics in wildlife protection discourse,
         and how do they differ in prevalence and engagement patterns?
    """

    print("\n" + "="*100)
    print("RQ1: TOPIC DISTRIBUTION AND PREVALENCE".center(100))
    print("="*100)

    # Topic frequency
    topic_stats = df.groupby('Assigned_Topic').agg({
        'Facebook Id': 'count',
        'Total Interactions': ['sum', 'mean', 'median'],
        'Likes': 'mean',
        'Comments': 'mean',
        'Shares': 'mean',
        'engagement_rate': 'mean',
        'is_viral': 'sum'
    }).round(2)

    topic_stats.columns = ['Posts_N', 'Total_Int', 'Avg_Int', 'Median_Int',
                          'Avg_Likes', 'Avg_Comments', 'Avg_Shares',
                          'Avg_Engagement_Rate', 'Viral_Posts']

    topic_stats['Pct_Corpus'] = (topic_stats['Posts_N'] / len(df) * 100).round(2)
    topic_stats['Pct_Viral'] = (topic_stats['Viral_Posts'] / topic_stats['Posts_N'] * 100).round(2)

    topic_stats = topic_stats.sort_values('Posts_N', ascending=False)

    # Print summary
    print("\nTopic Distribution Summary:")
    print(topic_stats[['Posts_N', 'Pct_Corpus', 'Avg_Int', 'Median_Int', 'Pct_Viral']].head(15))

    # Save to CSV
    topic_stats.to_csv('RQ1_topic_distribution.csv', encoding='utf-8-sig')
    print("\n✅ Saved: RQ1_topic_distribution.csv")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # Plot 1: Topic distribution
    top_topics = topic_stats.head(12)
    ax1 = axes[0, 0]
    ax1.barh(range(len(top_topics)), top_topics['Posts_N'], color='steelblue', alpha=0.8)
    ax1.set_yticks(range(len(top_topics)))
    ax1.set_yticklabels([t[:40] + '...' if len(t) > 40 else t for t in top_topics.index], fontsize=10)
    ax1.set_xlabel('Number of Posts', fontsize=12, fontweight='bold')
    ax1.set_title('Topic Distribution (Top 12)', fontsize=14, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)

    # Add percentage labels
    for i, (idx, row) in enumerate(top_topics.iterrows()):
        ax1.text(row['Posts_N'], i, f" {row['Pct_Corpus']:.1f}%",
                va='center', fontsize=9)

    # Plot 2: Average interactions by topic
    ax2 = axes[0, 1]
    top_engagement = topic_stats.nlargest(12, 'Avg_Int')
    ax2.barh(range(len(top_engagement)), top_engagement['Avg_Int'], color='coral', alpha=0.8)
    ax2.set_yticks(range(len(top_engagement)))
    ax2.set_yticklabels([t[:40] + '...' if len(t) > 40 else t for t in top_engagement.index], fontsize=10)
    ax2.set_xlabel('Average Interactions', fontsize=12, fontweight='bold')
    ax2.set_title('Average Engagement by Topic (Top 12)', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)

    # Plot 3: Virality rate by topic
    ax3 = axes[1, 0]
    top_viral = topic_stats.nlargest(12, 'Pct_Viral')
    ax3.barh(range(len(top_viral)), top_viral['Pct_Viral'], color='green', alpha=0.8)
    ax3.set_yticks(range(len(top_viral)))
    ax3.set_yticklabels([t[:40] + '...' if len(t) > 40 else t for t in top_viral.index], fontsize=10)
    ax3.set_xlabel('Viral Posts (%)', fontsize=12, fontweight='bold')
    ax3.set_title('Virality Rate by Topic (Top 12)', fontsize=14, fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)

    # Plot 4: Engagement breakdown
    ax4 = axes[1, 1]
    top_5 = topic_stats.head(5)
    x = np.arange(len(top_5))
    width = 0.25

    ax4.bar(x - width, top_5['Avg_Likes'], width, label='Likes', alpha=0.8)
    ax4.bar(x, top_5['Avg_Comments'], width, label='Comments', alpha=0.8)
    ax4.bar(x + width, top_5['Avg_Shares'], width, label='Shares', alpha=0.8)

    ax4.set_xticks(x)
    ax4.set_xticklabels([t[:25] + '...' if len(t) > 25 else t for t in top_5.index],
                        rotation=45, ha='right', fontsize=9)
    ax4.set_ylabel('Average Count', fontsize=12, fontweight='bold')
    ax4.set_title('Engagement Type Breakdown (Top 5 Topics)', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('RQ1_topic_distribution.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: RQ1_topic_distribution.png")
    plt.close()

    return topic_stats


# =============================================================================
# RQ2: EMOTIONAL POLARIZATION PATTERNS
# =============================================================================

def analyze_rq2_emotional_polarization(df):
    """
    RQ2: How does emotional polarization vary across different wildlife conservation
         topics, and what characteristics make certain topics more divisive?
    """

    print("\n" + "="*100)
    print("RQ2: EMOTIONAL POLARIZATION PATTERNS".center(100))
    print("="*100)

    # Remove UNASSIGNED
    df_topics = df[df['Assigned_Topic'] != 'UNASSIGNED'].copy()

    # Emotional metrics by topic
    emotional_stats = df_topics.groupby('Assigned_Topic').agg({
        'cohesion_score': ['mean', 'median', 'std'],
        'polarization_score': ['mean', 'median', 'std'],
        'Love': 'median',
        'Sad': 'median',
        'Angry': 'median',
        'Care': 'median'
    }).round(3)

    emotional_stats.columns = ['Cohesion_Mean', 'Cohesion_Median', 'Cohesion_Std',
                               'Polarization_Mean', 'Polarization_Median', 'Polarization_Std',
                               'Love_Median', 'Sad_Median', 'Angry_Median', 'Care_Median']

    # Calculate high/low cohesion percentages
    for topic in df_topics['Assigned_Topic'].unique():
        topic_data = df_topics[df_topics['Assigned_Topic'] == topic]
        high_coh = (topic_data['cohesion_score'] >= 0.66).sum()
        low_coh = (topic_data['cohesion_score'] < 0.33).sum()

        emotional_stats.loc[topic, 'Pct_High_Cohesion'] = (high_coh / len(topic_data) * 100)
        emotional_stats.loc[topic, 'Pct_Low_Cohesion'] = (low_coh / len(topic_data) * 100)

    emotional_stats = emotional_stats.sort_values('Cohesion_Mean', ascending=False)

    print("\nEmotional Polarization Summary:")
    print(emotional_stats[['Cohesion_Mean', 'Polarization_Mean',
                          'Pct_High_Cohesion', 'Pct_Low_Cohesion']].head(15))

    # Save
    emotional_stats.to_csv('RQ2_emotional_polarization.csv', encoding='utf-8-sig')
    print("\n✅ Saved: RQ2_emotional_polarization.csv")

    # Statistical test: Kruskal-Wallis
    topics = df_topics['Assigned_Topic'].unique()
    cohesion_by_topic = [df_topics[df_topics['Assigned_Topic'] == t]['cohesion_score'].dropna()
                        for t in topics]
    h_stat, p_value = kruskal(*cohesion_by_topic)

    print(f"\n📊 Kruskal-Wallis Test (Cohesion across topics):")
    print(f"   H-statistic: {h_stat:.4f}")
    print(f"   P-value: {p_value:.4e}")
    print(f"   Result: {'Significant differences exist' if p_value < 0.05 else 'No significant differences'}")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    # Plot 1: Cohesion vs Polarization scatter
    ax1 = axes[0, 0]
    topic_means = df_topics.groupby('Assigned_Topic').agg({
        'cohesion_score': 'mean',
        'polarization_score': 'mean',
        'Facebook Id': 'count'
    })

    scatter = ax1.scatter(topic_means['cohesion_score'],
                         topic_means['polarization_score'],
                         s=topic_means['Facebook Id']/10,
                         alpha=0.6,
                         c=range(len(topic_means)),
                         cmap='viridis',
                         edgecolors='black',
                         linewidth=1)

    ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
    ax1.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Mean Cohesion Score', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Mean Polarization Score', fontsize=13, fontweight='bold')
    ax1.set_title('Cohesion vs Polarization by Topic', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)

    # Annotate top 5 topics
    for topic in topic_means.nlargest(5, 'Facebook Id').index:
        ax1.annotate(topic[:20] + '...',
                    xy=(topic_means.loc[topic, 'cohesion_score'],
                        topic_means.loc[topic, 'polarization_score']),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8, bbox=dict(boxstyle='round,pad=0.3',
                                         facecolor='yellow', alpha=0.3))

    # Plot 2: Cohesion distribution by top topics
    ax2 = axes[0, 1]
    top_10_topics = df_topics['Assigned_Topic'].value_counts().head(10).index
    cohesion_data = [df_topics[df_topics['Assigned_Topic'] == t]['cohesion_score'].dropna()
                     for t in top_10_topics]

    bp = ax2.boxplot(cohesion_data, labels=[t[:15] + '...' for t in top_10_topics],
                     patch_artist=True, showmeans=True)

    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)

    ax2.set_xticklabels([t[:15] + '...' for t in top_10_topics],
                       rotation=45, ha='right', fontsize=9)
    ax2.set_ylabel('Cohesion Score', fontsize=13, fontweight='bold')
    ax2.set_title('Cohesion Distribution (Top 10 Topics)', fontsize=14, fontweight='bold')
    ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
    ax2.grid(axis='y', alpha=0.3)

    # Plot 3: Emotional reactions heatmap
    ax3 = axes[1, 0]
    reaction_cols = ['Love_Median', 'Sad_Median', 'Angry_Median', 'Care_Median']
    top_12 = emotional_stats.head(12)
    heatmap_data = top_12[reaction_cols].values

    im = ax3.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')
    ax3.set_xticks(range(len(reaction_cols)))
    ax3.set_xticklabels(['Love', 'Sad', 'Angry', 'Care'], fontsize=11)
    ax3.set_yticks(range(len(top_12)))
    ax3.set_yticklabels([t[:35] + '...' if len(t) > 35 else t for t in top_12.index], fontsize=9)
    ax3.set_title('Median Emotional Reactions by Topic (Top 12)', fontsize=14, fontweight='bold')

    plt.colorbar(im, ax=ax3, label='Median Count')

    # Plot 4: High vs Low Cohesion comparison
    ax4 = axes[1, 1]
    top_12_coh = emotional_stats.head(12)
    x = np.arange(len(top_12_coh))
    width = 0.35

    ax4.bar(x - width/2, top_12_coh['Pct_High_Cohesion'], width,
           label='High Cohesion (≥0.66)', color='green', alpha=0.7)
    ax4.bar(x + width/2, top_12_coh['Pct_Low_Cohesion'], width,
           label='Low Cohesion (<0.33)', color='red', alpha=0.7)

    ax4.set_xticks(x)
    ax4.set_xticklabels([t[:20] + '...' if len(t) > 20 else t for t in top_12_coh.index],
                       rotation=45, ha='right', fontsize=9)
    ax4.set_ylabel('Percentage (%)', fontsize=13, fontweight='bold')
    ax4.set_title('Cohesion Level Distribution (Top 12)', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('RQ2_emotional_polarization.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: RQ2_emotional_polarization.png")
    plt.close()

    return emotional_stats


# =============================================================================
# RQ3: ENGAGEMENT AND VIRALITY PATTERNS
# =============================================================================

def analyze_rq3_engagement_virality(df):
    """
    RQ3: What is the relationship between emotional dynamics and engagement/virality?
    """

    print("\n" + "="*100)
    print("RQ3: ENGAGEMENT AND VIRALITY PATTERNS".center(100))
    print("="*100)

    # Correlation analysis
    correlation_vars = ['cohesion_score', 'polarization_score', 'sentiment_score',
                       'Total Interactions', 'Likes', 'Comments', 'Shares',
                       'engagement_rate', 'share_to_like_ratio']

    corr_df = df[correlation_vars].corr(method='spearman')

    print("\nSpearman Correlation Matrix:")
    print(corr_df[['Total Interactions', 'engagement_rate', 'share_to_like_ratio']].round(3))

    # Save correlation matrix
    corr_df.to_csv('RQ3_correlation_matrix.csv', encoding='utf-8-sig')
    print("\n✅ Saved: RQ3_correlation_matrix.csv")

    # Viral vs Non-viral comparison
    viral_comparison = df.groupby('is_viral').agg({
        'cohesion_score': 'mean',
        'polarization_score': 'mean',
        'sentiment_score': 'mean',
        'Likes': 'mean',
        'Comments': 'mean',
        'Shares': 'mean'
    }).T

    viral_comparison.columns = ['Non-Viral', 'Viral']
    viral_comparison['Difference'] = viral_comparison['Viral'] - viral_comparison['Non-Viral']
    viral_comparison['Pct_Change'] = (viral_comparison['Difference'] /
                                     viral_comparison['Non-Viral'] * 100).round(2)

    print("\nViral vs Non-Viral Comparison:")
    print(viral_comparison.round(3))

    # Statistical tests
    viral_posts = df[df['is_viral'] == 1]
    non_viral_posts = df[df['is_viral'] == 0]

    u_stat_coh, p_coh = mannwhitneyu(viral_posts['cohesion_score'].dropna(),
                                     non_viral_posts['cohesion_score'].dropna())
    u_stat_pol, p_pol = mannwhitneyu(viral_posts['polarization_score'].dropna(),
                                     non_viral_posts['polarization_score'].dropna())

    print(f"\n📊 Mann-Whitney U Test (Viral vs Non-Viral):")
    print(f"   Cohesion: U={u_stat_coh:.0f}, p={p_coh:.4e}")
    print(f"   Polarization: U={u_stat_pol:.0f}, p={p_pol:.4e}")

    # Visualization
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

    # Plot 1: Correlation heatmap
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0,
               square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax1)
    ax1.set_title('Spearman Correlation Matrix', fontsize=14, fontweight='bold', pad=15)

    # Plot 2: Cohesion vs Engagement scatter
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.scatter(df['cohesion_score'], df['Total Interactions'],
               alpha=0.3, s=20, c='blue')
    ax2.set_xlabel('Cohesion Score', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Total Interactions', fontsize=11, fontweight='bold')
    ax2.set_title('Cohesion vs Engagement', fontsize=12, fontweight='bold')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)

    # Add correlation coefficient
    corr_coef = df[['cohesion_score', 'Total Interactions']].corr(method='spearman').iloc[0, 1]
    ax2.text(0.05, 0.95, f'ρ = {corr_coef:.3f}',
            transform=ax2.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Plot 3: Polarization vs Engagement scatter
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.scatter(df['polarization_score'], df['Total Interactions'],
               alpha=0.3, s=20, c='red')
    ax3.set_xlabel('Polarization Score', fontsize=11, fontweight='bold')
    ax3.set_ylabel('Total Interactions', fontsize=11, fontweight='bold')
    ax3.set_title('Polarization vs Engagement', fontsize=12, fontweight='bold')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3)

    corr_coef = df[['polarization_score', 'Total Interactions']].corr(method='spearman').iloc[0, 1]
    ax3.text(0.05, 0.95, f'ρ = {corr_coef:.3f}',
            transform=ax3.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Plot 4: Viral vs Non-Viral comparison
    ax4 = fig.add_subplot(gs[2, :])
    metrics = ['cohesion_score', 'polarization_score', 'Likes', 'Comments', 'Shares']
    x = np.arange(len(metrics))
    width = 0.35

    non_viral_means = [non_viral_posts[m].mean() for m in metrics]
    viral_means = [viral_posts[m].mean() for m in metrics]

    # Normalize for visualization
    non_viral_norm = [v / max(non_viral_means[i], viral_means[i]) for i, v in enumerate(non_viral_means)]
    viral_norm = [v / max(non_viral_means[i], viral_means[i]) for i, v in enumerate(viral_means)]

    ax4.bar(x - width/2, non_viral_norm, width, label='Non-Viral', alpha=0.8, color='lightblue')
    ax4.bar(x + width/2, viral_norm, width, label='Viral', alpha=0.8, color='orange')

    ax4.set_xticks(x)
    ax4.set_xticklabels(['Cohesion', 'Polarization', 'Likes', 'Comments', 'Shares'], fontsize=11)
    ax4.set_ylabel('Normalized Score', fontsize=12, fontweight='bold')
    ax4.set_title('Viral vs Non-Viral Posts Comparison (Normalized)', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.grid(axis='y', alpha=0.3)

    plt.savefig('RQ3_engagement_virality.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: RQ3_engagement_virality.png")
    plt.close()

    return corr_df, viral_comparison


# =============================================================================
# RQ4: GEOGRAPHIC PATTERNS
# =============================================================================

def analyze_rq4_geographic_patterns(df):
    """
    RQ4: How does wildlife conservation discourse vary across different
         Arabic-speaking countries?
    """

    print("\n" + "="*100)
    print("RQ4: GEOGRAPHIC PATTERNS".center(100))
    print("="*100)

    # Country-level analysis
    country_stats = df.groupby('Page Admin Top Country').agg({
        'Facebook Id': 'count',
        'Total Interactions': ['sum', 'mean'],
        'cohesion_score': 'mean',
        'polarization_score': 'mean',
        'is_viral': 'sum'
    }).round(2)

    country_stats.columns = ['Posts_N', 'Total_Int', 'Avg_Int',
                            'Avg_Cohesion', 'Avg_Polarization', 'Viral_Posts']
    country_stats['Pct_Viral'] = (country_stats['Viral_Posts'] /
                                   country_stats['Posts_N'] * 100).round(2)
    country_stats = country_stats.sort_values('Posts_N', ascending=False).head(15)

    print("\nTop 15 Countries:")
    print(country_stats)

    # Dominant topics by country
    country_topics = []
    for country in country_stats.index:
        country_df = df[df['Page Admin Top Country'] == country]
        top_topics = country_df['Assigned_Topic'].value_counts().head(3)
        topics_str = ', '.join([f"{t[:25]}({v})" for t, v in top_topics.items()])
        country_topics.append(topics_str)

    country_stats['Top_3_Topics'] = country_topics

    # Save
    country_stats.to_csv('RQ4_geographic_patterns.csv', encoding='utf-8-sig')
    print("\n✅ Saved: RQ4_geographic_patterns.csv")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # Plot 1: Posts by country
    ax1 = axes[0, 0]
    top_10 = country_stats.head(10)
    ax1.barh(range(len(top_10)), top_10['Posts_N'], color='teal', alpha=0.8)
    ax1.set_yticks(range(len(top_10)))
    ax1.set_yticklabels(top_10.index, fontsize=11)
    ax1.set_xlabel('Number of Posts', fontsize=12, fontweight='bold')
    ax1.set_title('Posts by Country (Top 10)', fontsize=14, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)

    # Plot 2: Average engagement by country
    ax2 = axes[0, 1]
    ax2.barh(range(len(top_10)), top_10['Avg_Int'], color='coral', alpha=0.8)
    ax2.set_yticks(range(len(top_10)))
    ax2.set_yticklabels(top_10.index, fontsize=11)
    ax2.set_xlabel('Average Interactions', fontsize=12, fontweight='bold')
    ax2.set_title('Average Engagement by Country (Top 10)', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)

    # Plot 3: Cohesion by country
    ax3 = axes[1, 0]
    top_10_coh = country_stats.nlargest(10, 'Avg_Cohesion')
    ax3.barh(range(len(top_10_coh)), top_10_coh['Avg_Cohesion'], color='green', alpha=0.8)
    ax3.set_yticks(range(len(top_10_coh)))
    ax3.set_yticklabels(top_10_coh.index, fontsize=11)
    ax3.set_xlabel('Average Cohesion Score', fontsize=12, fontweight='bold')
    ax3.set_title('Cohesion by Country (Top 10)', fontsize=14, fontweight='bold')
    ax3.axvline(x=0.5, color='red', linestyle='--', alpha=0.5)
    ax3.grid(axis='x', alpha=0.3)

    # Plot 4: Virality rate by country
    ax4 = axes[1, 1]
    top_10_viral = country_stats.nlargest(10, 'Pct_Viral')
    ax4.barh(range(len(top_10_viral)), top_10_viral['Pct_Viral'], color='purple', alpha=0.8)
    ax4.set_yticks(range(len(top_10_viral)))
    ax4.set_yticklabels(top_10_viral.index, fontsize=11)
    ax4.set_xlabel('Viral Posts (%)', fontsize=12, fontweight='bold')
    ax4.set_title('Virality Rate by Country (Top 10)', fontsize=14, fontweight='bold')
    ax4.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig('RQ4_geographic_patterns.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: RQ4_geographic_patterns.png")
    plt.close()

    return country_stats


# =============================================================================
# RQ5: CONTENT TYPE ANALYSIS
# =============================================================================

def analyze_rq5_content_type(df):
    """
    RQ5: How do different content types (Photo, Video, Status) perform
         in terms of engagement and emotional dynamics?
    """

    print("\n" + "="*100)
    print("RQ5: CONTENT TYPE ANALYSIS".center(100))
    print("="*100)

    # Content type analysis
    content_stats = df.groupby('Type').agg({
        'Facebook Id': 'count',
        'Total Interactions': ['sum', 'mean', 'median'],
        'Likes': 'mean',
        'Comments': 'mean',
        'Shares': 'mean',
        'cohesion_score': 'mean',
        'polarization_score': 'mean',
        'is_viral': 'sum'
    }).round(2)

    content_stats.columns = ['Posts_N', 'Total_Int', 'Avg_Int', 'Median_Int',
                            'Avg_Likes', 'Avg_Comments', 'Avg_Shares',
                            'Avg_Cohesion', 'Avg_Polarization', 'Viral_Posts']

    content_stats['Pct_Corpus'] = (content_stats['Posts_N'] / len(df) * 100).round(2)
    content_stats['Pct_Viral'] = (content_stats['Viral_Posts'] / content_stats['Posts_N'] * 100).round(2)
    content_stats = content_stats.sort_values('Posts_N', ascending=False)

    print("\nContent Type Summary:")
    print(content_stats)

    # Chi-square test: Content type vs Virality
    contingency = pd.crosstab(df['Type'], df['is_viral'])
    chi2, p_value, dof, expected = chi2_contingency(contingency)

    print(f"\n📊 Chi-Square Test (Content Type vs Virality):")
    print(f"   χ² = {chi2:.4f}")
    print(f"   P-value = {p_value:.4e}")
    print(f"   Result: {'Significant association' if p_value < 0.05 else 'No significant association'}")

    # Save
    content_stats.to_csv('RQ5_content_type.csv', encoding='utf-8-sig')
    print("\n✅ Saved: RQ5_content_type.csv")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    # Plot 1: Content type distribution
    ax1 = axes[0, 0]
    top_types = content_stats.head(8)
    ax1.bar(range(len(top_types)), top_types['Posts_N'], color='steelblue', alpha=0.8)
    ax1.set_xticks(range(len(top_types)))
    ax1.set_xticklabels(top_types.index, rotation=45, ha='right', fontsize=10)
    ax1.set_ylabel('Number of Posts', fontsize=12, fontweight='bold')
    ax1.set_title('Content Type Distribution', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)

    # Plot 2: Average engagement by content type
    ax2 = axes[0, 1]
    ax2.bar(range(len(top_types)), top_types['Avg_Int'], color='orange', alpha=0.8)
    ax2.set_xticks(range(len(top_types)))
    ax2.set_xticklabels(top_types.index, rotation=45, ha='right', fontsize=10)
    ax2.set_ylabel('Average Interactions', fontsize=12, fontweight='bold')
    ax2.set_title('Average Engagement by Content Type', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # Plot 3: Engagement breakdown
    ax3 = axes[1, 0]
    x = np.arange(len(top_types))
    width = 0.25

    ax3.bar(x - width, top_types['Avg_Likes'], width, label='Likes', alpha=0.8)
    ax3.bar(x, top_types['Avg_Comments'], width, label='Comments', alpha=0.8)
    ax3.bar(x + width, top_types['Avg_Shares'], width, label='Shares', alpha=0.8)

    ax3.set_xticks(x)
    ax3.set_xticklabels(top_types.index, rotation=45, ha='right', fontsize=10)
    ax3.set_ylabel('Average Count', fontsize=12, fontweight='bold')
    ax3.set_title('Engagement Type by Content Type', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(axis='y', alpha=0.3)

    # Plot 4: Cohesion & Polarization by content type
    ax4 = axes[1, 1]
    x = np.arange(len(top_types))
    width = 0.35

    ax4.bar(x - width/2, top_types['Avg_Cohesion'], width,
           label='Cohesion', color='green', alpha=0.7)
    ax4.bar(x + width/2, top_types['Avg_Polarization'], width,
           label='Polarization', color='red', alpha=0.7)

    ax4.set_xticks(x)
    ax4.set_xticklabels(top_types.index, rotation=45, ha='right', fontsize=10)
    ax4.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax4.set_title('Emotional Dynamics by Content Type', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.axhline(y=0.5, color='black', linestyle='--', alpha=0.3)
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('RQ5_content_type.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: RQ5_content_type.png")
    plt.close()

    return content_stats


# =============================================================================
# COMPREHENSIVE SUMMARY REPORT
# =============================================================================

def generate_summary_report(df, topic_stats, emotional_stats, corr_df,
                           country_stats, content_stats):
    """
    Generate comprehensive summary report for ICWSM paper
    """

    print("\n" + "="*100)
    print("GENERATING COMPREHENSIVE SUMMARY REPORT".center(100))
    print("="*100)

    with open('ICWSM_Analysis_Summary_Report.txt', 'w', encoding='utf-8') as f:
        f.write("="*100 + "\n")
        f.write("COMPREHENSIVE ANALYSIS REPORT FOR ICWSM PAPER\n")
        f.write("Wildlife Conservation Discourse on Arabic Facebook\n")
        f.write("="*100 + "\n\n")

        # Dataset overview
        f.write("DATASET OVERVIEW\n")
        f.write("-"*100 + "\n")
        f.write(f"Total Posts: {len(df):,}\n")
        f.write(f"Date Range: {df['Post Created Date'].min()} to {df['Post Created Date'].max()}\n")
        f.write(f"Number of Topics: {df['Assigned_Topic'].nunique()}\n")
        f.write(f"Number of Countries: {df['Page Admin Top Country'].nunique()}\n")
        f.write(f"Number of Content Types: {df['Type'].nunique()}\n\n")

        # RQ1 Summary
        f.write("RQ1: TOPIC DISTRIBUTION AND PREVALENCE\n")
        f.write("-"*100 + "\n")
        f.write("Top 10 Topics by Post Count:\n")
        f.write(topic_stats[['Posts_N', 'Pct_Corpus', 'Avg_Int', 'Pct_Viral']].head(10).to_string())
        f.write("\n\n")

        # RQ2 Summary
        f.write("RQ2: EMOTIONAL POLARIZATION PATTERNS\n")
        f.write("-"*100 + "\n")
        f.write("Top 10 Topics by Cohesion:\n")
        f.write(emotional_stats[['Cohesion_Mean', 'Polarization_Mean',
                                'Pct_High_Cohesion', 'Pct_Low_Cohesion']].head(10).to_string())
        f.write("\n\n")

        # RQ3 Summary
        f.write("RQ3: ENGAGEMENT AND VIRALITY PATTERNS\n")
        f.write("-"*100 + "\n")
        f.write("Key Correlations with Total Interactions:\n")
        f.write(corr_df['Total Interactions'].sort_values(ascending=False).to_string())
        f.write("\n\n")

        # RQ4 Summary
        f.write("RQ4: GEOGRAPHIC PATTERNS\n")
        f.write("-"*100 + "\n")
        f.write("Top 10 Countries:\n")
        f.write(country_stats[['Posts_N', 'Avg_Int', 'Avg_Cohesion', 'Pct_Viral']].head(10).to_string())
        f.write("\n\n")

        # RQ5 Summary
        f.write("RQ5: CONTENT TYPE ANALYSIS\n")
        f.write("-"*100 + "\n")
        f.write(content_stats[['Posts_N', 'Pct_Corpus', 'Avg_Int', 'Avg_Cohesion', 'Pct_Viral']].to_string())
        f.write("\n\n")

        f.write("="*100 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*100 + "\n")

    print("✅ Saved: ICWSM_Analysis_Summary_Report.txt")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_complete_analysis(df):
    """
    Run complete analysis for all research questions
    """

    print("\n" + "="*100)
    print("ICWSM PAPER - COMPREHENSIVE ANALYSIS".center(100))
    print("="*100)

    # Prepare data
    df = prepare_analysis_data(df)

    # Run all analyses
    topic_stats = analyze_rq1_topic_distribution(df)
    emotional_stats = analyze_rq2_emotional_polarization(df)
    corr_df, viral_comparison = analyze_rq3_engagement_virality(df)
    country_stats = analyze_rq4_geographic_patterns(df)
    content_stats = analyze_rq5_content_type(df)

    # Generate summary report
    generate_summary_report(df, topic_stats, emotional_stats, corr_df,
                           country_stats, content_stats)

    print("\n" + "="*100)
    print("ANALYSIS COMPLETE!".center(100))
    print("="*100)

    print("\n📄 Generated Files:")
    print("   • RQ1_topic_distribution.csv & .png")
    print("   • RQ2_emotional_polarization.csv & .png")
    print("   • RQ3_correlation_matrix.csv & RQ3_engagement_virality.png")
    print("   • RQ4_geographic_patterns.csv & .png")
    print("   • RQ5_content_type.csv & .png")
    print("   • ICWSM_Analysis_Summary_Report.txt")

    return {
        'topic_stats': topic_stats,
        'emotional_stats': emotional_stats,
        'correlation': corr_df,
        'viral_comparison': viral_comparison,
        'country_stats': country_stats,
        'content_stats': content_stats
    }


# =============================================================================
# USAGE
# =============================================================================

if __name__ == "__main__":

    # Load your dataframe
    #df = pd.read_csv('/content/Wildlife_12_topics.csv')

    # Run complete analysis
    results = run_complete_analysis(df)

    pass

In [ ]:
"""
Enhanced Comprehensive Analysis for ICWSM Paper
Includes overall statistics and label-based analyses
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set publication-quality style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11


# =============================================================================
# OVERALL DESCRIPTIVE STATISTICS
# =============================================================================

def analyze_overall_statistics(df):
    """
    Generate comprehensive overall descriptive statistics
    """

    print("\n" + "="*100)
    print("OVERALL DESCRIPTIVE STATISTICS".center(100))
    print("="*100)

    overall_stats = {
        'Metric': [],
        'Count': [],
        'Mean': [],
        'Median': [],
        'Std': [],
        'Min': [],
        'Max': [],
        'Q1': [],
        'Q3': []
    }

    # Engagement metrics
    engagement_vars = ['Total Interactions', 'Likes', 'Comments', 'Shares',
                      'Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care']

    for var in engagement_vars:
        if var in df.columns:
            data = df[var].dropna()
            overall_stats['Metric'].append(var)
            overall_stats['Count'].append(len(data))
            overall_stats['Mean'].append(data.mean())
            overall_stats['Median'].append(data.median())
            overall_stats['Std'].append(data.std())
            overall_stats['Min'].append(data.min())
            overall_stats['Max'].append(data.max())
            overall_stats['Q1'].append(data.quantile(0.25))
            overall_stats['Q3'].append(data.quantile(0.75))

    # Emotional metrics
    emotional_vars = ['sentiment_score', 'cohesion_score', 'polarization_score']

    for var in emotional_vars:
        if var in df.columns:
            data = df[var].dropna()
            overall_stats['Metric'].append(var)
            overall_stats['Count'].append(len(data))
            overall_stats['Mean'].append(data.mean())
            overall_stats['Median'].append(data.median())
            overall_stats['Std'].append(data.std())
            overall_stats['Min'].append(data.min())
            overall_stats['Max'].append(data.max())
            overall_stats['Q1'].append(data.quantile(0.25))
            overall_stats['Q3'].append(data.quantile(0.75))

    overall_df = pd.DataFrame(overall_stats).round(3)

    print("\nOverall Statistics Summary:")
    print(overall_df.to_string(index=False))

    # Save
    overall_df.to_csv('Overall_Descriptive_Statistics.csv', index=False, encoding='utf-8-sig')
    print("\n✅ Saved: Overall_Descriptive_Statistics.csv")

    # Categorical variables summary
    categorical_summary = {
        'Variable': [],
        'Unique_Values': [],
        'Most_Common': [],
        'Most_Common_Count': [],
        'Most_Common_Pct': []
    }

    categorical_vars = ['Assigned_Topic', 'Type', 'Page Admin Top Country',
                       'sentiment_label', 'cohesion_label', 'polarization_label']

    for var in categorical_vars:
        if var in df.columns:
            value_counts = df[var].value_counts()
            categorical_summary['Variable'].append(var)
            categorical_summary['Unique_Values'].append(len(value_counts))
            categorical_summary['Most_Common'].append(value_counts.index[0])
            categorical_summary['Most_Common_Count'].append(value_counts.iloc[0])
            categorical_summary['Most_Common_Pct'].append(
                (value_counts.iloc[0] / len(df) * 100).round(2)
            )

    categorical_df = pd.DataFrame(categorical_summary)

    print("\n\nCategorical Variables Summary:")
    print(categorical_df.to_string(index=False))

    categorical_df.to_csv('Overall_Categorical_Summary.csv', index=False, encoding='utf-8-sig')
    print("\n✅ Saved: Overall_Categorical_Summary.csv")

    return overall_df, categorical_df


# =============================================================================
# SENTIMENT LABEL ANALYSIS
# =============================================================================

def analyze_sentiment_labels(df):
    """
    Analyze posts by sentiment label (Positive, Negative, Neutral)
    """

    print("\n" + "="*100)
    print("SENTIMENT LABEL ANALYSIS".center(100))
    print("="*100)

    if 'sentiment_label' not in df.columns:
        print("⚠️  sentiment_label column not found. Skipping...")
        return None

    # Sentiment distribution
    sentiment_stats = df.groupby('sentiment_label').agg({
        'Facebook Id': 'count',
        'Total Interactions': ['sum', 'mean', 'median'],
        'Likes': 'mean',
        'Comments': 'mean',
        'Shares': 'mean',
        'cohesion_score': 'mean',
        'polarization_score': 'mean',
        'is_viral': 'sum'
    }).round(2)

    sentiment_stats.columns = ['Posts_N', 'Total_Int', 'Avg_Int', 'Median_Int',
                               'Avg_Likes', 'Avg_Comments', 'Avg_Shares',
                               'Avg_Cohesion', 'Avg_Polarization', 'Viral_Posts']

    sentiment_stats['Pct_Corpus'] = (sentiment_stats['Posts_N'] / len(df) * 100).round(2)
    sentiment_stats['Pct_Viral'] = (sentiment_stats['Viral_Posts'] /
                                    sentiment_stats['Posts_N'] * 100).round(2)

    print("\nSentiment Label Distribution:")
    print(sentiment_stats)

    # Statistical test
    sentiment_groups = df['sentiment_label'].unique()
    engagement_by_sentiment = [df[df['sentiment_label'] == s]['Total Interactions'].dropna()
                               for s in sentiment_groups]
    h_stat, p_value = kruskal(*engagement_by_sentiment)

    print(f"\n📊 Kruskal-Wallis Test (Engagement across sentiment labels):")
    print(f"   H-statistic: {h_stat:.4f}")
    print(f"   P-value: {p_value:.4e}")

    # Save
    sentiment_stats.to_csv('Sentiment_Label_Analysis.csv', encoding='utf-8-sig')
    print("\n✅ Saved: Sentiment_Label_Analysis.csv")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Plot 1: Distribution
    ax1 = axes[0, 0]
    sentiment_stats_sorted = sentiment_stats.sort_values('Posts_N', ascending=False)
    ax1.bar(range(len(sentiment_stats_sorted)), sentiment_stats_sorted['Posts_N'],
           color=['green', 'red', 'gray'][:len(sentiment_stats_sorted)], alpha=0.7)
    ax1.set_xticks(range(len(sentiment_stats_sorted)))
    ax1.set_xticklabels(sentiment_stats_sorted.index, fontsize=12)
    ax1.set_ylabel('Number of Posts', fontsize=13, fontweight='bold')
    ax1.set_title('Post Distribution by Sentiment Label', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)

    # Add percentage labels
    for i, (idx, row) in enumerate(sentiment_stats_sorted.iterrows()):
        ax1.text(i, row['Posts_N'], f"{row['Pct_Corpus']:.1f}%",
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    # Plot 2: Average engagement
    ax2 = axes[0, 1]
    ax2.bar(range(len(sentiment_stats_sorted)), sentiment_stats_sorted['Avg_Int'],
           color=['green', 'red', 'gray'][:len(sentiment_stats_sorted)], alpha=0.7)
    ax2.set_xticks(range(len(sentiment_stats_sorted)))
    ax2.set_xticklabels(sentiment_stats_sorted.index, fontsize=12)
    ax2.set_ylabel('Average Interactions', fontsize=13, fontweight='bold')
    ax2.set_title('Average Engagement by Sentiment Label', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # Plot 3: Engagement breakdown
    ax3 = axes[1, 0]
    x = np.arange(len(sentiment_stats_sorted))
    width = 0.25

    ax3.bar(x - width, sentiment_stats_sorted['Avg_Likes'], width,
           label='Likes', alpha=0.8, color='steelblue')
    ax3.bar(x, sentiment_stats_sorted['Avg_Comments'], width,
           label='Comments', alpha=0.8, color='orange')
    ax3.bar(x + width, sentiment_stats_sorted['Avg_Shares'], width,
           label='Shares', alpha=0.8, color='green')

    ax3.set_xticks(x)
    ax3.set_xticklabels(sentiment_stats_sorted.index, fontsize=12)
    ax3.set_ylabel('Average Count', fontsize=13, fontweight='bold')
    ax3.set_title('Engagement Type by Sentiment Label', fontsize=14, fontweight='bold')
    ax3.legend(fontsize=11)
    ax3.grid(axis='y', alpha=0.3)

    # Plot 4: Cohesion & Polarization
    ax4 = axes[1, 1]
    x = np.arange(len(sentiment_stats_sorted))
    width = 0.35

    ax4.bar(x - width/2, sentiment_stats_sorted['Avg_Cohesion'], width,
           label='Cohesion', alpha=0.7, color='green')
    ax4.bar(x + width/2, sentiment_stats_sorted['Avg_Polarization'], width,
           label='Polarization', alpha=0.7, color='red')

    ax4.set_xticks(x)
    ax4.set_xticklabels(sentiment_stats_sorted.index, fontsize=12)
    ax4.set_ylabel('Score', fontsize=13, fontweight='bold')
    ax4.set_title('Emotional Dynamics by Sentiment Label', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.axhline(y=0.5, color='black', linestyle='--', alpha=0.3)
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('Sentiment_Label_Analysis.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: Sentiment_Label_Analysis.png")
    plt.close()

    return sentiment_stats


# =============================================================================
# COHESION LABEL ANALYSIS
# =============================================================================

def analyze_cohesion_labels(df):
    """
    Analyze posts by cohesion label (High, Medium, Low)
    """

    print("\n" + "="*100)
    print("COHESION LABEL ANALYSIS".center(100))
    print("="*100)

    if 'cohesion_label' not in df.columns:
        print("⚠️  cohesion_label column not found. Skipping...")
        return None

    # Cohesion distribution
    cohesion_stats = df.groupby('cohesion_label').agg({
        'Facebook Id': 'count',
        'Total Interactions': ['sum', 'mean', 'median'],
        'Likes': 'mean',
        'Comments': 'mean',
        'Shares': 'mean',
        'sentiment_score': 'mean',
        'polarization_score': 'mean',
        'is_viral': 'sum'
    }).round(2)

    cohesion_stats.columns = ['Posts_N', 'Total_Int', 'Avg_Int', 'Median_Int',
                              'Avg_Likes', 'Avg_Comments', 'Avg_Shares',
                              'Avg_Sentiment', 'Avg_Polarization', 'Viral_Posts']

    cohesion_stats['Pct_Corpus'] = (cohesion_stats['Posts_N'] / len(df) * 100).round(2)
    cohesion_stats['Pct_Viral'] = (cohesion_stats['Viral_Posts'] /
                                   cohesion_stats['Posts_N'] * 100).round(2)

    # Order by cohesion level
    level_order = ['High', 'Medium', 'Low']
    cohesion_stats = cohesion_stats.reindex([l for l in level_order if l in cohesion_stats.index])

    print("\nCohesion Label Distribution:")
    print(cohesion_stats)

    # Statistical test
    cohesion_groups = df['cohesion_label'].unique()
    engagement_by_cohesion = [df[df['cohesion_label'] == c]['Total Interactions'].dropna()
                              for c in cohesion_groups]
    h_stat, p_value = kruskal(*engagement_by_cohesion)

    print(f"\n📊 Kruskal-Wallis Test (Engagement across cohesion labels):")
    print(f"   H-statistic: {h_stat:.4f}")
    print(f"   P-value: {p_value:.4e}")

    # Save
    cohesion_stats.to_csv('Cohesion_Label_Analysis.csv', encoding='utf-8-sig')
    print("\n✅ Saved: Cohesion_Label_Analysis.csv")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Plot 1: Distribution
    ax1 = axes[0, 0]
    colors_coh = ['green', 'orange', 'red']
    ax1.bar(range(len(cohesion_stats)), cohesion_stats['Posts_N'],
           color=colors_coh[:len(cohesion_stats)], alpha=0.7)
    ax1.set_xticks(range(len(cohesion_stats)))
    ax1.set_xticklabels(cohesion_stats.index, fontsize=12)
    ax1.set_ylabel('Number of Posts', fontsize=13, fontweight='bold')
    ax1.set_title('Post Distribution by Cohesion Label', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)

    for i, (idx, row) in enumerate(cohesion_stats.iterrows()):
        ax1.text(i, row['Posts_N'], f"{row['Pct_Corpus']:.1f}%",
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    # Plot 2: Average engagement
    ax2 = axes[0, 1]
    ax2.bar(range(len(cohesion_stats)), cohesion_stats['Avg_Int'],
           color=colors_coh[:len(cohesion_stats)], alpha=0.7)
    ax2.set_xticks(range(len(cohesion_stats)))
    ax2.set_xticklabels(cohesion_stats.index, fontsize=12)
    ax2.set_ylabel('Average Interactions', fontsize=13, fontweight='bold')
    ax2.set_title('Average Engagement by Cohesion Label', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # Plot 3: Virality rate
    ax3 = axes[1, 0]
    ax3.bar(range(len(cohesion_stats)), cohesion_stats['Pct_Viral'],
           color=colors_coh[:len(cohesion_stats)], alpha=0.7)
    ax3.set_xticks(range(len(cohesion_stats)))
    ax3.set_xticklabels(cohesion_stats.index, fontsize=12)
    ax3.set_ylabel('Viral Posts (%)', fontsize=13, fontweight='bold')
    ax3.set_title('Virality Rate by Cohesion Label', fontsize=14, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)

    for i, (idx, row) in enumerate(cohesion_stats.iterrows()):
        ax3.text(i, row['Pct_Viral'], f"{row['Pct_Viral']:.1f}%",
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    # Plot 4: Sentiment & Polarization
    ax4 = axes[1, 1]
    x = np.arange(len(cohesion_stats))
    width = 0.35

    ax4.bar(x - width/2, cohesion_stats['Avg_Sentiment'], width,
           label='Sentiment', alpha=0.7, color='blue')
    ax4.bar(x + width/2, cohesion_stats['Avg_Polarization'], width,
           label='Polarization', alpha=0.7, color='red')

    ax4.set_xticks(x)
    ax4.set_xticklabels(cohesion_stats.index, fontsize=12)
    ax4.set_ylabel('Score', fontsize=13, fontweight='bold')
    ax4.set_title('Sentiment & Polarization by Cohesion Label', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.axhline(y=0.5, color='black', linestyle='--', alpha=0.3)
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('Cohesion_Label_Analysis.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: Cohesion_Label_Analysis.png")
    plt.close()

    return cohesion_stats


# =============================================================================
# POLARIZATION LABEL ANALYSIS
# =============================================================================

def analyze_polarization_labels(df):
    """
    Analyze posts by polarization label (High, Medium, Low)
    """

    print("\n" + "="*100)
    print("POLARIZATION LABEL ANALYSIS".center(100))
    print("="*100)

    if 'polarization_label' not in df.columns:
        print("⚠️  polarization_label column not found. Skipping...")
        return None

    # Polarization distribution
    polarization_stats = df.groupby('polarization_label').agg({
        'Facebook Id': 'count',
        'Total Interactions': ['sum', 'mean', 'median'],
        'Likes': 'mean',
        'Comments': 'mean',
        'Shares': 'mean',
        'sentiment_score': 'mean',
        'cohesion_score': 'mean',
        'is_viral': 'sum'
    }).round(2)

    polarization_stats.columns = ['Posts_N', 'Total_Int', 'Avg_Int', 'Median_Int',
                                  'Avg_Likes', 'Avg_Comments', 'Avg_Shares',
                                  'Avg_Sentiment', 'Avg_Cohesion', 'Viral_Posts']

    polarization_stats['Pct_Corpus'] = (polarization_stats['Posts_N'] / len(df) * 100).round(2)
    polarization_stats['Pct_Viral'] = (polarization_stats['Viral_Posts'] /
                                       polarization_stats['Posts_N'] * 100).round(2)

    # Order by polarization level
    level_order = ['High', 'Medium', 'Low']
    polarization_stats = polarization_stats.reindex([l for l in level_order if l in polarization_stats.index])

    print("\nPolarization Label Distribution:")
    print(polarization_stats)

    # Statistical test
    polarization_groups = df['polarization_label'].unique()
    engagement_by_polarization = [df[df['polarization_label'] == p]['Total Interactions'].dropna()
                                  for p in polarization_groups]
    h_stat, p_value = kruskal(*engagement_by_polarization)

    print(f"\n📊 Kruskal-Wallis Test (Engagement across polarization labels):")
    print(f"   H-statistic: {h_stat:.4f}")
    print(f"   P-value: {p_value:.4e}")

    # Save
    polarization_stats.to_csv('Polarization_Label_Analysis.csv', encoding='utf-8-sig')
    print("\n✅ Saved: Polarization_Label_Analysis.csv")

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Plot 1: Distribution
    ax1 = axes[0, 0]
    colors_pol = ['darkred', 'orange', 'lightcoral']
    ax1.bar(range(len(polarization_stats)), polarization_stats['Posts_N'],
           color=colors_pol[:len(polarization_stats)], alpha=0.7)
    ax1.set_xticks(range(len(polarization_stats)))
    ax1.set_xticklabels(polarization_stats.index, fontsize=12)
    ax1.set_ylabel('Number of Posts', fontsize=13, fontweight='bold')
    ax1.set_title('Post Distribution by Polarization Label', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)

    for i, (idx, row) in enumerate(polarization_stats.iterrows()):
        ax1.text(i, row['Posts_N'], f"{row['Pct_Corpus']:.1f}%",
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    # Plot 2: Average engagement
    ax2 = axes[0, 1]
    ax2.bar(range(len(polarization_stats)), polarization_stats['Avg_Int'],
           color=colors_pol[:len(polarization_stats)], alpha=0.7)
    ax2.set_xticks(range(len(polarization_stats)))
    ax2.set_xticklabels(polarization_stats.index, fontsize=12)
    ax2.set_ylabel('Average Interactions', fontsize=13, fontweight='bold')
    ax2.set_title('Average Engagement by Polarization Label', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # Plot 3: Virality rate
    ax3 = axes[1, 0]
    ax3.bar(range(len(polarization_stats)), polarization_stats['Pct_Viral'],
           color=colors_pol[:len(polarization_stats)], alpha=0.7)
    ax3.set_xticks(range(len(polarization_stats)))
    ax3.set_xticklabels(polarization_stats.index, fontsize=12)
    ax3.set_ylabel('Viral Posts (%)', fontsize=13, fontweight='bold')
    ax3.set_title('Virality Rate by Polarization Label', fontsize=14, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)

    for i, (idx, row) in enumerate(polarization_stats.iterrows()):
        ax3.text(i, row['Pct_Viral'], f"{row['Pct_Viral']:.1f}%",
                ha='center', va='bottom', fontsize=11, fontweight='bold')

    # Plot 4: Sentiment & Cohesion
    ax4 = axes[1, 1]
    x = np.arange(len(polarization_stats))
    width = 0.35

    ax4.bar(x - width/2, polarization_stats['Avg_Sentiment'], width,
           label='Sentiment', alpha=0.7, color='blue')
    ax4.bar(x + width/2, polarization_stats['Avg_Cohesion'], width,
           label='Cohesion', alpha=0.7, color='green')

    ax4.set_xticks(x)
    ax4.set_xticklabels(polarization_stats.index, fontsize=12)
    ax4.set_ylabel('Score', fontsize=13, fontweight='bold')
    ax4.set_title('Sentiment & Cohesion by Polarization Label', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.axhline(y=0.5, color='black', linestyle='--', alpha=0.3)
    ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('Polarization_Label_Analysis.png', dpi=300, bbox_inches='tight')
    print("✅ Saved: Polarization_Label_Analysis.png")
    plt.close()

    return polarization_stats


# =============================================================================
# CROSS-TABULATION ANALYSIS
# =============================================================================

def analyze_label_crosstabs(df):
    """
    Analyze relationships between different label categories
    """

    print("\n" + "="*100)
    print("CROSS-TABULATION ANALYSIS".center(100))
    print("="*100)

    # Sentiment × Cohesion
    if 'sentiment_label' in df.columns and 'cohesion_label' in df.columns:
        crosstab_sent_coh = pd.crosstab(
            df['sentiment_label'],
            df['cohesion_label'],
            normalize='index'
        ) * 100

        print("\nSentiment × Cohesion (%)")
        print(crosstab_sent_coh.round(2))
        crosstab_sent_coh.to_csv('Crosstab_Sentiment_Cohesion.csv', encoding='utf-8-sig')

    # Sentiment × Polarization
    if 'sentiment_label' in df.columns and 'polarization_label' in df.columns:
        crosstab_sent_pol = pd.crosstab(
            df['sentiment_label'],
            df['polarization_label'],
            normalize='index'
        ) * 100

        print("\nSentiment × Polarization (%)")
        print(crosstab_sent_pol.round(2))
        crosstab_sent_pol.to_csv('Crosstab_Sentiment_Polarization.csv', encoding='utf-8-sig')

    # Cohesion × Polarization
    if 'cohesion_label' in df.columns and 'polarization_label' in df.columns:
        crosstab_coh_pol = pd.crosstab(
            df['cohesion_label'],
            df['polarization_label'],
            normalize='index'
        ) * 100

        print("\nCohesion × Polarization (%)")
        print(crosstab_coh_pol.round(2))
        crosstab_coh_pol.to_csv('Crosstab_Cohesion_Polarization.csv', encoding='utf-8-sig')

    print("\n✅ Saved crosstab CSV files")


# =============================================================================
# UPDATED MAIN EXECUTION
# =============================================================================

def run_complete_analysis(df):
    """
    Run complete analysis including overall and label-based analyses
    """

    print("\n" + "="*100)
    print("ICWSM PAPER - COMPREHENSIVE ANALYSIS (ENHANCED)".center(100))
    print("="*100)

    # Prepare data
    df = prepare_analysis_data(df)

    # Overall statistics
    overall_df, categorical_df = analyze_overall_statistics(df)

    # Label-based analyses
    sentiment_stats = analyze_sentiment_labels(df)
    cohesion_stats = analyze_cohesion_labels(df)
    polarization_stats = analyze_polarization_labels(df)

    # Cross-tabulations
    analyze_label_crosstabs(df)

    # Run all RQ analyses (from previous code)
    topic_stats = analyze_rq1_topic_distribution(df)
    emotional_stats = analyze_rq2_emotional_polarization(df)
    corr_df, viral_comparison = analyze_rq3_engagement_virality(df)
    country_stats = analyze_rq4_geographic_patterns(df)
    content_stats = analyze_rq5_content_type(df)

    # Generate summary report
    generate_summary_report(df, topic_stats, emotional_stats, corr_df,
                           country_stats, content_stats)

    print("\n" + "="*100)
    print("ANALYSIS COMPLETE!".center(100))
    print("="*100)

    print("\n📄 Generated Files:")
    print("   Overall Statistics:")
    print("   • Overall_Descriptive_Statistics.csv")
    print("   • Overall_Categorical_Summary.csv")
    print("\n   Label-Based Analyses:")
    print("   • Sentiment_Label_Analysis.csv & .png")
    print("   • Cohesion_Label_Analysis.csv & .png")
    print("   • Polarization_Label_Analysis.csv & .png")
    print("   • Crosstab_*.csv (3 files)")
    print("\n   Research Questions:")
    print("   • RQ1_topic_distribution.csv & .png")
    print("   • RQ2_emotional_polarization.csv & .png")
    print("   • RQ3_correlation_matrix.csv & RQ3_engagement_virality.png")
    print("   • RQ4_geographic_patterns.csv & .png")
    print("   • RQ5_content_type.csv & .png")
    print("   • ICWSM_Analysis_Summary_Report.txt")

    return {
        'overall_stats': overall_df,
        'categorical_stats': categorical_df,
        'sentiment_stats': sentiment_stats,
        'cohesion_stats': cohesion_stats,
        'polarization_stats': polarization_stats,
        'topic_stats': topic_stats,
        'emotional_stats': emotional_stats,
        'correlation': corr_df,
        'viral_comparison': viral_comparison,
        'country_stats': country_stats,
        'content_stats': content_stats
    }


# =============================================================================
# USAGE
# =============================================================================

if __name__ == "__main__":

    # Load your dataframe
    # df = pd.read_csv('df_processed_with_topics.csv')

    # Run complete analysis
    results = run_complete_analysis(df)

    pass

In [ ]:
"""
Cohesion vs Polarization Heatmap Analysis
Multiple visualization styles for ICWSM paper
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# Set publication-quality style
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12


# =============================================================================
# COHESION VS POLARIZATION HEATMAP
# =============================================================================

def create_cohesion_polarization_heatmap(df, bin_size=0.1,
                                         save_path='Cohesion_Polarization_Heatmap.png'):
    """
    Create heatmap showing distribution of posts across cohesion-polarization space

    Parameters:
    -----------
    df : pandas.DataFrame
        Your dataframe with cohesion_score and polarization_score
    bin_size : float
        Size of bins for discretization (default: 0.1)
    save_path : str
        Path to save the figure
    """

    print("\n" + "="*100)
    print("COHESION VS POLARIZATION HEATMAP".center(100))
    print("="*100)

    # Prepare data
    df_clean = df[['cohesion_score', 'polarization_score', 'Total Interactions']].dropna()

    print(f"\n✅ Analyzing {len(df_clean):,} posts with valid cohesion and polarization scores")

    # Create bins
    cohesion_bins = np.arange(0, 1 + bin_size, bin_size)
    polarization_bins = np.arange(0, 1 + bin_size, bin_size)

    # Discretize scores
    df_clean['cohesion_bin'] = pd.cut(df_clean['cohesion_score'],
                                      bins=cohesion_bins,
                                      labels=cohesion_bins[:-1].round(2),
                                      include_lowest=True)
    df_clean['polarization_bin'] = pd.cut(df_clean['polarization_score'],
                                          bins=polarization_bins,
                                          labels=polarization_bins[:-1].round(2),
                                          include_lowest=True)

    # Create pivot table (count of posts)
    heatmap_counts = df_clean.groupby(['polarization_bin', 'cohesion_bin']).size().unstack(fill_value=0)

    # Create pivot table (average engagement)
    heatmap_engagement = df_clean.groupby(['polarization_bin', 'cohesion_bin'])['Total Interactions'].mean().unstack(fill_value=0)

    print(f"\n✅ Created heatmap with {len(cohesion_bins)-1} × {len(polarization_bins)-1} bins")

    # Create figure with multiple panels
    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

    # =========================================================================
    # HEATMAP 1: Post Count Density
    # =========================================================================
    ax1 = fig.add_subplot(gs[0, 0])

    # Log transform for better visualization
    heatmap_counts_log = np.log10(heatmap_counts + 1)

    im1 = ax1.imshow(heatmap_counts_log.values,
                     cmap='YlOrRd',
                     aspect='auto',
                     origin='lower',
                     interpolation='nearest')

    # Set ticks
    ax1.set_xticks(np.arange(len(heatmap_counts.columns)))
    ax1.set_yticks(np.arange(len(heatmap_counts.index)))
    ax1.set_xticklabels([f'{float(x):.1f}' for x in heatmap_counts.columns],
                        rotation=45, ha='right', fontsize=10)
    ax1.set_yticklabels([f'{float(x):.1f}' for x in heatmap_counts.index],
                        fontsize=10)

    # Labels
    ax1.set_xlabel('Cohesion Score', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Polarization Score', fontsize=14, fontweight='bold')
    ax1.set_title('(A) Post Density Distribution\n(Log10 Scale)',
                  fontsize=16, fontweight='bold', pad=15)

    # Colorbar
    cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    cbar1.set_label('Log10(Count + 1)', rotation=270, labelpad=20, fontsize=12)

    # Add quadrant lines
    mid_x = len(heatmap_counts.columns) / 2
    mid_y = len(heatmap_counts.index) / 2
    ax1.axvline(x=mid_x, color='white', linestyle='--', linewidth=2, alpha=0.7)
    ax1.axhline(y=mid_y, color='white', linestyle='--', linewidth=2, alpha=0.7)

    # Add quadrant labels
    ax1.text(mid_x * 0.5, mid_y * 1.5, 'Low Coh.\nHigh Pol.',
            ha='center', va='center', fontsize=11, color='white',
            weight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    ax1.text(mid_x * 1.5, mid_y * 1.5, 'High Coh.\nHigh Pol.',
            ha='center', va='center', fontsize=11, color='white',
            weight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    ax1.text(mid_x * 0.5, mid_y * 0.5, 'Low Coh.\nLow Pol.',
            ha='center', va='center', fontsize=11, color='white',
            weight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    ax1.text(mid_x * 1.5, mid_y * 0.5, 'High Coh.\nLow Pol.',
            ha='center', va='center', fontsize=11, color='white',
            weight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))

    # =========================================================================
    # HEATMAP 2: Average Engagement
    # =========================================================================
    ax2 = fig.add_subplot(gs[0, 1])

    # Log transform engagement for better visualization
    heatmap_engagement_log = np.log10(heatmap_engagement + 1)

    im2 = ax2.imshow(heatmap_engagement_log.values,
                     cmap='viridis',
                     aspect='auto',
                     origin='lower',
                     interpolation='nearest')

    ax2.set_xticks(np.arange(len(heatmap_engagement.columns)))
    ax2.set_yticks(np.arange(len(heatmap_engagement.index)))
    ax2.set_xticklabels([f'{float(x):.1f}' for x in heatmap_engagement.columns],
                        rotation=45, ha='right', fontsize=10)
    ax2.set_yticklabels([f'{float(x):.1f}' for x in heatmap_engagement.index],
                        fontsize=10)

    ax2.set_xlabel('Cohesion Score', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Polarization Score', fontsize=14, fontweight='bold')
    ax2.set_title('(B) Average Engagement\n(Log10 Scale)',
                  fontsize=16, fontweight='bold', pad=15)

    cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    cbar2.set_label('Log10(Avg Interactions + 1)', rotation=270, labelpad=20, fontsize=12)

    # Quadrant lines
    ax2.axvline(x=mid_x, color='white', linestyle='--', linewidth=2, alpha=0.7)
    ax2.axhline(y=mid_y, color='white', linestyle='--', linewidth=2, alpha=0.7)

    # =========================================================================
    # HEATMAP 3: Percentage Distribution
    # =========================================================================
    ax3 = fig.add_subplot(gs[1, 0])

    # Calculate percentage
    heatmap_pct = (heatmap_counts / heatmap_counts.sum().sum() * 100)

    im3 = ax3.imshow(heatmap_pct.values,
                     cmap='RdYlGn_r',
                     aspect='auto',
                     origin='lower',
                     interpolation='nearest',
                     vmin=0,
                     vmax=heatmap_pct.values.max())

    ax3.set_xticks(np.arange(len(heatmap_pct.columns)))
    ax3.set_yticks(np.arange(len(heatmap_pct.index)))
    ax3.set_xticklabels([f'{float(x):.1f}' for x in heatmap_pct.columns],
                        rotation=45, ha='right', fontsize=10)
    ax3.set_yticklabels([f'{float(x):.1f}' for x in heatmap_pct.index],
                        fontsize=10)

    ax3.set_xlabel('Cohesion Score', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Polarization Score', fontsize=14, fontweight='bold')
    ax3.set_title('(C) Percentage Distribution',
                  fontsize=16, fontweight='bold', pad=15)

    cbar3 = plt.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
    cbar3.set_label('Percentage (%)', rotation=270, labelpad=20, fontsize=12)

    # Add percentage values in cells (for values > 0.5%)
    for i in range(len(heatmap_pct.index)):
        for j in range(len(heatmap_pct.columns)):
            val = heatmap_pct.values[i, j]
            if val > 0.5:  # Only show if > 0.5%
                text_color = 'white' if val > heatmap_pct.values.max() * 0.6 else 'black'
                ax3.text(j, i, f'{val:.1f}',
                        ha='center', va='center',
                        color=text_color, fontsize=8, weight='bold')

    # Quadrant lines
    ax3.axvline(x=mid_x, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax3.axhline(y=mid_y, color='blue', linestyle='--', linewidth=2, alpha=0.7)

    # =========================================================================
    # HEATMAP 4: Contour Plot
    # =========================================================================
    ax4 = fig.add_subplot(gs[1, 1])

    # Create meshgrid
    X, Y = np.meshgrid(range(len(heatmap_counts.columns)),
                       range(len(heatmap_counts.index)))
    Z = heatmap_counts_log.values

    # Contour plot
    contourf = ax4.contourf(X, Y, Z, levels=15, cmap='YlOrRd', alpha=0.8)
    contour = ax4.contour(X, Y, Z, levels=8, colors='black',
                         linewidths=0.5, alpha=0.4)

    ax4.clabel(contour, inline=True, fontsize=8, fmt='%.1f')

    ax4.set_xticks(np.arange(len(heatmap_counts.columns)))
    ax4.set_yticks(np.arange(len(heatmap_counts.index)))
    ax4.set_xticklabels([f'{float(x):.1f}' for x in heatmap_counts.columns],
                        rotation=45, ha='right', fontsize=10)
    ax4.set_yticklabels([f'{float(x):.1f}' for x in heatmap_counts.index],
                        fontsize=10)

    ax4.set_xlabel('Cohesion Score', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Polarization Score', fontsize=14, fontweight='bold')
    ax4.set_title('(D) Density Contour Plot',
                  fontsize=16, fontweight='bold', pad=15)

    cbar4 = plt.colorbar(contourf, ax=ax4, fraction=0.046, pad=0.04)
    cbar4.set_label('Log10(Count + 1)', rotation=270, labelpad=20, fontsize=12)

    # Quadrant lines
    ax4.axvline(x=mid_x, color='blue', linestyle='--', linewidth=2, alpha=0.7)
    ax4.axhline(y=mid_y, color='blue', linestyle='--', linewidth=2, alpha=0.7)

    # Save figure
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\n✅ Saved: {save_path}")
    plt.close()

    # Save data tables
    heatmap_counts.to_csv('Heatmap_Post_Counts.csv', encoding='utf-8-sig')
    heatmap_engagement.to_csv('Heatmap_Avg_Engagement.csv', encoding='utf-8-sig')
    heatmap_pct.to_csv('Heatmap_Percentage_Distribution.csv', encoding='utf-8-sig')

    print("✅ Saved: Heatmap_Post_Counts.csv")
    print("✅ Saved: Heatmap_Avg_Engagement.csv")
    print("✅ Saved: Heatmap_Percentage_Distribution.csv")

    # Print summary statistics
    print("\n" + "="*100)
    print("HEATMAP SUMMARY STATISTICS".center(100))
    print("="*100)

    # Quadrant analysis
    low_coh_high_pol = df_clean[
        (df_clean['cohesion_score'] < 0.5) & (df_clean['polarization_score'] >= 0.5)
    ]
    high_coh_high_pol = df_clean[
        (df_clean['cohesion_score'] >= 0.5) & (df_clean['polarization_score'] >= 0.5)
    ]
    low_coh_low_pol = df_clean[
        (df_clean['cohesion_score'] < 0.5) & (df_clean['polarization_score'] < 0.5)
    ]
    high_coh_low_pol = df_clean[
        (df_clean['cohesion_score'] >= 0.5) & (df_clean['polarization_score'] < 0.5)
    ]

    print(f"\nQuadrant Distribution:")
    print(f"  Low Cohesion / High Polarization:  {len(low_coh_high_pol):>8,} ({len(low_coh_high_pol)/len(df_clean)*100:>5.2f}%)")
    print(f"  High Cohesion / High Polarization: {len(high_coh_high_pol):>8,} ({len(high_coh_high_pol)/len(df_clean)*100:>5.2f}%)")
    print(f"  Low Cohesion / Low Polarization:   {len(low_coh_low_pol):>8,} ({len(low_coh_low_pol)/len(df_clean)*100:>5.2f}%)")
    print(f"  High Cohesion / Low Polarization:  {len(high_coh_low_pol):>8,} ({len(high_coh_low_pol)/len(df_clean)*100:>5.2f}%)")

    print(f"\nAverage Engagement by Quadrant:")
    print(f"  Low Cohesion / High Polarization:  {low_coh_high_pol['Total Interactions'].mean():>8,.1f}")
    print(f"  High Cohesion / High Polarization: {high_coh_high_pol['Total Interactions'].mean():>8,.1f}")
    print(f"  Low Cohesion / Low Polarization:   {low_coh_low_pol['Total Interactions'].mean():>8,.1f}")
    print(f"  High Cohesion / Low Polarization:  {high_coh_low_pol['Total Interactions'].mean():>8,.1f}")

    print("\n" + "="*100)

    return {
        'counts': heatmap_counts,
        'engagement': heatmap_engagement,
        'percentage': heatmap_pct
    }


# =============================================================================
# SIMPLIFIED SINGLE HEATMAP (FOR PAPER)
# =============================================================================

def create_simple_cohesion_polarization_heatmap(df,
                                                save_path='Cohesion_Polarization_Simple.png'):
    """
    Create a single, clean heatmap for publication (most important one)
    """

    print("\n" + "="*100)
    print("SIMPLE COHESION VS POLARIZATION HEATMAP (PUBLICATION)".center(100))
    print("="*100)

    # Prepare data
    df_clean = df[['cohesion_score', 'polarization_score']].dropna()

    # Create bins (0.1 intervals)
    bins = np.arange(0, 1.1, 0.1)

    df_clean['cohesion_bin'] = pd.cut(df_clean['cohesion_score'],
                                      bins=bins,
                                      labels=bins[:-1].round(1),
                                      include_lowest=True)
    df_clean['polarization_bin'] = pd.cut(df_clean['polarization_score'],
                                          bins=bins,
                                          labels=bins[:-1].round(1),
                                          include_lowest=True)

    # Create pivot table
    heatmap_data = df_clean.groupby(['polarization_bin', 'cohesion_bin']).size().unstack(fill_value=0)

    # Convert to percentage
    heatmap_pct = (heatmap_data / heatmap_data.sum().sum() * 100)

    # Create figure
    fig, ax = plt.subplots(figsize=(14, 12))

    # Create heatmap
    sns.heatmap(heatmap_pct,
                annot=True,
                fmt='.1f',
                cmap='YlOrRd',
                cbar_kws={'label': 'Percentage of Posts (%)'},
                linewidths=0.5,
                linecolor='gray',
                square=True,
                ax=ax)

    # Labels
    ax.set_xlabel('Cohesion Score', fontsize=16, fontweight='bold', labelpad=12)
    ax.set_ylabel('Polarization Score', fontsize=16, fontweight='bold', labelpad=12)
    ax.set_title('Distribution of Posts: Cohesion vs Polarization',
                 fontsize=18, fontweight='bold', pad=20)

    # Add quadrant lines
    ax.axvline(x=5, color='blue', linestyle='--', linewidth=2.5, alpha=0.7)
    ax.axhline(y=5, color='blue', linestyle='--', linewidth=2.5, alpha=0.7)

    # Add quadrant labels
    ax.text(2.5, 7.5, 'Low Cohesion\nHigh Polarization',
            ha='center', va='center', fontsize=13, weight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.3))
    ax.text(7.5, 7.5, 'High Cohesion\nHigh Polarization',
            ha='center', va='center', fontsize=13, weight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.3))
    ax.text(2.5, 2.5, 'Low Cohesion\nLow Polarization',
            ha='center', va='center', fontsize=13, weight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightcoral', alpha=0.3))
    ax.text(7.5, 2.5, 'High Cohesion\nLow Polarization',
            ha='center', va='center', fontsize=13, weight='bold',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.3))

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\n✅ Saved: {save_path}")
    plt.close()

    return heatmap_pct


# =============================================================================
# USAGE
# =============================================================================

if __name__ == "__main__":

    # Load your dataframe
    # df = pd.read_csv('df_processed_with_topics.csv')

    # Create comprehensive 4-panel heatmap
    #results = create_cohesion_polarization_heatmap(df, bin_size=0.1)

    # Create simple publication-ready heatmap
    heatmap_pct = create_simple_cohesion_polarization_heatmap(df)

    pass

In [ ]:
"""
Cohesion vs Polarization Heatmap Analysis
Multiple visualization styles for ICWSM paper
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/COLAB/Wildlife_12_topics.csv')

In [ ]:
"""
Country Statistics with Page Count - CORRECTED VERSION
Full country names, Pages | N | Eng | Coh | Pol column order
"""

import pandas as pd
import numpy as np

def analyze_country_with_pages(df):
    """
    Analyze countries with page count included
    """

    print("\n" + "="*100)
    print("COUNTRY STATISTICS WITH PAGE COUNT".center(100))
    print("="*100)

    # Convert numeric columns
    numeric_cols = ['Total Interactions', 'cohesion_score', 'polarization_score']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Aggregate by country
    country_stats = df.groupby('Page Admin Top Country').agg({
        'Page Name': 'nunique',  # Unique page count
        'Facebook Id': 'count',  # Post count
        'Total Interactions': 'mean',
        'cohesion_score': 'mean',
        'polarization_score': 'mean'
    }).round(2)

    country_stats.columns = ['Pages_N', 'Posts_N', 'Avg_Int', 'Avg_Cohesion', 'Avg_Polarization']

    # Sort by post count
    country_stats = country_stats.sort_values('Posts_N', ascending=False)

    # Calculate percentage of corpus
    country_stats['Pct_Corpus'] = (country_stats['Posts_N'] / len(df) * 100).round(2)

    # Calculate posts per page
    country_stats['Posts_Per_Page'] = (country_stats['Posts_N'] / country_stats['Pages_N']).round(1)

    print(f"\n📊 Total posts in dataset: {len(df):,}")
    print(f"\n📊 Top 10 Countries with Page Information:")
    print(country_stats[['Pages_N', 'Posts_N', 'Pct_Corpus', 'Posts_Per_Page',
                        'Avg_Int', 'Avg_Cohesion', 'Avg_Polarization']].head(10).to_string())

    # Explanation for percentage differences
    print("\n" + "="*100)
    print("PERCENTAGE CALCULATION EXPLANATION".center(100))
    print("="*100)
    print("\nThe percentages might differ from your previous table because:")
    print("1. Different subset of data (e.g., filtered vs unfiltered)")
    print("2. Missing/null values in Page Admin Top Country column")
    print("3. Posts with UNASSIGNED topics removed")
    print("4. Data cleaning steps applied")
    print(f"\nCurrent calculation: Post count / Total posts ({len(df):,}) × 100")

    # Check for missing countries
    missing_country = df['Page Admin Top Country'].isna().sum()
    if missing_country > 0:
        print(f"\n⚠️  Warning: {missing_country:,} posts have missing country data ({missing_country/len(df)*100:.2f}%)")

    # Save to CSV
    country_stats.to_csv('Country_Statistics_With_Pages.csv', encoding='utf-8-sig')
    print(f"\n✅ Saved: Country_Statistics_With_Pages.csv")

    return country_stats


def generate_latex_table(country_stats, top_n=10):
    """
    Generate LaTeX table: Country | Pages | N | Eng | Coh | Pol
    Full country names, no abbreviations
    """

    print("\n" + "="*100)
    print("GENERATING LATEX TABLE".center(100))
    print("="*100)

    # Get top N countries
    top_countries = country_stats.head(top_n)

    # Start LaTeX table
    latex_code = r"""\begin{table}[t]
\caption{Top 10 countries by post volume: unique pages, number of posts (\textit{N}), average interactions per post (Eng.), average cohesion score (Coh.), and average polarization score (Pol.).}
\label{tab:country_stats}
\centering
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lrrrrr}
\hline
\textbf{Country} & \textbf{Pages} & \textbf{N} & \textbf{Eng.} & \textbf{Coh.} & \textbf{Pol.}  \\
\hline
"""

    # Add data rows
    for country, row in top_countries.iterrows():
        # Use full country name
        country_name = country

        # Format numbers
        pages = f"{int(row['Pages_N']):,}"
        posts = f"{int(row['Posts_N']):,}"
        engagement = f"{row['Avg_Int']:.2f}"
        cohesion = f"{row['Avg_Cohesion']:.2f}"
        polarization = f"{row['Avg_Polarization']:.2f}"

        # Add row
        latex_code += f"{country_name:<15} & {pages:>6} & {posts:>7} & {engagement:>6} & {cohesion:>4} & {polarization:>4} \\\\\n"

    # Close table
    latex_code += r"""\hline
\end{tabular}
\end{table}"""

    # Print LaTeX code
    print("\n" + "="*100)
    print("LATEX TABLE CODE:")
    print("="*100)
    print(latex_code)
    print("="*100)

    # Save to file
    with open('country_table_final.tex', 'w', encoding='utf-8') as f:
        f.write(latex_code)

    print("\n✅ Saved: country_table_final.tex")

    return latex_code


def check_percentage_differences(df, country_stats):
    """
    Debug function to understand percentage differences
    """

    print("\n" + "="*100)
    print("DEBUGGING PERCENTAGE DIFFERENCES".center(100))
    print("="*100)

    # Original total from your table
    original_total = 130869  # Sum of your original counts: 50700+10782+10532+...
    current_total = len(df)

    print(f"\nOriginal dataset total (from your table): {original_total:,}")
    print(f"Current dataset total: {current_total:,}")
    print(f"Difference: {current_total - original_total:,} posts")

    if current_total != original_total:
        print("\nPossible reasons for difference:")
        print("1. Additional data cleaning/filtering applied")
        print("2. Removed posts with missing values")
        print("3. Removed UNASSIGNED topics")
        print("4. Different time period")
        print("5. Duplicate removal")

    # Compare percentages
    print(f"\n{'Country':<15} {'Original %':<12} {'Current %':<12} {'Difference':<12}")
    print("-" * 55)

    original_percentages = {
        'Egypt': 38.74,
        'Iraq': 8.24,
        'Algeria': 8.05,
        'Syria': 7.37,
        'Libya': 6.33,
        'Morocco': 4.73,
        'Saudi Arabia': 4.52,
        'Jordan': 3.73,
        'Tunisia': 3.17,
        'Palestine': 3.01
    }

    for country, orig_pct in original_percentages.items():
        if country in country_stats.index:
            current_pct = country_stats.loc[country, 'Pct_Corpus']
            diff = current_pct - orig_pct
            print(f"{country:<15} {orig_pct:>10.2f}% {current_pct:>10.2f}% {diff:>+10.2f}%")
        else:
            print(f"{country:<15} {orig_pct:>10.2f}% {'N/A':>10} {'N/A':>10}")


def generate_footnote_version(country_stats, top_n=10):
    """
    Generate version with footnote explaining calculation
    """

    top_countries = country_stats.head(top_n)
    total_posts = country_stats['Posts_N'].sum()

    latex_code = r"""\begin{table}[t]
\caption{Top 10 countries by post volume: unique pages, number of posts (\textit{N}), average interactions per post (Eng.), average cohesion score (Coh.), and average polarization score (Pol.).\protect\footnotemark}
\label{tab:country_stats}
\centering
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lrrrrr}
\hline
\textbf{Country} & \textbf{Pages} & \textbf{N} & \textbf{Eng.} & \textbf{Coh.} & \textbf{Pol.}  \\
\hline
"""

    for country, row in top_countries.iterrows():
        pages = f"{int(row['Pages_N']):,}"
        posts = f"{int(row['Posts_N']):,}"
        engagement = f"{row['Avg_Int']:.2f}"
        cohesion = f"{row['Avg_Cohesion']:.2f}"
        polarization = f"{row['Avg_Polarization']:.2f}"

        latex_code += f"{country:<15} & {pages:>6} & {posts:>7} & {engagement:>6} & {cohesion:>4} & {polarization:>4} \\\\\n"

    latex_code += r"""\hline
\end{tabular}
\footnotetext{"""
    latex_code += f"Based on {total_posts:,} posts from the analyzed dataset."
    latex_code += r"""}
\end{table}"""

    print("\n" + "="*100)
    print("VERSION WITH FOOTNOTE:")
    print("="*100)
    print(latex_code)
    print("="*100)

    with open('country_table_with_footnote.tex', 'w', encoding='utf-8') as f:
        f.write(latex_code)

    print("\n✅ Saved: country_table_with_footnote.tex")

    return latex_code


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def create_country_table_final(df):
    """
    Complete workflow with debugging
    """

    print("\n" + "="*100)
    print("CREATE COUNTRY TABLE - FINAL VERSION".center(100))
    print("="*100)

    # Analyze
    country_stats = analyze_country_with_pages(df)

    # Debug percentages
    check_percentage_differences(df, country_stats)

    # Generate LaTeX tables
    latex_table = generate_latex_table(country_stats, top_n=10)

    print("\n" + "-"*100)

    latex_footnote = generate_footnote_version(country_stats, top_n=10)

    print("\n" + "="*100)
    print("✅ COMPLETE!".center(100))
    print("="*100)

    print("\n📄 Generated Files:")
    print("   • Country_Statistics_With_Pages.csv")
    print("   • country_table_final.tex")
    print("   • country_table_with_footnote.tex")

    print("\n💡 Column Order: Country | Pages | N | Eng. | Coh. | Pol.")
    print("💡 Full country names (no abbreviations)")
    print("💡 Check console output above for percentage difference explanation")

    return country_stats


# =============================================================================
# USAGE
# =============================================================================

if __name__ == "__main__":

    # Load your dataframe
    #df = pd.read_csv('df_processed_with_topics.csv')

    # Create final table
    country_stats = create_country_table_final(df)

In [ ]:
import pandas as pd

# Load your dataframe
#df = pd.read_csv('your_file.csv')

# Method 1: Count unique pages by Page Name
unique_pages = df['Page Name'].nunique()
print(f"Unique pages: {unique_pages:,}")

# Method 2: Get the list of unique page names
unique_page_list = df['Page Name'].unique()
print(f"Total unique pages: {len(unique_page_list):,}")

# Method 3: Count unique Facebook IDs (more accurate)
unique_facebook_ids = df['Facebook Id'].nunique()
print(f"Unique Facebook IDs: {unique_facebook_ids:,}")

# Method 4: Get detailed page info
page_info = df.groupby('Page Name')['Facebook Id'].agg(['count', 'first']).reset_index()
page_info.columns = ['Page_Name', 'Post_Count', 'Facebook_ID']
print(f"\nTotal unique pages: {len(page_info):,}")

# Save unique pages list
page_info.to_csv('unique_pages.csv', index=False, encoding='utf-8-sig')
print(f"✅ Saved unique pages list")

# Quick summary
print(f"\nSummary:")
print(f"Total posts: {len(df):,}")
print(f"Unique pages: {unique_pages:,}")
print(f"Avg posts per page: {len(df)/unique_pages:.1f}")

In [ ]:
df['sentiment_label'].value_counts()

In [ ]:
df.columns

In [ ]:
df['cohesion_label'].value_counts()

In [ ]:
df['polarization_label'].value_counts()

In [ ]:
def create_pivot_emotion_table(df):
    result = {}

    for col in ["sentiment_label", "cohesion_label", "polarization_label"]:
        counts = df[col].value_counts()
        perc = (counts / counts.sum() * 100).round(2)
        result[col.replace("_label", "").capitalize()] = (
            counts.astype(str) + " (" + perc.astype(str) + "%)"
        )

    table = pd.DataFrame(result).fillna("0 (0%)")
    return table

pivot_table = create_pivot_emotion_table(df)
pivot_table


In [ ]:
import pandas as pd

# Create summary table grouped by topic
summary_table = df.groupby('Assigned_Topic').agg({
    'Facebook Id': 'count',  # Number of posts
    'Total Interactions': 'mean',  # Average interactions only
    'cohesion_score': ['mean', 'median'],  # Cohesion statistics
    'polarization_score': ['mean', 'median']  # Polarization statistics
}).reset_index()

# Flatten column names
summary_table.columns = [
    'Topic',
    'Posts',
    'Avg_Int',
    'Cohesion_Mean',
    'Cohesion_Median',
    'Polar_Mean',
    'Polar_Median'
]

# Round numeric columns
summary_table['Avg_Int'] = summary_table['Avg_Int'].round(1)
summary_table['Cohesion_Mean'] = summary_table['Cohesion_Mean'].round(3)
summary_table['Cohesion_Median'] = summary_table['Cohesion_Median'].round(3)
summary_table['Polar_Mean'] = summary_table['Polar_Mean'].round(3)
summary_table['Polar_Median'] = summary_table['Polar_Median'].round(3)

# Sort by number of posts (descending)
summary_table = summary_table.sort_values('Posts', ascending=False)

# Display table
print(summary_table.to_string(index=False))

# Optional: Save to CSV
summary_table.to_csv('topic_summary.csv', index=False)

# Optional: Create LaTeX table for paper
print("\n\nLaTeX format:")
print(summary_table.to_latex(index=False, float_format="%.3f"))

In [ ]:
# Create styled table for better visualization
styled_table = summary_table.style\
    .format({
        'Posts': '{:,.0f}',
        'Avg_Int': '{:.1f}',
        'Cohesion_Mean': '{:.3f}',
        'Cohesion_Median': '{:.3f}',
        'Polar_Mean': '{:.3f}',
        'Polar_Median': '{:.3f}'
    })\
    .background_gradient(subset=['Cohesion_Mean'], cmap='RdYlGn')\
    .background_gradient(subset=['Polar_Mean'], cmap='RdYlGn_r')

styled_table

In [ ]:
import pandas as pd

# Create summary table grouped by topic
summary_table = df.groupby('Assigned_Topic').agg({
    'Facebook Id': 'count',  # Number of posts
    'Total Interactions': 'mean',  # Average interactions only
    'cohesion_score': ['mean', 'median'],  # Cohesion statistics
    'polarization_score': ['mean', 'median']  # Polarization statistics
}).reset_index()

# Flatten column names
summary_table.columns = [
    'Topic',
    'Posts',
    'Avg_Int',
    'Cohesion_Mean',
    'Cohesion_Median',
    'Polar_Mean',
    'Polar_Median'
]

# Round numeric columns
summary_table['Avg_Int'] = summary_table['Avg_Int'].round(1)
summary_table['Cohesion_Mean'] = summary_table['Cohesion_Mean'].round(3)
summary_table['Cohesion_Median'] = summary_table['Cohesion_Median'].round(3)
summary_table['Polar_Mean'] = summary_table['Polar_Mean'].round(3)
summary_table['Polar_Median'] = summary_table['Polar_Median'].round(3)

# Sort by number of posts (descending)
summary_table = summary_table.sort_values('Posts', ascending=False)

# Generate LaTeX with custom two-line headers
latex_code = r"""\begin{table}[ht]
\centering
\caption{Emotional Response Metrics by Topic}
\label{tab:topic_emotional_metrics}
\begin{tabular}{lrrrrrr}
\toprule
 & & \makecell{Avg\\Int.} & \multicolumn{2}{c}{Cohesion} & \multicolumn{2}{c}{Polarization} \\
\cmidrule(lr){4-5} \cmidrule(lr){6-7}
Topic & Posts &  & Mean & Median & Mean & Median \\
\midrule
"""

# Add data rows - without comma formatting
for _, row in summary_table.iterrows():
    latex_code += f"{row['Topic']} & {int(row['Posts'])} & {row['Avg_Int']:.1f} & {row['Cohesion_Mean']:.3f} & {row['Cohesion_Median']:.3f} & {row['Polar_Mean']:.3f} & {row['Polar_Median']:.3f} \\\\\n"

latex_code += r"""\bottomrule
\end{tabular}
\end{table}"""

print(latex_code)

# Save to file
with open('topic_table.tex', 'w') as f:
    f.write(latex_code)

In [ ]:
# Generate LaTeX with makecell package for better header formatting
latex_code = r"""\begin{table}[ht]
\centering
\caption{Emotional Response Metrics by Topic}
\label{tab:topic_emotional_metrics}
\begin{tabular}{lrrrrrr}
\toprule
 & & \makecell{Avg\\Int.} & \multicolumn{2}{c}{Cohesion} & \multicolumn{2}{c}{Polarization} \\
\cmidrule(lr){4-5} \cmidrule(lr){6-7}
Topic & Posts &  & Mean & Median & Mean & Median \\
\midrule
"""

# Add data rows
for _, row in summary_table.iterrows():
    latex_code += f"{row['Topic']} & {row['Posts']:,} & {row['Avg_Int']:.1f} & {row['Cohesion_Mean']:.3f} & {row['Cohesion_Median']:.3f} & {row['Polar_Mean']:.3f} & {row['Polar_Median']:.3f} \\\\\n"

latex_code += r"""\bottomrule
\end{tabular}
\end{table}"""

print(latex_code)

In [ ]:
"""
Temporal Stability Analysis for RQ3
Statistical approach instead of visualization
"""

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

def analyze_temporal_stability(df):
    """
    Test whether cohesion/polarization change significantly over time
    """

    print("\n" + "="*100)
    print("RQ3: TEMPORAL STABILITY ANALYSIS".center(100))
    print("="*100)

    # Ensure date column
    df['Post Created Date'] = pd.to_datetime(df['Post Created Date'])

    # Create time periods (weekly or bi-weekly)
    df['week'] = df['Post Created Date'].dt.isocalendar().week
    df['time_period'] = pd.cut(df['week'], bins=4, labels=['Early', 'Mid-Early', 'Mid-Late', 'Late'])

    # Overall temporal stability
    print("\n📊 OVERALL TEMPORAL STABILITY")
    print("-" * 100)

    weekly_cohesion = df.groupby('week')['cohesion_score'].mean()
    weekly_polarization = df.groupby('week')['polarization_score'].mean()

    # Calculate coefficient of variation (CV = std/mean)
    cv_cohesion = (weekly_cohesion.std() / weekly_cohesion.mean()) * 100
    cv_polarization = (weekly_polarization.std() / weekly_polarization.mean()) * 100

    print(f"\nCohesion across weeks:")
    print(f"  Mean: {weekly_cohesion.mean():.3f}")
    print(f"  Std: {weekly_cohesion.std():.3f}")
    print(f"  CV: {cv_cohesion:.2f}%")
    print(f"  Range: [{weekly_cohesion.min():.3f}, {weekly_cohesion.max():.3f}]")

    print(f"\nPolarization across weeks:")
    print(f"  Mean: {weekly_polarization.mean():.3f}")
    print(f"  Std: {weekly_polarization.std():.3f}")
    print(f"  CV: {cv_polarization:.2f}%")
    print(f"  Range: [{weekly_polarization.min():.3f}, {weekly_polarization.max():.3f}]")

    # Kruskal-Wallis test: Does cohesion/polarization differ across time periods?
    time_periods = df['time_period'].dropna().unique()
    cohesion_by_period = [df[df['time_period'] == p]['cohesion_score'].dropna() for p in time_periods]
    polarization_by_period = [df[df['time_period'] == p]['polarization_score'].dropna() for p in time_periods]

    h_coh, p_coh = kruskal(*cohesion_by_period)
    h_pol, p_pol = kruskal(*polarization_by_period)

    print(f"\n📊 Kruskal-Wallis Test (Time Period Effect):")
    print(f"  Cohesion: H={h_coh:.4f}, p={p_coh:.4e} {'***' if p_coh < 0.001 else '**' if p_coh < 0.01 else '*' if p_coh < 0.05 else 'ns'}")
    print(f"  Polarization: H={h_pol:.4f}, p={p_pol:.4e} {'***' if p_pol < 0.001 else '**' if p_pol < 0.01 else '*' if p_pol < 0.05 else 'ns'}")

    # Topic-level temporal stability
    print("\n" + "="*100)
    print("TOPIC-LEVEL TEMPORAL STABILITY".center(100))
    print("="*100)

    top_topics = df['Assigned_Topic'].value_counts().head(8).index

    topic_stability = []

    for topic in top_topics:
        topic_df = df[df['Assigned_Topic'] == topic]

        weekly_topic_coh = topic_df.groupby('week')['cohesion_score'].mean()
        weekly_topic_pol = topic_df.groupby('week')['polarization_score'].mean()

        cv_coh = (weekly_topic_coh.std() / weekly_topic_coh.mean()) * 100
        cv_pol = (weekly_topic_pol.std() / weekly_topic_pol.mean()) * 100

        # Trend test (Spearman correlation with week number)
        if len(weekly_topic_coh) > 2:
            trend_coh, p_trend_coh = stats.spearmanr(weekly_topic_coh.index, weekly_topic_coh.values)
            trend_pol, p_trend_pol = stats.spearmanr(weekly_topic_pol.index, weekly_topic_pol.values)
        else:
            trend_coh, p_trend_coh = np.nan, np.nan
            trend_pol, p_trend_pol = np.nan, np.nan

        topic_stability.append({
            'Topic': topic[:40] + '...' if len(topic) > 40 else topic,
            'Posts': len(topic_df),
            'Coh_Mean': weekly_topic_coh.mean(),
            'Coh_CV': cv_coh,
            'Coh_Trend_ρ': trend_coh,
            'Coh_Trend_p': p_trend_coh,
            'Pol_Mean': weekly_topic_pol.mean(),
            'Pol_CV': cv_pol,
            'Pol_Trend_ρ': trend_pol,
            'Pol_Trend_p': p_trend_pol
        })

    stability_df = pd.DataFrame(topic_stability)

    print("\nTop 8 Topics - Temporal Stability Metrics:")
    print(stability_df.to_string(index=False))

    # Save
    stability_df.to_csv('RQ3_Temporal_Stability.csv', index=False, encoding='utf-8-sig')
    print("\n✅ Saved: RQ3_Temporal_Stability.csv")

    return stability_df, weekly_cohesion, weekly_polarization


def analyze_geographic_cohesion(df):
    """
    Geographic variation in cohesion
    """

    print("\n" + "="*100)
    print("RQ3: GEOGRAPHIC COHESION PATTERNS".center(100))
    print("="*100)

    # Country-level cohesion
    country_cohesion = df.groupby('Page Admin Top Country').agg({
        'Facebook Id': 'count',
        'cohesion_score': ['mean', 'std'],
        'polarization_score': ['mean', 'std']
    }).round(3)

    country_cohesion.columns = ['Posts', 'Coh_Mean', 'Coh_Std', 'Pol_Mean', 'Pol_Std']
    country_cohesion = country_cohesion.sort_values('Posts', ascending=False).head(15)

    print("\nTop 15 Countries - Cohesion Patterns:")
    print(country_cohesion.to_string())

    # Statistical test: Do countries differ in cohesion?
    top_countries = country_cohesion.head(10).index
    cohesion_by_country = [df[df['Page Admin Top Country'] == c]['cohesion_score'].dropna()
                           for c in top_countries]

    h_stat, p_value = kruskal(*cohesion_by_country)

    print(f"\n📊 Kruskal-Wallis Test (Geographic Cohesion Differences):")
    print(f"  H={h_stat:.4f}, p={p_value:.4e} {'***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'}")

    # Regional groupings
    maghreb = ['Algeria', 'Morocco', 'Tunisia', 'Libya', 'Mauritania']
    levant = ['Syria', 'Jordan', 'Palestine', 'Lebanon']
    gulf = ['Saudi Arabia', 'UAE', 'Kuwait', 'Qatar', 'Bahrain', 'Oman']

    df['region'] = 'Other'
    df.loc[df['Page Admin Top Country'].isin(maghreb), 'region'] = 'Maghreb'
    df.loc[df['Page Admin Top Country'].isin(levant), 'region'] = 'Levant'
    df.loc[df['Page Admin Top Country'].isin(gulf), 'region'] = 'Gulf'
    df.loc[df['Page Admin Top Country'] == 'Egypt', 'region'] = 'Egypt'
    df.loc[df['Page Admin Top Country'] == 'Iraq', 'region'] = 'Iraq'

    regional_cohesion = df.groupby('region').agg({
        'Facebook Id': 'count',
        'cohesion_score': ['mean', 'std'],
        'polarization_score': ['mean', 'std']
    }).round(3)

    regional_cohesion.columns = ['Posts', 'Coh_Mean', 'Coh_Std', 'Pol_Mean', 'Pol_Std']
    regional_cohesion = regional_cohesion.sort_values('Posts', ascending=False)

    print("\n\nRegional Patterns:")
    print(regional_cohesion.to_string())

    # Save
    country_cohesion.to_csv('RQ3_Geographic_Cohesion.csv', encoding='utf-8-sig')
    regional_cohesion.to_csv('RQ3_Regional_Cohesion.csv', encoding='utf-8-sig')

    print("\n✅ Saved: RQ3_Geographic_Cohesion.csv")
    print("✅ Saved: RQ3_Regional_Cohesion.csv")

    return country_cohesion, regional_cohesion


def generate_rq3_summary_table(stability_df, country_cohesion):
    """
    Create publication-ready summary tables
    """

    print("\n" + "="*100)
    print("GENERATING RQ3 SUMMARY TABLES".center(100))
    print("="*100)

    # LaTeX table for temporal stability
    latex_temporal = r"""\begin{table}[t]
\caption{Temporal stability of cohesion and polarization across top topics. CV = coefficient of variation; $\rho$ = Spearman correlation with time.}
\label{tab:temporal_stability}
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\begin{tabular}{lrrrrr}
\hline
\textbf{Topic} & \textbf{Posts} & \textbf{Coh. Mean} & \textbf{Coh. CV} & \textbf{Pol. Mean} & \textbf{Pol. CV} \\
\hline
"""

    for _, row in stability_df.head(8).iterrows():
        topic = row['Topic'][:30] + '...' if len(row['Topic']) > 30 else row['Topic']
        latex_temporal += f"{topic:<35} & {int(row['Posts']):>6,} & {row['Coh_Mean']:.3f} & {row['Coh_CV']:.1f}\\% & {row['Pol_Mean']:.3f} & {row['Pol_CV']:.1f}\\% \\\\\n"

    latex_temporal += r"""\hline
\end{tabular}
\end{table}"""

    print("\nLATEX TABLE - TEMPORAL STABILITY:")
    print(latex_temporal)

    with open('rq3_temporal_table.tex', 'w', encoding='utf-8') as f:
        f.write(latex_temporal)

    # LaTeX table for geographic cohesion
    latex_geographic = r"""\begin{table}[t]
\caption{Geographic variation in emotional cohesion (top 10 countries).}
\label{tab:geographic_cohesion}
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lrrrr}
\hline
\textbf{Country} & \textbf{Posts} & \textbf{Coh. Mean} & \textbf{Coh. Std} & \textbf{Pol. Mean} \\
\hline
"""

    for country, row in country_cohesion.head(10).iterrows():
        latex_geographic += f"{country:<15} & {int(row['Posts']):>6,} & {row['Coh_Mean']:.3f} & {row['Coh_Std']:.3f} & {row['Pol_Mean']:.3f} \\\\\n"

    latex_geographic += r"""\hline
\end{tabular}
\end{table}"""

    print("\nLATEX TABLE - GEOGRAPHIC COHESION:")
    print(latex_geographic)

    with open('rq3_geographic_table.tex', 'w', encoding='utf-8') as f:
        f.write(latex_geographic)

    print("\n✅ Saved: rq3_temporal_table.tex")
    print("✅ Saved: rq3_geographic_table.tex")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def analyze_rq3_complete(df):
    """
    Complete RQ3 analysis
    """

    print("\n" + "="*100)
    print("RQ3 ANALYSIS - COMPLETE".center(100))
    print("="*100)

    # Temporal stability
    stability_df, weekly_coh, weekly_pol = analyze_temporal_stability(df)

    # Geographic patterns
    country_cohesion, regional_cohesion = analyze_geographic_cohesion(df)

    # Generate tables
    generate_rq3_summary_table(stability_df, country_cohesion)

    print("\n" + "="*100)
    print("✅ RQ3 ANALYSIS COMPLETE".center(100))
    print("="*100)

    return {
        'temporal_stability': stability_df,
        'country_cohesion': country_cohesion,
        'regional_cohesion': regional_cohesion,
        'weekly_cohesion': weekly_coh,
        'weekly_polarization': weekly_pol
    }


# Usage
if __name__ == "__main__":
    #df = pd.read_csv('df_processed_with_topics.csv')
    results = analyze_rq3_complete(df)

In [ ]:
"""
Print Top 10 Countries Per Topic in Compact Format
"""

def print_compact_topic_countries(results, top_n=10):
    """
    Print top N countries per topic in a clean, compact format
    """

    print("\n" + "="*100)
    print(f"TOP {top_n} COUNTRIES PER TOPIC - COMPACT VIEW".center(100))
    print("="*100)

    # Get the compact dataframe
    compact = results['compact']

    # Sort by total posts (descending)
    compact = compact.sort_values('Total_Posts', ascending=False)

    for idx, row in compact.iterrows():
        topic = row['Topic']
        countries = row['Top_5_Countries']  # This actually has top_n countries
        total = row['Total_Posts']

        # Shorten topic name if too long
        if len(topic) > 70:
            topic_display = topic[:67] + '...'
        else:
            topic_display = topic

        print(f"\n{topic_display}")
        print(f"{'─' * 100}")
        print(f"Total Posts: {total:,}")
        print(f"Countries: {countries}")

    print("\n" + "="*100)


def print_detailed_topic_countries(results, top_n=10):
    """
    Print top N countries with detailed statistics
    """

    print("\n" + "="*100)
    print(f"TOP {top_n} COUNTRIES PER TOPIC - DETAILED VIEW".center(100))
    print("="*100)

    detailed = results['detailed']

    # Get unique topics sorted by total posts
    topics_sorted = detailed.groupby('Topic')['Posts'].sum().sort_values(ascending=False).index

    for topic in topics_sorted:
        topic_data = detailed[detailed['Topic'] == topic].head(top_n)

        # Shorten topic name
        if len(topic) > 70:
            topic_display = topic[:67] + '...'
        else:
            topic_display = topic

        print(f"\n{'='*100}")
        print(f"{topic_display}")
        print(f"{'='*100}")

        total_posts = topic_data['Posts'].sum()
        print(f"Total Posts: {total_posts:,}\n")

        # Table header
        print(f"{'#':<4} {'Country':<25} {'Posts':<12} {'% Topic':<10} {'Avg Int':<10} {'Coh':<6} {'Pol':<6}")
        print("─" * 100)

        # Print each country
        for _, row in topic_data.iterrows():
            print(f"{row['Rank']:<4} {row['Country']:<25} {row['Posts']:<12,} "
                  f"{row['Pct_Topic']:<9.1f}% {row['Avg_Int']:<9.1f} "
                  f"{row['Avg_Cohesion']:<5.2f} {row['Avg_Polarization']:<5.2f}")

    print("\n" + "="*100)


def print_one_line_per_topic(results):
    """
    Print one line per topic with just country names
    """

    print("\n" + "="*100)
    print("ONE-LINE FORMAT: TOPIC → COUNTRIES".center(100))
    print("="*100)

    compact = results['compact'].sort_values('Total_Posts', ascending=False)

    for idx, row in compact.iterrows():
        topic = row['Topic']
        countries = row['Top_5_Countries']

        # Truncate topic if needed
        if len(topic) > 50:
            topic_short = topic[:47] + '...'
        else:
            topic_short = topic

        print(f"\n{topic_short:<50} → {countries}")

    print("\n" + "="*100)


def export_to_markdown(results, filename='topic_countries.md'):
    """
    Export to markdown format for easy reading
    """

    print(f"\n📝 Exporting to markdown: {filename}")

    compact = results['compact'].sort_values('Total_Posts', ascending=False)

    with open(filename, 'w', encoding='utf-8') as f:
        f.write("# Top 10 Countries Per Topic\n\n")
        f.write("Wildlife Conservation Discourse on Arabic Facebook\n\n")
        f.write("---\n\n")

        for idx, row in compact.iterrows():
            topic = row['Topic']
            countries = row['Top_5_Countries']
            total = row['Total_Posts']

            f.write(f"## {topic}\n\n")
            f.write(f"**Total Posts:** {total:,}\n\n")
            f.write(f"**Top Countries:** {countries}\n\n")
            f.write("---\n\n")

    print(f"✅ Saved: {filename}")


# =============================================================================
# USAGE
# =============================================================================

# After running your analysis:
# results = analyze_topic_countries(df, top_n=10)

# Option 1: Simple compact view
print_compact_topic_countries(results, top_n=10)

# Option 2: Detailed view with statistics
#print_detailed_topic_countries(results, top_n=10)

# Option 3: One-liner format
#print_one_line_per_topic(results)

# Option 4: Export to markdown file
#export_to_markdown(results, 'topic_countries.md')

In [ ]:
"""
Sensitivity Analysis for Reaction Weights and Sentiment Mapping
Address reviewer concerns about reaction semantics
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

def sensitivity_analysis_weights(df):
    """
    Test multiple weighting schemes to assess robustness
    """

    print("\n" + "="*100)
    print("SENSITIVITY ANALYSIS: ALTERNATIVE WEIGHTING SCHEMES".center(100))
    print("="*100)

    # Convert to numeric
    reactions = ['Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care', 'Likes']
    for col in reactions:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Define alternative weighting schemes
    weighting_schemes = {
        'Original': {
            'Love': 2, 'Wow': 1, 'Care': 0.5, 'Likes': 1,
            'Sad': -1, 'Angry': -2
        },
        'Equal': {
            'Love': 1, 'Wow': 1, 'Care': 1, 'Likes': 1,
            'Sad': -1, 'Angry': -1
        },
        'Conservative': {
            'Love': 1.5, 'Wow': 1, 'Care': 0.75, 'Likes': 1,
            'Sad': -1, 'Angry': -1.5
        },
        'Intensity': {
            'Love': 3, 'Wow': 1, 'Care': 1, 'Likes': 1,
            'Sad': -2, 'Angry': -3
        },
        'No_Weighting': {
            'Love': 1, 'Wow': 1, 'Care': 1, 'Likes': 1,
            'Sad': -1, 'Angry': -1
        }
    }

    results = []

    for scheme_name, weights in weighting_schemes.items():
        # Calculate sentiment with this scheme
        positive = (df['Love'] * weights['Love'] +
                   df['Wow'] * weights['Wow'] +
                   df['Care'] * weights['Care'] +
                   df['Likes'] * weights['Likes'])

        negative = (df['Sad'] * abs(weights['Sad']) +
                   df['Angry'] * abs(weights['Angry']))

        total_reactions = positive + negative
        sentiment = np.where(total_reactions > 0,
                            (positive - negative) / total_reactions, 0)

        results.append({
            'Scheme': scheme_name,
            'Mean_Sentiment': sentiment.mean(),
            'Std_Sentiment': sentiment.std(),
            'Median_Sentiment': np.median(sentiment),
            'Positive_Posts_Pct': (sentiment > 0).sum() / len(sentiment) * 100,
            'Negative_Posts_Pct': (sentiment < 0).sum() / len(sentiment) * 100,
            'Neutral_Posts_Pct': (sentiment == 0).sum() / len(sentiment) * 100
        })

    results_df = pd.DataFrame(results)

    print("\n📊 SENTIMENT SCORES ACROSS WEIGHTING SCHEMES:")
    print(results_df.to_string(index=False))

    # Calculate correlation between schemes
    print("\n📊 CORRELATION BETWEEN SCHEMES:")
    sentiment_by_scheme = {}
    for scheme_name, weights in weighting_schemes.items():
        positive = (df['Love'] * weights['Love'] +
                   df['Wow'] * weights['Wow'] +
                   df['Care'] * weights['Care'] +
                   df['Likes'] * weights['Likes'])
        negative = (df['Sad'] * abs(weights['Sad']) +
                   df['Angry'] * abs(weights['Angry']))
        total_reactions = positive + negative
        sentiment_by_scheme[scheme_name] = np.where(total_reactions > 0,
                                                     (positive - negative) / total_reactions, 0)

    corr_matrix = pd.DataFrame(sentiment_by_scheme).corr()
    print(corr_matrix.round(3))

    # Save results
    results_df.to_csv('Sensitivity_Weighting_Schemes.csv', index=False, encoding='utf-8-sig')
    print("\n✅ Saved: Sensitivity_Weighting_Schemes.csv")

    return results_df, corr_matrix


def test_haha_wow_assignment(df):
    """
    Test sensitivity to Haha/Wow assignment
    """

    print("\n" + "="*100)
    print("SENSITIVITY ANALYSIS: HAHA/WOW ASSIGNMENT".center(100))
    print("="*100)

    # Convert to numeric
    for col in ['Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care', 'Likes']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Test different Haha/Wow assignment strategies
    strategies = {
        'Original': 'majority_positive',
        'Always_Positive': 'always_positive',
        'Always_Negative': 'always_negative',
        'Exclude': 'exclude',
        'Neutral': 'neutral'
    }

    results = []

    for strategy_name, method in strategies.items():
        positive = df['Love'] * 2 + df['Care'] * 0.5 + df['Likes']
        negative = df['Sad'] + df['Angry'] * 2

        if method == 'majority_positive':
            # Original: assign to majority
            haha_wow = df['Haha'] + df['Wow']
            positive_side = positive > negative
            positive += np.where(positive_side, haha_wow, 0)
            negative += np.where(~positive_side, haha_wow, 0)
        elif method == 'always_positive':
            positive += df['Haha'] + df['Wow']
        elif method == 'always_negative':
            negative += df['Haha'] + df['Wow']
        elif method == 'exclude':
            pass  # Don't include Haha/Wow
        elif method == 'neutral':
            pass  # Treat as neutral

        total_reactions = positive + negative
        sentiment = np.where(total_reactions > 0,
                            (positive - negative) / total_reactions, 0)

        results.append({
            'Strategy': strategy_name,
            'Mean_Sentiment': sentiment.mean(),
            'Positive_Pct': (sentiment > 0).sum() / len(sentiment) * 100,
            'Posts_Affected': ((df['Haha'] > 0) | (df['Wow'] > 0)).sum(),
            'Pct_Affected': ((df['Haha'] > 0) | (df['Wow'] > 0)).sum() / len(df) * 100
        })

    results_df = pd.DataFrame(results)

    print("\n📊 SENTIMENT UNDER DIFFERENT HAHA/WOW STRATEGIES:")
    print(results_df.round(3).to_string(index=False))

    results_df.to_csv('Sensitivity_Haha_Wow_Assignment.csv', index=False, encoding='utf-8-sig')
    print("\n✅ Saved: Sensitivity_Haha_Wow_Assignment.csv")

    return results_df


def country_specific_baselines(df):
    """
    Analyze country-specific reaction patterns
    """

    print("\n" + "="*100)
    print("COUNTRY-SPECIFIC REACTION BASELINES".center(100))
    print("="*100)

    # Convert to numeric
    reactions = ['Love', 'Wow', 'Haha', 'Sad', 'Angry', 'Care', 'Likes']
    for col in reactions:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Calculate reaction proportions by country
    country_reactions = []

    top_countries = df['Page Admin Top Country'].value_counts().head(10).index

    for country in top_countries:
        country_df = df[df['Page Admin Top Country'] == country]

        total_reactions = country_df[reactions].sum().sum()

        country_reactions.append({
            'Country': country,
            'Posts': len(country_df),
            'Love_Pct': country_df['Love'].sum() / total_reactions * 100,
            'Likes_Pct': country_df['Likes'].sum() / total_reactions * 100,
            'Wow_Pct': country_df['Wow'].sum() / total_reactions * 100,
            'Haha_Pct': country_df['Haha'].sum() / total_reactions * 100,
            'Sad_Pct': country_df['Sad'].sum() / total_reactions * 100,
            'Angry_Pct': country_df['Angry'].sum() / total_reactions * 100,
            'Care_Pct': country_df['Care'].sum() / total_reactions * 100
        })

    country_df = pd.DataFrame(country_reactions)

    print("\n📊 REACTION DISTRIBUTION BY COUNTRY (% of total reactions):")
    print(country_df.round(2).to_string(index=False))

    # Test if distributions differ significantly
    print("\n📊 STATISTICAL TEST: Do countries differ in reaction usage?")
    print("(Chi-square test for each reaction type)")

    for reaction in reactions:
        # Create contingency table
        contingency = []
        for country in top_countries:
            country_data = df[df['Page Admin Top Country'] == country]
            reaction_count = country_data[reaction].sum()
            other_count = country_data[[r for r in reactions if r != reaction]].sum().sum()
            contingency.append([reaction_count, other_count])

        chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
        sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
        print(f"   {reaction:<10} χ²={chi2:>8.2f}, p={p_value:.4e} {sig}")

    country_df.to_csv('Country_Reaction_Baselines.csv', index=False, encoding='utf-8-sig')
    print("\n✅ Saved: Country_Reaction_Baselines.csv")

    return country_df


def visualize_sensitivity(results_df, corr_matrix):
    """
    Create visualization for sensitivity analysis
    """

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Sentiment scores by scheme
    ax1 = axes[0]
    x = np.arange(len(results_df))
    ax1.bar(x, results_df['Mean_Sentiment'], alpha=0.7, color='steelblue')
    ax1.errorbar(x, results_df['Mean_Sentiment'],
                yerr=results_df['Std_Sentiment'],
                fmt='none', color='black', capsize=5)
    ax1.set_xticks(x)
    ax1.set_xticklabels(results_df['Scheme'], rotation=45, ha='right')
    ax1.set_ylabel('Mean Sentiment Score', fontsize=12, fontweight='bold')
    ax1.set_title('(A) Sentiment Scores Across Weighting Schemes',
                 fontsize=13, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5)

    # Plot 2: Correlation heatmap
    ax2 = axes[1]
    sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
               center=0.95, vmin=0.9, vmax=1.0,
               cbar_kws={'label': 'Spearman Correlation'},
               ax=ax2)
    ax2.set_title('(B) Correlation Between Weighting Schemes',
                 fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.savefig('Sensitivity_Analysis.png', dpi=300, bbox_inches='tight')
    print("\n✅ Saved: Sensitivity_Analysis.png")
    plt.close()


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_complete_sensitivity_analysis(df):
    """
    Run all sensitivity analyses
    """

    print("\n" + "="*100)
    print("COMPREHENSIVE SENSITIVITY ANALYSIS".center(100))
    print("="*100)

    # 1. Weighting schemes
    results_df, corr_matrix = sensitivity_analysis_weights(df)

    # 2. Haha/Wow assignment
    haha_results = test_haha_wow_assignment(df)

    # 3. Country baselines
    country_baselines = country_specific_baselines(df)

    # 4. Visualization
    visualize_sensitivity(results_df, corr_matrix)

    print("\n" + "="*100)
    print("✅ SENSITIVITY ANALYSIS COMPLETE".center(100))
    print("="*100)

    print("\n📄 Key Findings:")
    print(f"   • Correlation between schemes: {corr_matrix.min().min():.3f} - {corr_matrix.max().max():.3f}")
    print(f"   • Sentiment range across schemes: {results_df['Mean_Sentiment'].min():.3f} - {results_df['Mean_Sentiment'].max():.3f}")
    print(f"   • Posts with Haha/Wow: {haha_results.loc[0, 'Posts_Affected']:,.0f} ({haha_results.loc[0, 'Pct_Affected']:.1f}%)")

    return {
        'weighting': results_df,
        'correlation': corr_matrix,
        'haha_wow': haha_results,
        'country_baselines': country_baselines
    }


# Usage
if __name__ == "__main__":
    #df = pd.read_csv('df_processed_with_topics.csv')
    results = run_complete_sensitivity_analysis(df)

In [ ]:
# =============================================================================
# Step 1: Read CSV from Google Drive
# =============================================================================

# Your Google Drive link
drive_link = "https://drive.google.com/file/d/1cOvHsO8KPmaGFsYO8zFZXfiNJrLHVxye/view?usp=drive_link"
#drive_link="https://drive.google.com/file/d/1SgmH-S5d3-s3SpOzC3-LzBUWX27Ydc6P/view?usp=drive_link"
# Extract file ID
file_id = drive_link.split("/d/")[1].split("/")[0]

# Download
print("Downloading...")
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, "temp.csv", quiet=False)

# Read CSV
print("\nReading CSV...")
df_llama = pd.read_csv("temp.csv", encoding='utf-8', low_memory=False)
print(f"✅ Loaded {len(df_llama):,} rows")

In [ ]:
print(df_llama[df_llama['LLM_Topic'] == 'INVALID_OUTPUT'][['URL', 'LLM_Topic']])

In [ ]:
df1=pd.read_csv('/content/drive/MyDrive/wildlife_annotation/df_sampled_2000.csv')

In [ ]:
# Get URLs where LLM_Topic is INVALID_OUTPUT
invalid_urls = df_llama[df_llama['LLM_Topic'] == 'INVALID_OUTPUT']['URL']

# Filter df1 for those URLs and extract the Message column
df1_invalid_messages = df1[df1['URL'].isin(invalid_urls)][['URL', 'Message']]

print(df1_invalid_messages)

In [ ]:
df_llm=pd.read_csv('/content/drive/MyDrive/wildlife_annotation/llm_ensemble_majority_vote.csv')

In [ ]:
df_bge.columns

In [ ]:
# Reassigning to the dataframe
df_bge = df_bge.rename(columns={'SBERT_Best_Topic': 'SBERT_bGE'})

In [ ]:
# Merge Human_Topic into df_llm based on 'URL'
df_llm_22 = df_llm_22.merge(df_bge[['URL','SBERT_bGE']], on='URL', how='left')

In [ ]:
df_llm_22.head(2)

In [ ]:
df_llm_22.columns

In [ ]:
# Select specified columns into df_2000_final
df_2000_final = df_llm_22[[
    'URL',
    'LLM_Topic_llama',
    'LLM_Topic_gpt',
    'LLM_Topic_gemini',
    'Human_Topic',
    'Qwen_topic',
    'SEBRT_miniML',
    'SBERT_e5Large',
    'SBERT_bGE'
]]

In [ ]:
df_2000_final.to_csv('/content/drive/MyDrive/wildlife_annotation/df_2000_final.csv')

In [ ]:
df_e5['SBERT_Best_Topic'].value_counts()

In [ ]:
df_llm['Majority_Topic'].value_counts()

In [ ]:
df_2000_final=pd.read_csv('/content/drive/MyDrive/wildlife_annotation/df_2000_final_all_valid.csv')

In [ ]:

import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    multilabel_confusion_matrix
)

# ============================================================
# 1. DEFINE LABELS AND MODEL COLUMNS
# ============================================================

TOPICS = [
    "Captivity, Zoos & Ethical Wildlife Management",
    "Climate Change Impact on Wildlife",
    "Habitat Loss, Deforestation & Ecosystem Degradation",
    "Wildlife Rehabilitation & Rescue",
    "Wildlife Awareness, Media & Activism",
    "Human-Wildlife Conflict",
    "Marine & Ocean Conservation",
    "Endangered Species & Extinction Risk",
    "Wildlife Crime, Poaching & Enforcement",
    "Conservation Policy, Law & Funding",
    "Indigenous & Community-Based Conservation",
    "National Parks & Protected Areas",
    "UNASSIGNED"
]

MODEL_COLUMNS = {
    "Llama": "LLM_Topic_llama",
    "GPT": "LLM_Topic_gpt",
    "Gemini": "LLM_Topic_gemini",
    "Qwen": "Qwen_topic",
    "MiniLM": "SEBRT_miniML",
    "E5-Large": "SBERT_e5Large",
    "BGE-M3": "SBERT_bGE"
}

# ============================================================
# 2. PREPARE DATA
# ============================================================

columns = (
    ["URL", "Human_Topic"] +
    list(MODEL_COLUMNS.values())
)

df_eval = df_2000_final[columns].copy()

# Keep only rows with human annotations
df_eval = df_eval[
    df_eval["Human_Topic"].notna()
].copy()

print(f"Evaluation posts: {len(df_eval):,}")


# ============================================================
# 3. CONVERT LABEL STRINGS TO LABEL SETS
# ============================================================

def parse_labels(value):
    """
    Handles:
      - Single topic
      - Multiple topics separated by ;
      - Lists or tuples of topic names
      - UNASSIGNED

    Does not silently convert missing predictions
    into UNASSIGNED.
    """

    if isinstance(value, (list, tuple, set)):
        labels = [str(x).strip() for x in value]

    elif pd.isna(value):
        return []

    else:
        text = str(value).strip()

        if not text:
            return []

        labels = [
            x.strip()
            for x in text.split(";")
        ]

    labels = [
        label for label in labels
        if label
    ]

    invalid = set(labels) - set(TOPICS)

    if invalid:
        raise ValueError(
            f"Unknown topic labels: {invalid}"
        )

    # UNASSIGNED must be mutually exclusive
    if "UNASSIGNED" in labels and len(labels) > 1:
        raise ValueError(
            "UNASSIGNED cannot appear with another topic."
        )

    return sorted(set(labels))


df_eval["Human_labels"] = (
    df_eval["Human_Topic"].apply(parse_labels)
)

if df_eval["Human_labels"].apply(len).eq(0).any():
    raise ValueError(
        "Some human annotations are missing or empty."
    )


# ============================================================
# 4. ONE-VS-REST EVALUATION
# ============================================================

mlb = MultiLabelBinarizer(classes=TOPICS)
mlb.fit([TOPICS])

y_true = mlb.transform(
    df_eval["Human_labels"]
)

results = []
per_topic_results = []

for model_name, column in MODEL_COLUMNS.items():

    pred_labels = df_eval[column].apply(parse_labels)

    # Report missing predictions rather than treating
    # them as UNASSIGNED.
    missing = pred_labels.apply(len).eq(0)

    if missing.any():
        raise ValueError(
            f"{model_name}: {missing.sum()} missing predictions. "
            "Inspect these before calculating metrics."
        )

    y_pred = mlb.transform(pred_labels)

    # Standard multi-label exact-match accuracy
    exact_accuracy = accuracy_score(
        y_true, y_pred
    )

    # Overlap accuracy:
    # At least one predicted topic matches a human topic.
    overlap_accuracy = np.mean(
        np.logical_and(y_true, y_pred).any(axis=1)
    )

    # Standard macro metrics across 13 labels
    macro_precision = precision_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    # One-vs-rest accuracy for each label
    # TN, FP, FN, TP
    cm = multilabel_confusion_matrix(
        y_true, y_pred
    )

    class_accuracy = []

    for i, matrix in enumerate(cm):

        tn, fp, fn, tp = matrix.ravel()
        total = tn + fp + fn + tp

        acc = (
            (tp + tn) / total
            if total else np.nan
        )

        class_accuracy.append(acc)

        per_topic_results.append({
            "Model": model_name,
            "Topic": TOPICS[i],
            "OvR_Accuracy": acc,
            "Precision": precision_score(
                y_true[:, i],
                y_pred[:, i],
                zero_division=0
            ),
            "Recall": recall_score(
                y_true[:, i],
                y_pred[:, i],
                zero_division=0
            ),
            "F1": f1_score(
                y_true[:, i],
                y_pred[:, i],
                zero_division=0
            )
        })

    macro_accuracy = np.mean(class_accuracy)

    results.append({
        "Model": model_name,
        "Exact_Accuracy": exact_accuracy,
        "Overlap_Accuracy": overlap_accuracy,
        "Macro_Accuracy_OvR": macro_accuracy,
        "Macro_Precision": macro_precision,
        "Macro_Recall": macro_recall,
        "Macro_F1": macro_f1
    })


# ============================================================
# 5. RESULTS
# ============================================================

results_df = pd.DataFrame(results)

per_topic_df = pd.DataFrame(per_topic_results)

results_df = results_df.round(4)
per_topic_df = per_topic_df.round(4)

print("\nMODEL COMPARISON")
print(results_df.to_string(index=False))

print("\nPER-TOPIC RESULTS")
print(per_topic_df.to_string(index=False))


# ============================================================
# 6. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "wildlife_model_macro_evaluation.csv",
    index=False
)

per_topic_df.to_csv(
    "wildlife_per_topic_evaluation.csv",
    index=False
)

print("\nEvaluation completed.")


In [ ]:
df_eval["Human_Topic"].value_counts()

In [ ]:
for model_name, col in MODEL_COLUMNS.items():
    invalid = (
        df_eval[col]
        .astype("string")
        .str.strip()
        .eq("INVALID_OUTPUT")
    )

    print(f"{model_name}: {invalid.sum()} INVALID_OUTPUT")

In [ ]:
for model_name, col in MODEL_COLUMNS.items():
    print(f"\n{model_name}")

    labels = df_eval[col].dropna().astype(str).str.strip()

    print(labels.value_counts().head(20))

In [ ]:

import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    multilabel_confusion_matrix
)

# Assumes TOPICS and MODEL_COLUMNS are already defined
# from your previous evaluation code.

df_eval = df_2000_final[
    ["URL", "Human_Topic"] + list(MODEL_COLUMNS.values())
].copy()

# Parse the human reference labels
df_eval["Human_labels"] = (
    df_eval["Human_Topic"].apply(parse_labels)
)

if df_eval["Human_labels"].apply(len).eq(0).any():
    raise ValueError("Missing human annotations found.")

# Keep only the common valid subset
valid_mask = pd.Series(True, index=df_eval.index)

for model_name, column in MODEL_COLUMNS.items():
    valid_mask &= df_eval[column].apply(
        lambda value: (
            isinstance(value, str)
            and value.strip() != "INVALID_OUTPUT"
            and value.strip() != ""
            and value.strip() in TOPICS
        )
    )

df_clean = df_eval.loc[valid_mask].copy()

print("Original posts:", len(df_eval))
print("Common valid posts:", len(df_clean))
print("Excluded posts:", len(df_eval) - len(df_clean))

# Multi-label binarization
mlb = MultiLabelBinarizer(classes=TOPICS)
mlb.fit([TOPICS])

y_true = mlb.transform(df_clean["Human_labels"])

results = []

for model_name, column in MODEL_COLUMNS.items():

    y_pred = mlb.transform(
        df_clean[column].apply(parse_labels)
    )

    # One-vs-rest confusion matrices
    matrices = multilabel_confusion_matrix(
        y_true, y_pred
    )

    class_accuracies = []

    for cm in matrices:
        tn, fp, fn, tp = cm.ravel()

        class_accuracies.append(
            (tp + tn) / (tp + tn + fp + fn)
        )

    results.append({
        "Model": model_name,

        "Exact_Accuracy": accuracy_score(
            y_true, y_pred
        ),

        "Overlap_Accuracy": np.mean(
            (y_true & y_pred).any(axis=1)
        ),

        "Macro_Accuracy_OvR": np.mean(
            class_accuracies
        ),

        "Macro_Precision": precision_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),

        "Macro_Recall": recall_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        )
    })

results_df = pd.DataFrame(results).round(4)

print(results_df.to_string(index=False))


In [ ]:
label_counts = df_eval["Human_labels"].apply(len)

print("Single-label posts:", (label_counts == 1).sum())
print("Multi-label posts:", (label_counts > 1).sum())
print("Maximum labels per post:", label_counts.max())

print("\nHuman label distribution:")
print(df_eval["Human_Topic"].value_counts())

In [ ]:
df_eval.head(2)

In [ ]:
df_human=pd.read_csv('/content/drive/MyDrive/wildlife_annotation/df_2000_final_all_valid.csv')

In [ ]:
df.columns

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# THRESHOLD SENSITIVITY DATA
# Previously calculated on the same 2,000 posts
# ============================================================

bge_thresholds = [
    0.300, 0.325, 0.350, 0.375, 0.400, 0.425,
    0.450, 0.475, 0.500, 0.525, 0.550, 0.575, 0.600
]

bge_coverage = [
    97.60, 91.55, 78.85, 59.75, 40.60, 26.15,
    14.60, 8.45, 4.00, 2.15, 1.20, 0.70, 0.20
]

e5_thresholds = [
    0.760, 0.770, 0.780, 0.790, 0.800, 0.810,
    0.820, 0.830, 0.840, 0.850, 0.860, 0.870,
    0.880, 0.890
]

e5_coverage = [
    100.00, 99.40, 97.35, 90.05, 71.30, 48.05,
    26.85, 11.75, 5.40, 1.95, 0.70, 0.20,
    0.05, 0.00
]

# ============================================================
# CREATE FIGURE
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.0, 2.65),
    sharey=True
)

models = [
    ("(a) BGE-M3", bge_thresholds, bge_coverage),
    ("(b) E5-Large", e5_thresholds, e5_coverage)
]

for ax, (name, thresholds, coverage) in zip(axes, models):

    ax.plot(
        thresholds,
        coverage,
        marker="o",
        markersize=3.5,
        linewidth=1.8,
        color="#245C89"
    )

    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Cosine similarity threshold")
    ax.set_ylim(-2, 103)

    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.3
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_xticks(np.linspace(
        min(thresholds),
        max(thresholds),
        5
    ))

    ax.tick_params(axis="x", rotation=0)

axes[0].set_ylabel("Assignment coverage (%)")

fig.tight_layout(w_pad=1.5)

# Save high-quality formats for LaTeX
fig.savefig(
    "threshold_sensitivity.pdf",
    bbox_inches="tight"
)

fig.savefig(
    "threshold_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import matplotlib.pyplot as plt

# -----------------------------
# Threshold results
# -----------------------------
bge_thresholds = [0.300, 0.325, 0.350, 0.375, 0.400, 0.425,
                  0.450, 0.475, 0.500, 0.525, 0.550, 0.575, 0.600]
bge_coverage = [97.60, 91.55, 78.85, 59.75, 40.60, 26.15,
                14.60, 8.45, 4.00, 2.15, 1.20, 0.70, 0.20]

e5_thresholds = [0.760, 0.770, 0.780, 0.790, 0.800, 0.810,
                 0.820, 0.830, 0.840, 0.850, 0.860, 0.870,
                 0.880, 0.890]
e5_coverage = [100.00, 99.40, 97.35, 90.05, 71.30, 48.05,
               26.85, 11.75, 5.40, 1.95, 0.70, 0.20,
               0.05, 0.00]

# -----------------------------
# Figure
# -----------------------------
plt.figure(figsize=(6.4, 3.0))

plt.plot(bge_thresholds, bge_coverage, marker='o', linewidth=1.8, label='BGE-M3')
plt.plot(e5_thresholds, e5_coverage, marker='s', linewidth=1.8, label='E5-Large')

plt.xlabel("Cosine similarity threshold")
plt.ylabel("Assignment coverage (%)")
plt.ylim(-2, 102)
plt.xlim(0.28, 0.90)

plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.legend(frameon=False)
plt.tight_layout()

plt.savefig("threshold_sensitivity_single.pdf", bbox_inches="tight")
plt.savefig("threshold_sensitivity_single.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df_bge=pd.read_csv('/content/sample_data/Wildlife_12_topics_bge-m3csv.csv')
df_e5=pd.read_csv('/content/sample_data/Wildlife_12_topics_e5_large22.csv')
#df_human=pd.read_csv('df_2000_final_all_valid.csv')

In [ ]:
df_human.columns

In [ ]:

import pandas as pd
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

# ============================================================
# STEP 1: GET HUMAN ANNOTATIONS
# ============================================================

human = df_human[
    ["URL", "Human_Topic"]
].copy()

human["URL"] = human["URL"].astype(str).str.strip()

# Check for duplicate URLs
if human["URL"].duplicated().any():
    raise ValueError("Duplicate URLs found in df_human.")

print("Human-annotated posts:", len(human))


# ============================================================
# STEP 2: GET E5-LARGE SIMILARITY SCORES
# ============================================================

e5 = df_e5[
    ["URL", "Topic_Similarity"]
].copy()

e5["URL"] = e5["URL"].astype(str).str.strip()

e5 = e5.rename(columns={
    "Topic_Similarity": "E5_Similarity"
})

if e5["URL"].duplicated().any():
    raise ValueError("Duplicate URLs found in df_e5.")


# ============================================================
# STEP 3: GET BGE-M3 SIMILARITY SCORES
# ============================================================

bge = df_bge[
    ["URL", "Topic_Similarity"]
].copy()

bge["URL"] = bge["URL"].astype(str).str.strip()

bge = bge.rename(columns={
    "Topic_Similarity": "BGE_Similarity"
})

if bge["URL"].duplicated().any():
    raise ValueError("Duplicate URLs found in df_bge.")


# ============================================================
# STEP 4: MERGE BY URL
# ============================================================

comparison = (
    human
    .merge(e5, on="URL", how="inner", validate="one_to_one")
    .merge(bge, on="URL", how="inner", validate="one_to_one")
)

print("\nAligned posts:", len(comparison))

if len(comparison) != len(human):
    raise ValueError(
        "Some human-annotated posts could not be matched "
        "with embedding scores. Check the URL values."
    )


# ============================================================
# STEP 5: CREATE BINARY HUMAN REFERENCE LABELS
# ============================================================

if comparison["Human_Topic"].isna().any():
    raise ValueError("Missing human annotations found.")

comparison["Human_Topic"] = (
    comparison["Human_Topic"]
    .astype(str)
    .str.strip()
)

# 1 = One of the wildlife topics
# 0 = UNASSIGNED
comparison["Relevant"] = (
    comparison["Human_Topic"] != "UNASSIGNED"
).astype(int)

print("\nHuman reference distribution:")
print(comparison["Relevant"].value_counts())

print(
    "Relevant prevalence:",
    round(comparison["Relevant"].mean(), 4)
)


# ============================================================
# STEP 6: CALCULATE AUROC AND AVERAGE PRECISION
# ============================================================

results = []

for model_name, score_col in {
    "E5-Large": "E5_Similarity",
    "BGE-M3": "BGE_Similarity"
}.items():

    valid = comparison[score_col].notna()

    y_true = comparison.loc[valid, "Relevant"]
    scores = comparison.loc[valid, score_col]

    if y_true.nunique() < 2:
        raise ValueError(
            f"{model_name}: Both relevant and "
            "UNASSIGNED examples are required."
        )

    auroc = roc_auc_score(
        y_true,
        scores
    )

    ap = average_precision_score(
        y_true,
        scores
    )

    results.append({
        "Model": model_name,
        "N": len(y_true),
        "AUROC": auroc,
        "Average_Precision": ap,
        "Positive_Prevalence": y_true.mean()
    })


results_df = pd.DataFrame(results)

print("\nMODEL RELEVANCE DETECTION PERFORMANCE")
print(results_df.round(4).to_string(index=False))


# ============================================================
# STEP 7: SAVE RESULTS
# ============================================================

results_df.to_csv(
    "embedding_relevance_auc_ap.csv",
    index=False
)

comparison.to_csv(
    "embedding_human_aligned_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nEvaluation completed.")


In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.0, 2.8)
)

model_scores = {
    "E5-Large": "E5_Similarity",
    "BGE-M3": "BGE_Similarity"
}

for name, column in model_scores.items():

    valid = comparison[column].notna()

    y = comparison.loc[valid, "Relevant"]
    scores = comparison.loc[valid, column]

    # ROC curve
    fpr, tpr, _ = roc_curve(y, scores)

    auc = roc_auc_score(y, scores)

    axes[0].plot(
        fpr,
        tpr,
        linewidth=1.7,
        label=f"{name} (AUC={auc:.3f})"
    )

    # Precision-recall curve
    precision, recall, _ = precision_recall_curve(
        y, scores
    )

    ap = average_precision_score(y, scores)

    axes[1].plot(
        recall,
        precision,
        linewidth=1.7,
        label=f"{name} (AP={ap:.3f})"
    )


# Random classifier reference
axes[0].plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1
)

# AP prevalence baseline
baseline = comparison["Relevant"].mean()

axes[1].axhline(
    baseline,
    linestyle="--",
    color="gray",
    linewidth=1,
    label=f"Prevalence={baseline:.3f}"
)

axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("(a) ROC")

axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("(b) Precision–Recall")

for ax in axes:
    ax.legend(fontsize=7, frameon=False)
    ax.grid(axis="both", alpha=0.2)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

plt.tight_layout()

plt.savefig(
    "embedding_relevance_roc_pr.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "embedding_relevance_roc_pr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model_columns = {
    "Llama-3.1-70B": "LLM_Topic_llama",
    "GPT-5.1": "LLM_Topic_gpt",
    "Gemini-2.5-Flash": "LLM_Topic_gemini",
    "Qwen3-8B": "Qwen_topic",
    "MiniLM": "SEBRT_miniML",
    "E5-Large": "SBERT_e5Large",
    "BGE-M3": "SBERT_bGE"
}

unassigned = "UNASSIGNED"

def normalize_labels(series):
    return series.astype("string").str.strip().str.upper()

human = normalize_labels(df_human["Human_Topic"])
valid_human = human.notna()

y_true = (human[valid_human] != unassigned).astype(int)

results = []

for model, column in model_columns.items():

    predictions = normalize_labels(
        df_human.loc[valid_human, column]
    )

    # Do not silently treat missing predictions as valid labels
    valid = predictions.notna()

    y = y_true[valid]
    pred = (predictions[valid] != unassigned).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": model,
        "N": len(y),
        "Assigned (%)": 100 * pred.mean(),
        "Precision": precision_score(
            y, pred, zero_division=0
        ),
        "Recall": recall_score(
            y, pred, zero_division=0
        ),
        "F1": f1_score(
            y, pred, zero_division=0
        ),
        "FPR": fp / (fp + tn) if fp + tn else 0
    })

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))


In [ ]:

import pandas as pd
import numpy as np

# ============================================================
# CONFIGURATION
# ============================================================

model_columns = {
    "Llama-3.1-70B": "LLM_Topic_llama",
    "GPT-5.1": "LLM_Topic_gpt",
    "Gemini-2.5-Flash": "LLM_Topic_gemini",
    "Qwen3-8B": "Qwen_topic"
}

UNASSIGNED = "UNASSIGNED"

df_error = df_human.copy()

def normalize_topic(series):
    return series.astype("string").str.strip().str.upper()

df_error["Human_Normalized"] = normalize_topic(
    df_error["Human_Topic"]
)

# ============================================================
# STEP 1: CLASSIFY ERROR TYPES
# ============================================================

summary_rows = []
error_details = []

for model, column in model_columns.items():

    human = df_error["Human_Normalized"]
    pred = normalize_topic(df_error[column])

    valid = human.notna() & pred.notna()

    h = human[valid]
    p = pred[valid]

    correct_unassigned = (
        (h == UNASSIGNED) & (p == UNASSIGNED)
    )

    false_positive = (
        (h == UNASSIGNED) & (p != UNASSIGNED)
    )

    false_negative = (
        (h != UNASSIGNED) & (p == UNASSIGNED)
    )

    correct_topic = (
        (h != UNASSIGNED) & (p == h)
    )

    wrong_topic = (
        (h != UNASSIGNED) &
        (p != UNASSIGNED) &
        (p != h)
    )

    summary_rows.append({
        "Model": model,
        "N": len(h),
        "Correct_UNASSIGNED": correct_unassigned.sum(),
        "False_Positive": false_positive.sum(),
        "False_Negative": false_negative.sum(),
        "Wrong_Topic": wrong_topic.sum(),
        "Correct_Topic": correct_topic.sum(),
        "Total_Errors": (
            false_positive.sum() +
            false_negative.sum() +
            wrong_topic.sum()
        )
    })

    types = np.select(
        [
            correct_unassigned,
            false_positive,
            false_negative,
            wrong_topic,
            correct_topic
        ],
        [
            "Correct UNASSIGNED",
            "False Positive",
            "False Negative",
            "Wrong Topic",
            "Correct Topic"
        ],
        default="Other"
    )

    details = df_error.loc[valid].copy()
    details["Model"] = model
    details["Predicted_Topic"] = p.values
    details["Error_Type"] = types

    error_details.append(details)


error_summary = pd.DataFrame(summary_rows)

error_details = pd.concat(
    error_details,
    ignore_index=True
)

print("\nERROR SUMMARY")
print(error_summary.to_string(index=False))

# Verify that all examples fall into one of five groups
assert (
    error_summary[
        [
            "Correct_UNASSIGNED",
            "False_Positive",
            "False_Negative",
            "Wrong_Topic",
            "Correct_Topic"
        ]
    ].sum(axis=1).values
    ==
    error_summary["N"].values
).all()


In [ ]:

# ============================================================
# FALSE POSITIVE DISTRIBUTION BY PREDICTED TOPIC
# ============================================================

false_positives = error_details[
    error_details["Error_Type"] == "False Positive"
].copy()

fp_topics = pd.crosstab(
    false_positives["Predicted_Topic"],
    false_positives["Model"]
)

fp_topics = fp_topics.reindex(
    columns=model_columns.keys(),
    fill_value=0
)

print("\nFALSE POSITIVES BY PREDICTED TOPIC")
print(fp_topics.to_string())

# Percentage of each model's false positives
fp_topic_pct = (
    fp_topics
    .div(fp_topics.sum(axis=0), axis=1)
    .mul(100)
    .round(1)
)

print("\nFALSE POSITIVE DISTRIBUTION (%)")
print(fp_topic_pct.to_string())


In [ ]:

# ============================================================
# FALSE NEGATIVES BY HUMAN TOPIC
# ============================================================

false_negatives = error_details[
    error_details["Error_Type"] == "False Negative"
].copy()

fn_topics = pd.crosstab(
    false_negatives["Human_Normalized"],
    false_negatives["Model"]
)

fn_topics = fn_topics.reindex(
    columns=model_columns.keys(),
    fill_value=0
)

print("\nFALSE NEGATIVES BY HUMAN TOPIC")
print(fn_topics.to_string())

# False-negative rate within each human topic
human_topic_counts = (
    df_error.loc[
        df_error["Human_Normalized"] != UNASSIGNED,
        "Human_Normalized"
    ]
    .value_counts()
)

fn_rate_by_topic = (
    fn_topics
    .div(human_topic_counts, axis=0)
    .mul(100)
    .round(1)
)

print("\nFALSE NEGATIVE RATE BY HUMAN TOPIC (%)")
print(fn_rate_by_topic.to_string())


In [ ]:

# ============================================================
# WRONG-TOPIC CONFUSIONS
# ============================================================

wrong_topics = error_details[
    error_details["Error_Type"] == "Wrong Topic"
].copy()

topic_confusions = (
    wrong_topics
    .groupby(
        ["Model", "Human_Normalized", "Predicted_Topic"]
    )
    .size()
    .reset_index(name="Count")
    .sort_values(
        ["Model", "Count"],
        ascending=[True, False]
    )
)

print("\nTOP WRONG-TOPIC CONFUSIONS")

for model in model_columns:
    print(f"\n{model}")
    print(
        topic_confusions[
            topic_confusions["Model"] == model
        ].head(10).to_string(index=False)
    )


In [ ]:

# ============================================================
# EXPORT MISCLASSIFIED POSTS
# ============================================================

errors_only = error_details[
    error_details["Error_Type"].isin([
        "False Positive",
        "False Negative",
        "Wrong Topic"
    ])
].copy()

# Check whether text is already available
text_columns = [
    c for c in ["Message", "Image Text", "Description"]
    if c in errors_only.columns
]

export_columns = [
    "URL",
    "Model",
    "Human_Topic",
    "Predicted_Topic",
    "Error_Type"
] + text_columns

errors_only[export_columns].to_csv(
    "llm_error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

sampled_errors = pd.concat(
    [
        group.sample(
            n=min(10, len(group)),
            random_state=42
        )
        for _, group in errors_only.groupby(
            ["Model", "Error_Type"]
        )
    ],
    ignore_index=True
)

sampled_errors[export_columns].to_csv(
    "llm_error_examples.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Exported:", len(sampled_errors), "examples")
print("Exported error analysis files.")


In [ ]:
print("\n1. ERROR SUMMARY")
print(error_summary.to_string(index=False))

print("\n2. FALSE POSITIVES BY TOPIC")
print(fp_topics.to_string())

print("\n3. FALSE NEGATIVES BY TOPIC")
print(fn_topics.to_string())

print("\n4. TOP WRONG-TOPIC CONFUSIONS")
print(
    topic_confusions
    .groupby("Model")
    .head(5)
    .to_string(index=False)
)